<a href="https://colab.research.google.com/github/TomasMarques175/PBD/blob/main/PBD_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup**


In [ ]:
# @title Mount drive - Only if you are working with Colab, discard otherwise

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title Imports
from collections import defaultdict, Counter
import matplotlib.patches as patches
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import os
from PIL import Image
import numpy as np
import pandas as pd
import re
from scipy.io import wavfile
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.cluster import AgglomerativeClustering
import string
import umap
import random
from sklearn.manifold import Isomap
from sklearn.metrics import silhouette_score
from scipy.sparse import SparseEfficiencyWarning
import warnings
from sklearn.metrics import pairwise_distances
from scipy.optimize import linear_sum_assignment
import pickle

!pip install intervaltree
from intervaltree import IntervalTree

# Download stopwords if you haven't already
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# @title Select Analysis Mode and Video
# @markdown Choose whether to analyze a single video or multiple videos.
# @markdown If analyzing a single video, specify the video name.

# !!!!! Don't Change this !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
multiple_videos = True  # Set to True to process all debates
# !!!!! Don't Change this !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# Path to the features folder
features_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features'

# === Logic for Loading Data ===

i=0
while i<2:
  if not multiple_videos:
      # Analyze a single video
      video = 'chega-be'  # change as needed
      print(f"Loading data for single video: {video}")

      data = pd.read_pickle(os.path.join(features_path, video + '_visual.pkl'))

      display(data.head())

      frame = 175

  else:
      # Analyze multiple videos
      print("Loading data for ALL videos...")

      # List all *_visual.pkl files
      visual_files = [f for f in os.listdir(features_path) if f.endswith('_visual.pkl')]

      all_videos_data = {}

      for file in visual_files:
          video_name = file.replace('_visual.pkl', '')
          file_path = os.path.join(features_path, file)

          try:
              df = pd.read_pickle(file_path)
              all_videos_data[video_name] = df
              print(f"✓ Loaded {video_name} with {len(df)} frames.")
          except Exception as e:
              print(f"✗ Failed to load {video_name}: {e}")

      print(f"\nTotal videos loaded: {len(all_videos_data)}")
      multiple_videos = False
  # Pick one of the videos (only relevant if multiple_videos is False)
  i = i+1
multiple_videos = False

# -- **Visual Data** --


## **-All Videos-**


In [ ]:
# @title Multiple videos flag
# Set flag for single or multiple videos (for all the code)
multiple_videos = True

In [ ]:
# @title List all Pickle (.pkl) files

pklfiles = [f for f in os.listdir('/content/drive/MyDrive/PBD/Project/Project_Data/Features') if os.path.isfile(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Features', f)) and f.endswith('_visual.pkl')]
print(pklfiles)

In [ ]:
# @title Order frame names and Verification for All Videos

# Assuming 'all_videos_data' dictionary is already populated with DataFrames for each video

print("Processing and verifying data for all videos:")

# Iterate through each video and its DataFrame in the dictionary
for video_name, video_data_df in all_videos_data.items():

    print(f"\n--- Processing video: {video_name} ---")

    # Sort the DataFrame by the 'filename' column for the current video
    # .sort_values(by='filename') sorts the DataFrame
    # .reset_index(drop=True) resets the index after sorting and drops the old index
    sorted_video_data = video_data_df.sort_values(by='filename').reset_index(drop=True)

    # Update the dictionary with the sorted DataFrame (optional, but good practice)
    all_videos_data[video_name] = sorted_video_data


    # Print first lines to verify the order for the current video
    print(f"Data after sorting by filename for {video_name}:")
    # .head(5) gets the first 5 rows of the DataFrame
    # ['filename'] selects the 'filename' column
    # .tolist() converts the selected column to a list for easier printing
    print(sorted_video_data['filename'].head(5).tolist())


    # Print number of frames for the current video
    print(f"Number of frames for {video_name}: {len(sorted_video_data)}")

In [ ]:
# @title Lets visualize some features in the images - Detections (Random Frame)

# Define variables for visualization based on analysis mode
if multiple_videos:
    # If multiple videos are loaded, select a random video and a random frame from it
    video_names = list(all_videos_data.keys())
    video_to_visualize = np.random.choice(video_names) # Randomly select a video name
    video_to_visualize = "chega-be"
    current_video_data = all_videos_data[video_to_visualize]
    total_frames_in_video = len(current_video_data)
    frame_to_visualize = np.random.randint(0, total_frames_in_video) # Randomly select a frame index

    print(f"Visualizing random frame {frame_to_visualize} from randomly selected video: {video_to_visualize}")

else:
    # If analyzing a single video, use the previously loaded 'data' DataFrame
    # Ensure 'data' is loaded - this would have happened if multiple_videos was False
    try:
        current_video_data = data
        total_frames_in_video = len(current_video_data)
        frame_to_visualize = np.random.randint(0, total_frames_in_video) # Randomly select a frame index
        video_to_visualize = video # Use the video name defined for single video mode

        print(f"Visualizing random frame {frame_to_visualize} from single video: {video_to_visualize}")

    except NameError:
        print("Error: 'data' DataFrame not found. Ensure 'multiple_videos' was False when loading a single video, or load it here.")
        exit() # Stop execution if data isn't available


# Get frame information using the specified frame index and data
frame_id = current_video_data.iloc[frame_to_visualize]['filename']
detections_data = current_video_data.iloc[frame_to_visualize]['detections']


# Construct the full path to the image file
image_path = os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Video_Frames', video_to_visualize, frame_id)
print(f"Image path: {image_path}")

# Open the image
try:
    img = Image.open(image_path)
except FileNotFoundError:
    print(f"Error: Image file not found at {image_path}. Please check the path and file name.")
    exit() # Stop execution if image is not found

# Display the image with detections
fig, ax = plt.subplots(1) # Create a figure and axes
ax.imshow(img)

print("\nDetections for this frame:")
print(detections_data)

# Add bounding boxes and labels to the image
for bbox in detections_data:
    # Ensure bbox has at least 6 elements (x, y, width, height, class, confidence)
    if len(bbox) >= 6:
        x, y, width, height, obj_class, confidence = bbox
        # Create a rectangle patch
        rect = patches.Rectangle((x, y), width, height, linewidth=1, edgecolor='r', facecolor='none')
        # Add patch to the image
        ax.add_patch(rect)
        # Add text label (object class and confidence)
        ax.text(x, y, f'{obj_class} ({100*confidence:.1f}%)',
                verticalalignment='top',
                color='r', fontsize=8, bbox=dict(boxstyle='round,pad=0.2', fc='yellow', alpha=0.5)) # Added background to text

    else:
        print(f"Warning: Skipping detection with unexpected format: {bbox}")


# Remove axis ticks for a cleaner look
ax.set_xticks([])
ax.set_yticks([])

# Set title
ax.set_title(f"Detections on Frame: {frame_id}")

# Show the plot
plt.show()

In [ ]:
# @title Total Object Counts

# Ensure necessary imports are present (should be from previous cells)
# from PIL import Image
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches
# import numpy as np

# This structure will store data either for the single video or for all
analysis_data = {}

if multiple_videos:
    print("Analyzing object counts for ALL videos...")
    # If multiple videos, the analysis data is all_videos_data
    if 'all_videos_data' in locals():
        analysis_data = all_videos_data
    else:
        print("Error: 'all_videos_data' not found. Please ensure multiple videos were loaded in a previous cell.")
        exit() # Stop execution if multiple videos data is not available
else:
    print(f"Analyzing object counts for single video: {video}") # Assuming 'video' is defined
    # If single video, the analysis data is the 'data' DataFrame
    if 'data' in locals():
        # We wrap the single DataFrame in a dictionary structure
        # This allows us to use the same loop logic for both cases
        analysis_data[video] = data
    else:
         print("Error: 'data' DataFrame not found. Please ensure single video data was loaded in a previous cell.")
         exit() # Stop execution if single video data is not available


# Dictionary to store combined total counts across all analyzed data (optional)
# This will be combined for all videos if multiple_videos is True,
# or just the total for the single video if multiple_videos is False.
combined_total_counts = {}

# Now, iterate through the analysis_data dictionary
for video_name, video_data_df in analysis_data.items():

    print(f"\n--- Analyzing video: {video_name} ---")

    num_frames = len(video_data_df)
    object_counts_this_video = [] # Store object counts per frame for this video

    for frame_index in range(num_frames):
        # Using .iloc is fine here as the DataFrame should be sorted by filename already
        detections = video_data_df.iloc[frame_index]['detections']

        frame_counts = {}
        for bbox in detections:
            # Ensure bbox has enough elements before accessing index 4
            if len(bbox) > 4:
                object_name = bbox[4] # Get the object name
                frame_counts[object_name] = frame_counts.get(object_name, 0) + 1
            else:
                print(f"Warning: Skipping detection with unexpected format in {video_name}, frame {frame_index}: {bbox}")

        object_counts_this_video.append(frame_counts)


    # --- Calculate and Display Total Counts for THIS Video ---
    total_counts_this_video = {}
    for frame_counts in object_counts_this_video:
        for object_name, count in frame_counts.items():
            total_counts_this_video[object_name] = total_counts_this_video.get(object_name, 0) + count

            # Always update the combined total counts dictionary
            # If multiple_videos is False, this will just be the total for the single video
            combined_total_counts[object_name] = combined_total_counts.get(object_name, 0) + count


    print(f"Total Object Counts for {video_name}:")
    print(total_counts_this_video)

    # Create histogram for THIS video's total counts
    if total_counts_this_video: # Only plot if there are any detections
        plt.figure(figsize=(8, 6))
        plt.bar(total_counts_this_video.keys(), total_counts_this_video.values())
        plt.xlabel("Object Type")
        plt.ylabel("Total Count")
        plt.title(f"Total Object Counts in Video: {video_name}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print(f"No objects detected in {video_name} for total count plot.")


    # --- Prepare and Plot Scatter Plot for THIS Video's Object Counts per Frame ---

    # Get unique object types for THIS video
    all_object_types_this_video = set()
    for frame_counts in object_counts_this_video:
        all_object_types_this_video.update(frame_counts.keys())
    all_object_types_this_video = sorted(list(all_object_types_this_video))

    if all_object_types_this_video: # Only plot if there are any object types
        frame_numbers = np.arange(num_frames) + 1
        object_type_counts_filtered = {object_type: {'frames': [], 'counts': []} for object_type in all_object_types_this_video}

        for frame_index, frame_counts in enumerate(object_counts_this_video):
            for object_type in all_object_types_this_video:
                count = frame_counts.get(object_type, 0)
                if count > 0:
                    object_type_counts_filtered[object_type]['frames'].append(frame_numbers[frame_index])
                    object_type_counts_filtered[object_type]['counts'].append(count)

        plt.figure(figsize=(12, 6))

        for i, object_type in enumerate(all_object_types_this_video):
            plt.scatter(object_type_counts_filtered[object_type]['frames'],
                        object_type_counts_filtered[object_type]['counts'],
                        label=object_type, alpha=0.6, s=10)

        plt.xlabel("Frame Number")
        plt.ylabel("Object Count")
        plt.title(f"Object Counts per Frame ({video_name})") # Changed title to include video name
        # Adjusting xticks might need consideration based on num_frames
        # plt.xticks(np.arange(1, num_frames + 1, 100))
        plt.ylim(0, max(5, max([max(v['counts'], default=0) for v in object_type_counts_filtered.values()] + [1]))) # Adjust y-limit dynamically, minimum 5
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print(f"No objects detected in {video_name} for scatter plot.")

# --- Final Combined Plot (Optional) ---
# This will show the total counts across all videos analyzed
# (which is just the single video if multiple_videos is False)

print("\n--- Combined Total Object Counts Across Analyzed Data ---")
print(combined_total_counts)

if combined_total_counts and (multiple_videos or len(analysis_data) == 1): # Plot if there are counts and it's either multi-video or the single video case
    plt.figure(figsize=(10, 7))
    plt.bar(combined_total_counts.keys(), combined_total_counts.values())
    plt.xlabel("Object Type")
    plt.ylabel("Total Count")
    title_suffix = "Across All Videos" if multiple_videos else "in Single Video"
    plt.title(f"Combined Total Object Counts {title_suffix}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# @title Box-Plot of the Confidences for each Type of Object
# @@markdown Makes more sense to analyse each type separatly. This way we can ignore possible errors in classification easier.

# Ensure necessary imports are present (should be from previous cells)
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np

# This structure will hold data either for the single video or for all
analysis_data = {}

if multiple_videos:
    print("Collecting confidence data for ALL videos...")
    # If multiple videos, the analysis data is all_videos_data
    if 'all_videos_data' in locals():
        analysis_data = all_videos_data
    else:
        print("Error: 'all_videos_data' not found. Please ensure multiple videos were loaded in a previous cell.")
        exit() # Stop execution if multiple videos data is not available
else:
    print(f"Collecting confidence data for single video: {video}") # Assuming 'video' is defined
    # If single video, the analysis data is the 'data' DataFrame
    if 'data' in locals():
        # We wrap the single DataFrame in a dictionary structure
        # This allows us to use the same loop logic for collecting data
        analysis_data[video] = data
    else:
         print("Error: 'data' DataFrame not found. Please ensure single video data was loaded in a previous cell.")
         exit() # Stop execution if single video data is not available


# Dictionaries to collect confidence values and counts across ALL analyzed data
confidence_by_object = {}
count_by_object = {}

# Iterate through each video and its DataFrame in the analysis data
for video_name, video_data_df in analysis_data.items():

    # Iterate through each frame in the current video's DataFrame
    # len(video_data_df) gets the number of frames in the current video
    for frame_index in range(len(video_data_df)):
        # Get the detections for the current frame
        # .iloc[frame_index]['detections'] accesses the 'detections' list for the frame at the current index
        detections = video_data_df.iloc[frame_index]['detections']

        # Process detections in the current frame
        for detection in detections:
            # Ensure detection has enough elements before accessing index 4 and 5
            if len(detection) > 5: # Bbox should have at least 6 elements: x, y, w, h, class, confidence
                object_class = detection[4] # Get the object class name
                confidence = detection[5]   # Get the confidence score

                # Collect confidence values
                if object_class not in confidence_by_object:
                  confidence_by_object[object_class] = []
                  count_by_object[object_class] = 0 # Initialize count for this object class
                confidence_by_object[object_class].append(confidence)
                count_by_object[object_class] += 1 # Increment count for this object class
            else:
                # Optional: Print a warning if a detection is in an unexpected format
                # print(f"Warning: Skipping detection with unexpected format in {video_name}, frame {frame_index}: {detection}")
                pass # Or just pass silently if you don't want warnings for malformed data


# --- Display Counts and Prepare Data for Plotting ---

print("\nTotal Detections by Object Class Across Analyzed Data:")
# Print counts for verification
for obj_class in count_by_object:
  print(f"{obj_class}: {count_by_object[obj_class]}")

# Prepare data for plotting
# Get a list of object class names that have detections
object_classes_with_data = sorted(list(confidence_by_object.keys()))

# Get a list of confidence lists, one list per object class, in sorted order
confidence_values = [confidence_by_object[obj] for obj in object_classes_with_data]


# --- Create and Display Plots ---

# Only attempt to plot if there are any object classes with detections
if object_classes_with_data:

    # Create the figure and subplots (2 rows, 1 column)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))

    # Box Plot
    ax1.boxplot(confidence_values, labels=object_classes_with_data, showfliers=True) # Use the list of classes with data as labels
    ax1.set_xlabel("Object Class")
    ax1.set_ylabel("Confidence Level")
    title_suffix = "Across All Videos" if multiple_videos else "in Single Video"
    ax1.set_title(f"Box Plot of Confidence Levels by Object Class ({title_suffix})") # Dynamic title
    ax1.tick_params(axis='x', rotation=45) # Rotate x-axis labels

    # Scatter Plot of Individual Confidence Values
    # Iterate through the sorted list of object classes that have data
    for i, obj_class in enumerate(object_classes_with_data):
        y_values = confidence_by_object[obj_class] # Get the confidence scores for this class
        # Create x-values: use the index (i) + 1 for horizontal position, repeated for each confidence value
        x_values = [i + 1] * len(y_values)
        # Plot the scatter points for this object class
        ax2.scatter(x_values, y_values, alpha=0.5, label=obj_class, s=10) # alpha for transparency, label for legend, s for marker size

    ax2.set_xticks(np.arange(1, len(object_classes_with_data) + 1)) # Set x-ticks at the positions used in the scatter plot
    ax2.set_xticklabels(object_classes_with_data, rotation=45) # Label the x-ticks with object class names
    ax2.set_xlabel("Object Class")
    ax2.set_ylabel("Confidence Level")
    ax2.set_title(f"Individual Confidence Values by Object Class ({title_suffix})") # Dynamic title
    ax2.legend() # Display the legend
    ax2.grid(axis='y', linestyle='--', alpha=0.7) # Add a horizontal grid


    plt.tight_layout() # Adjust subplot parameters for a tight layout
    plt.show() # Display the figure

else:
    print("No object detections found in the analyzed data to plot confidence levels.")

In [ ]:
# @title Visualize Poses (Random Frame)

# Ensure necessary imports are present (should be from previous cells)
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np
# from PIL import Image
# import os

# Define variables for visualization based on analysis mode
if multiple_videos:
    # If multiple videos are loaded, select a random video and a random frame from it
    video_names = list(all_videos_data.keys())
    video_to_visualize = np.random.choice(video_names) # Randomly select a video name
    current_video_data = all_videos_data[video_to_visualize]
    total_frames_in_video = len(current_video_data)
    frame_to_visualize = np.random.randint(0, total_frames_in_video) # Randomly select a frame index

    print(f"Visualizing poses on random frame {frame_to_visualize} from randomly selected video: {video_to_visualize}")

else:
    # If analyzing a single video, use the previously loaded 'data' DataFrame
    # Ensure 'data' is loaded - this would have happened if multiple_videos was False
    try:
        current_video_data = data
        total_frames_in_video = len(current_video_data)
        frame_to_visualize = np.random.randint(0, total_frames_in_video) # Randomly select a frame index
        video_to_visualize = video # Use the video name defined for single video mode

        print(f"Visualizing poses on random frame {frame_to_visualize} from single video: {video_to_visualize}")

    except NameError:
        print("Error: 'data' DataFrame not found. Ensure 'multiple_videos' was False when loading a single video, or load it here.")
        exit() # Stop execution if data isn't available

# Get frame information using the specified frame index and data
frame_id = current_video_data.iloc[frame_to_visualize]['filename']
poses_data = np.array(current_video_data.iloc[frame_to_visualize]['poses'])


# Construct the full path to the image file
image_path = os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Video_Frames', video_to_visualize, frame_id)
print(f"Image path: {image_path}")

# Open the image
try:
  img = Image.open(image_path)
except FileNotFoundError:
  print(f"Error: Image file not found at {image_path}. Please check the path and file name.")
  exit() # Stop execution if image is not found

# Display the image with poses
fig, ax = plt.subplots(1) # Create a figure and axes
ax.imshow(img)

width, height = img.size
print(f"\nPose data shape for this frame: {poses_data.shape}")

# Plot keypoints for each pose in the frame
for pose in poses_data:

  # Ensure pose has the expected structure (at least keypoints x 5 values)
  if pose.shape[1] >= 5:
    # convert normalized landmark to pixel position
    x = np.floor(pose[:,0] * width)
    y = np.floor(pose[:,1] * height)

    # Plot keypoints based on visibility and presence thresholds
    for i in range(pose.shape[0]):
      if pose[i,3]>.5 and pose[i,4]>.5: #visibility and presence thresholds
        ax.scatter(x[i], y[i], color='r' if i == 0 else 'w', s=50 if i == 0 else 1)

  else:
        print(f"Warning: Skipping pose with unexpected shape in {video_to_visualize}, frame {frame_to_visualize}. Expected shape with at least 5 columns per keypoint, but got {pose.shape}.")


# Remove axis ticks for a cleaner look
ax.set_xticks([])
ax.set_yticks([])

# Set title
ax.set_title(f"Pose Detections on Frame: {frame_id} ({video_to_visualize})")

# Show the plot
plt.show()

In [ ]:
# @title Visibility and Presence Box-plot

# Ensure necessary imports are present (should be from previous cells)
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np

# This structure will hold data either for the single video or for all
analysis_data = {}

if multiple_videos:
    print("Collecting visibility and presence data for ALL videos...")
    # If multiple videos, the analysis data is all_videos_data
    if 'all_videos_data' in locals():
        analysis_data = all_videos_data
    else:
        print("Error: 'all_videos_data' not found. Please ensure multiple videos were loaded in a previous cell.")
        exit() # Stop execution if multiple videos data is not available
else:
    print(f"Collecting visibility and presence data for single video: {video}") # Assuming 'video' is defined
    # If single video, the analysis data is the 'data' DataFrame
    if 'data' in locals():
        # We wrap the single DataFrame in a dictionary structure
        # This allows us to use the same loop logic for collecting data
        analysis_data[video] = data
    else:
         print("Error: 'data' DataFrame not found. Please ensure single video data was loaded in a previous cell.")
         exit() # Stop execution if single video data is not available


# Lists to collect all visibility and presence values across ALL analyzed data
visibility_values_all_frames = []
presence_values_all_frames = []

# Iterate through each video and its DataFrame in the analysis data
for video_name, video_data_df in analysis_data.items():

    # Iterate through each frame in the current video's DataFrame
    for frame_index in range(len(video_data_df)):
        # Get the poses for the current frame
        poses = np.array(video_data_df.iloc[frame_index]['poses'])

        # Iterate through each pose in the frame
        for pose in poses:
            # Ensure pose has the expected structure before accessing columns
            if pose.shape[1] >= 5: # Pose keypoints should have at least 5 values (x,y,z,visibility,presence)
                # Iterate through each keypoint in the pose
                for i in range(pose.shape[0]):
                    # Collect visibility and presence values
                    visibility_values_all_frames.append(pose[i, 3])
                    presence_values_all_frames.append(pose[i, 4])
            else:
                 # Optional: Print a warning for malformed pose data
                 # print(f"Warning: Skipping pose with unexpected shape in {video_name}, frame {frame_index}. Expected shape with at least 5 columns per keypoint, but got {pose.shape}.")
                 pass # Or just pass silently


# --- Create and Display Box Plots ---

# Only attempt to plot if there are any visibility or presence values collected
if visibility_values_all_frames or presence_values_all_frames:

    plt.figure(figsize=(10, 6))

    # Box Plot for Visibility
    plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
    plt.boxplot(visibility_values_all_frames, showfliers = True)
    title_suffix = "Across All Videos" if multiple_videos else "in Single Video"
    plt.title(f'Box Plot of Visibility Values ({title_suffix})') # Dynamic title
    plt.ylabel('Visibility')
    plt.ylim(0, 1.05) # Set y-limit to 0-1 range for scores

    # Box Plot for Presence
    plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
    plt.boxplot(presence_values_all_frames, showfliers = True)
    plt.title(f'Box Plot of Presence Values ({title_suffix})') # Dynamic title
    plt.ylabel('Presence')
    plt.ylim(0, 1.05) # Set y-limit to 0-1 range for scores


    plt.tight_layout() # Adjust subplot parameters for a tight layout
    plt.show() # Display the figure

else:
    print("No pose detections found in the analyzed data to plot visibility and presence levels.")

In [ ]:
# @title Visual Data Analysis - Pose Variance

# This structure will store variance data, either for the single video or for all
variance_data_to_plot = {}
# Dictionary to store number of persons per frame for each video
persons_per_frame_data = {}

if multiple_videos:
    print("\nCalculating and plotting pose keypoint variance and person count for ALL videos...")
    # If multiple videos are loaded, use the data from all_videos_data
    if 'all_videos_data' in locals():
        analysis_data = all_videos_data # Use the loaded multi-video data
    else:
        print("Error: 'all_videos_data' not found. Please ensure multiple videos were loaded in a previous cell.")
        exit() # Stop execution if multiple videos data is not available

else:
    print(f"\nCalculating and plotting pose keypoint variance and person count for single video: {video}") # Assuming 'video' is defined
    # If a single video is loaded, use the 'data' DataFrame
    if 'data' in locals():
        # Wrap the single DataFrame in a dictionary structure for consistent processing
        analysis_data = {video: data}
    else:
         print("Error: 'data' DataFrame not found. Please ensure single video data was loaded in a previous cell.")
         exit() # Stop execution if single video data is not available


# Now, iterate through the analysis_data dictionary (which contains either one or many videos)
for video_name, video_data_df in analysis_data.items():
    print(f"\n--- Analyzing variance and person count for video: {video_name} ---")

    variance_per_frame = []
    persons_per_frame = [] # List to store number of persons per frame
    num_frames = len(video_data_df)

    # Calculate variance and count persons for each frame
    for frame_index in range(num_frames):
        poses = np.array(video_data_df.iloc[frame_index]['poses'])
        frame_variances = []

        # Count the number of poses (persons) in the current frame
        num_persons_in_frame = len(poses)
        persons_per_frame.append(num_persons_in_frame)

        # Process each pose in the frame
        for pose in poses:
            # Ensure pose has the expected structure (at least keypoints x 3 values for coords)
            if pose.shape[1] >= 3:
                # Extract keypoint coordinates (assuming first 3 columns are x, y, z)
                keypoints_coords = pose[:, :3]

                # Calculate variance for each coordinate and sum them up
                # Check if there's more than one keypoint to calculate variance
                if keypoints_coords.shape[0] > 1:
                    variance = np.sum(np.var(keypoints_coords, axis=0))
                    frame_variances.append(variance)
                else:
                     # If only one keypoint, variance is 0
                     frame_variances.append(0)
            else:
                # Handle cases where pose data might not have enough columns
                print(f"Warning: Skipping pose with unexpected shape in {video_name}, frame {frame_index}. Expected shape with at least 3 columns per keypoint, but got {pose.shape}.")


        # Calculate the average variance across all poses in the frame
        if frame_variances:
            variance_per_frame.append(np.mean(frame_variances))
        else:
            # Append 0 if no poses or malformed poses in the frame
            variance_per_frame.append(0)

    # Store the variance data for this video (optional)
    variance_data_to_plot[video_name] = variance_per_frame
    # Store the persons per frame data for this video (optional)
    persons_per_frame_data[video_name] = persons_per_frame

    # --- Create and Display Combined Scatter Plot for THIS Video ---
    plt.figure(figsize=(15, 8)) # Increased figure size

    # Plot Variance
    plt.scatter(np.arange(len(variance_per_frame)), variance_per_frame, s=10, color='blue', label='Average Pose Variance')

    # Plot Number of Persons
    plt.scatter(np.arange(len(persons_per_frame)), persons_per_frame, s=10, color='red', label='Number of Persons Detected')

    plt.xlabel("Frame Number")
    # The Y-axis is now shared between Variance and Person Count, so a generic label might be best
    plt.ylabel("Value (Variance or Person Count)")
    plt.xlim(0, len(variance_per_frame))
    plt.title(f"Pose Analysis Across Frames ({video_name})") # Include video name in title
    plt.grid(True)
    plt.legend() # Add a legend to distinguish the data series
    plt.show()

# After the loop, variance_data_to_plot and persons_per_frame_data dictionaries contain data for all analyzed videos
# You could add code here to do further analysis on this combined data if needed.

In [ ]:
# @title K-Means Clustering on Pose Variance for 'livre-chega'

# Ensure necessary imports are present (should be from previous cells)
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("\n--- Performing K-Means Clustering on Pose Variance for 'livre-chega' ---")

video_to_cluster = 'livre-chega'

# Check if the data for 'livre-chega' is available
if multiple_videos and video_to_cluster in all_videos_data:
    # Ensure 'all_videos_variance_data' is populated from the previous variance analysis cell
    # If not, you would need to re-calculate it here or ensure the dependency is met.
    # Assuming all_videos_variance_data was created and populated correctly.
    if 'variance_data_to_plot' in locals() and video_to_cluster in variance_data_to_plot:
        variance_data = variance_data_to_plot[video_to_cluster] # Get variance from the previously calculated data
        if variance_data is None:
             print(f"Error: Variance data for '{video_to_cluster}' found in all_videos_variance_data but is None.")
        else:
            print(f"Clustering variance data for video: {video_to_cluster}")

            # Reshape the data for K-Means (needs to be a 2D array)
            variance_data_reshaped = np.array(variance_data).reshape(-1, 1)

            # --- Finding the Best K using the Elbow Method and Silhouette Score ---
            print("\nFinding the best K...")
            max_k = 10  # Maximum number of clusters to check
            distortions = []
            silhouette_scores = [] # Keep silhouette scores for analysis
            k_range = range(2, max_k + 1) # K-Means is typically used for k > 1

            for k in k_range:
                kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
                kmeans.fit(variance_data_reshaped)
                distortions.append(kmeans.inertia_)
                if k > 1:
                     score = silhouette_score(variance_data_reshaped, kmeans.labels_)
                     silhouette_scores.append(score)
                else:
                     silhouette_scores.append(None)


            # Plot the Elbow Method and Silhouette Scores (as before)
            plt.figure(figsize=(12, 5))
            plt.subplot(1, 2, 1)
            plt.plot(k_range, distortions, marker='o')
            plt.xlabel("Number of Clusters (K)")
            plt.ylabel("Distortion (Inertia)")
            plt.title("Elbow Method for Optimal K")
            plt.xticks(k_range)
            plt.grid(True)

            plt.subplot(1, 2, 2)
            silhouette_k_range = [k for k in k_range if k > 1]
            valid_silhouette_scores = [score for score in silhouette_scores if score is not None]
            if valid_silhouette_scores:
                plt.plot(silhouette_k_range, valid_silhouette_scores, marker='o')
                plt.xlabel("Number of Clusters (K)")
                plt.ylabel("Silhouette Score")
                plt.title("Silhouette Scores for Different K")
                plt.xticks(silhouette_k_range)
                plt.grid(True)
            else:
                 plt.text(0.5, 0.5, 'No silhouette scores calculated (requires K > 1)', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)


            plt.tight_layout()
            plt.show()

            print("\nDistortion (Inertia) for each K:")
            for k, distortion in zip(k_range, distortions):
                print(f"K={k}: {distortion:.2f}")

            print("\nSilhouette Score for each K (higher is better):")
            for k, score in zip(k_range, silhouette_scores):
                 if score is not None:
                    print(f"K={k}: {score:.4f}")
                 else:
                    print(f"K={k}: N/A (requires K > 1)")


            # --- Performing K-Means with Chosen K (Example: K=4) ---
            chosen_k = 4 # You should adjust this based on the Elbow/Silhouette plots

            print(f"\nPerforming K-Means clustering with K = {chosen_k}")
            kmeans = KMeans(n_clusters=chosen_k, random_state=42, n_init=10)
            clusters = kmeans.fit_predict(variance_data_reshaped)

            # --- Sort Clusters by Average Variance ---
            # Calculate average variance for each original cluster label
            average_variances = {}
            for i in range(chosen_k):
                cluster_frames_indices = np.where(clusters == i)[0]
                if len(cluster_frames_indices) > 0:
                    average_variances[i] = np.mean(np.array(variance_data)[cluster_frames_indices])
                else:
                    average_variances[i] = 0 # Handle empty clusters

            # Get original cluster labels sorted by average variance (ascending)
            # sorted(average_variances.items(), key=lambda item: item[1]) sorts the dictionary items by their values (average variance)
            # [item[0] for item in ...] extracts just the original cluster labels (keys) in the sorted order
            sorted_original_cluster_labels = [item[0] for item in sorted(average_variances.items(), key=lambda item: item[1])]

            # Create a mapping from original cluster label to new sorted label (0, 1, 2, ...)
            original_to_sorted_label_map = {original_label: new_label for new_label, original_label in enumerate(sorted_original_cluster_labels)}

            # Create a new 'sorted_clusters' array with the updated labels
            sorted_clusters = np.array([original_to_sorted_label_map[label] for label in clusters])

            print("\nAverage Variance by (Original) Cluster Label:")
            for original_label in sorted_original_cluster_labels:
                 print(f"Original Cluster {original_label}: {average_variances[original_label]:.4f}")

            print("\nMapping from Original Cluster Label to Sorted Cluster Label (by Variance):")
            for original_label in sorted(original_to_sorted_label_map.keys()):
                print(f"Original {original_label} -> Sorted {original_to_sorted_label_map[original_label]}")


            # --- Visualize Clusters (Sorted) ---
            plt.figure(figsize=(12, 6))
            # Use the new 'sorted_clusters' array for coloring
            scatter = plt.scatter(np.arange(len(variance_data)), variance_data, c=sorted_clusters, cmap='viridis', s=10)
            plt.xlabel("Frame Number")
            plt.ylabel("Average Variance of Keypoint Coordinates")
            plt.title(f"K-Means Clustering (K={chosen_k}, Sorted by Variance) on Pose Variance for '{video_to_cluster}'")
            plt.grid(True)

            # Add a legend for the sorted clusters
            legend_labels = [f'Cluster {i+1} (Avg Var: {average_variances[sorted_original_cluster_labels[i]]:.4f})' for i in range(chosen_k)]
            handles, labels = scatter.legend_elements(prop="colors", alpha=0.6)
            # Need to sort handles and labels based on the sorted cluster order for the legend
            # The legend_elements are typically in order of appearance in the 'c' array or by default color order.
            # Let's create a custom legend or ensure the scatter plot colors map to the sorted labels.
            # A simpler approach is to use the sorted average variances directly in the legend labels
            plt.legend(handles, legend_labels, loc="upper right", title="Clusters (Sorted by Avg Var)")


            plt.show()

            # The 'sorted_clusters' variable now holds the cluster labels sorted by average variance.
            # You would use 'sorted_clusters' instead of 'clusters' in the subsequent stacked histogram code block
            # to ensure the clusters in that plot are also sorted by variance.


    else:
        print(f"Error: 'all_videos_variance_data' not found or data for '{video_to_cluster}' is missing. Please ensure previous cells ran successfully.")


elif not multiple_videos and 'data' in locals() and video == video_to_cluster:
     print(f"\nClustering variance data for single video: {video_to_cluster}")
     print("Sorting by variance in single video mode requires similar logic to the multiple videos mode.")
     # To run the clustering and sorting here, you would need the 'variance_per_frame' list from the single video analysis
     # And then apply the same sorting logic based on average variance.
     # if 'variance_per_frame' in locals():
     #     variance_data = variance_per_frame
     #     variance_data_reshaped = np.array(variance_data).reshape(-1, 1)
     #     # Proceed with finding best K, clustering, and sorting as in the multiple_videos block
     #     # ... (copy the relevant clustering and sorting code from above)
     #     # Make sure to define 'sorted_clusters' here if you want to use it in subsequent cells.
     # else:
     #      print("Variance data ('variance_per_frame') for the single video not found.")


else:
    print(f"Data for '{video_to_cluster}' not found in all_videos_data or multiple video analysis is not enabled.")

In [ ]:
# @title Stacked Histogram of Persons per Frame by Cluster

# Ensure that the 'livre-chega' video was processed and clustered
video_to_analyze = 'livre-chega'

if multiple_videos and video_to_analyze in all_videos_data:

    # Check if clustering and variance calculation were successful for this video
    # We need both the 'sorted_clusters' array (from the modified K-Means block)
    # and the 'persons_per_frame' list from the Pose Variance Analysis block.
    if 'sorted_clusters' in locals() and 'persons_per_frame_data' in locals() and video_to_analyze in persons_per_frame_data:

        print(f"\n--- Analyzing and Plotting Stacked Person Counts by (Sorted) Cluster for '{video_to_analyze}' ---")

        # Get the persons per frame data for the target video
        persons_per_frame_list = persons_per_frame_data[video_to_analyze]

        # Ensure the number of frames in sorted_clusters and persons_per_frame match
        if len(sorted_clusters) != len(persons_per_frame_list):
            print("Error: Number of frames in sorted_clusters and persons_per_frame do not match. Cannot create stacked histogram.")
        else:
            # Dictionary to store counts of persons per frame, separated by the NEW sorted cluster label
            # Key: sorted_cluster_label, Value: dictionary (Key: num_persons, Value: count)
            persons_count_by_sorted_cluster = {}

            # Find all unique 'number of persons' values across all frames
            all_num_persons_set = set()
            for frame_index in range(len(sorted_clusters)):
                sorted_cluster_label = sorted_clusters[frame_index] # Use sorted_clusters here
                num_persons = persons_per_frame_list[frame_index]
                all_num_persons_set.add(num_persons)

                if sorted_cluster_label not in persons_count_by_sorted_cluster:
                    persons_count_by_sorted_cluster[sorted_cluster_label] = {}

                persons_count_by_sorted_cluster[sorted_cluster_label][num_persons] = persons_count_by_sorted_cluster[sorted_cluster_label].get(num_persons, 0) + 1

            # Sort the NEW cluster labels (which are already sorted by variance)
            sorted_new_cluster_labels = sorted(persons_count_by_sorted_cluster.keys())

            # Sort the unique 'number of persons' values
            all_num_persons = sorted(list(all_num_persons_set))
            # If no persons detected at all, handle gracefully
            if not all_num_persons and max(persons_per_frame_list, default=0) == 0:
                 all_num_persons = [0]
            elif not all_num_persons: # Fallback check
                 all_num_persons = [0]


            # Prepare data for plotting in a stacked format
            # Rows will be 'clusters' (sorted), columns will be 'number of persons'
            plot_data = pd.DataFrame(0, index=[f'Cluster {label + 1}' for label in sorted_new_cluster_labels], columns=[f'{n} Person(s)' for n in all_num_persons])

            # Fill the DataFrame with the counts
            for sorted_cluster_label in sorted_new_cluster_labels:
                cluster_name = f'Cluster {sorted_cluster_label + 1}'
                counts_data = persons_count_by_sorted_cluster[sorted_cluster_label]
                for num_persons, count in counts_data.items():
                     person_col_name = f'{num_persons} Person(s)'
                     if cluster_name in plot_data.index and person_col_name in plot_data.columns:
                         plot_data.loc[cluster_name, person_col_name] = count


            # Plot the stacked bar chart
            if not plot_data.empty:
                # Create a figure and axes for the plot
                fig, ax = plt.subplots(figsize=(12, 7)) # Adjust figure size

                # Plot the data as a stacked bar chart
                plot_data.plot(kind='bar', stacked=True, ax=ax, width=0.8) # Use stacked=True

                ax.set_xlabel("Cluster (Sorted by Average Pose Variance)") # Update X-axis label
                ax.set_ylabel("Number of Frames")
                ax.set_title(f"Distribution of Persons per Frame within Clusters ('{video_to_analyze}')")
                ax.set_xticks(np.arange(len(sorted_new_cluster_labels))) # Set ticks based on the number of clusters
                ax.set_xticklabels(plot_data.index, rotation=0) # Use sorted cluster names as labels
                ax.legend(title="Persons per Frame") # Legend shows what each color segment represents
                ax.grid(axis='y', linestyle='--', alpha=0.7) # Add a horizontal grid
                plt.tight_layout() # Adjust layout to prevent labels overlapping
                plt.show()
            else:
                 print("Plotting data is empty. No stacked histogram to plot.")


    else:
        print(f"Could not find 'sorted_clusters' or 'persons_per_frame_data' for '{video_to_analyze}'. Please run the previous cells successfully.")

elif not multiple_videos and 'data' in locals() and video == video_to_analyze:
    print(f"Analysis for single video '{video_to_analyze}'. This block is currently set up for clustering data from multiple videos.")
    print("To run this for a single video, ensure K-Means and pose variance analysis ran and adapt this code block to use single video data.")
    # Add logic here if you want to handle the single video case specifically
    # You would need 'sorted_clusters' and 'persons_per_frame' available from previous single-video analysis
    # if 'sorted_clusters' in locals() and 'variance_per_frame' in locals() and 'persons_per_frame' in locals():
    #     # Adapt the counting and plotting logic from above for single_video
    #     pass # Replace with actual code
    # else:
    #      print("Required data for single video analysis not found.")

else:
    print(f"Data for '{video_to_analyze}' not found in all_videos_data or multiple video analysis is not enabled.")

In [ ]:
# @title Contingency Table and Chi-Square Test of Independence (Excluding 0 and 3 Persons)

# Ensure necessary imports are present
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np
from IPython.display import display

# Ensure that the 'livre-chega' video was processed and clustered
video_to_analyze = 'livre-chega'

if multiple_videos and video_to_analyze in all_videos_data:

    # Check if clustering and variance calculation were successful for this video
    # We need both the 'sorted_clusters' array (from the modified K-Means block)
    # and the 'persons_per_frame' list from the Pose Variance Analysis block.
    if 'sorted_clusters' in locals() and 'persons_per_frame_data' in locals() and video_to_analyze in persons_per_frame_data:

        print(f"\n--- Generating Contingency Table and performing Chi-Square Test for '{video_to_analyze}' (Excluding 0 and 3 Persons) ---")

        # Get the persons per frame data for the target video
        persons_per_frame_list = persons_per_frame_data[video_to_analyze]

        # Ensure the number of frames in sorted_clusters and persons_per_frame match
        if len(sorted_clusters) != len(persons_per_frame_list):
            print("Error: Number of frames in sorted_clusters and persons_per_frame do not match. Cannot perform contingency analysis or Chi-Square test.")
        else:
            # Create a temporary DataFrame to filter
            temp_df = pd.DataFrame({
                'cluster': sorted_clusters,
                'persons': persons_per_frame_list
            })

            # Filter out rows where the number of persons is 0 or 3
            filtered_df = temp_df[(temp_df['persons'] != 0) & (temp_df['persons'] != 3)].copy()

            # Check if there's any data left after filtering
            if filtered_df.empty:
                print("\nNo data remaining after excluding frames with 0 and 3 persons. Cannot generate contingency table or perform Chi-Square test.")
            else:
                # Create the observed contingency table from the filtered data
                observed_contingency_table_filtered = pd.crosstab(filtered_df['cluster'], filtered_df['persons'])

                print("\nObserved Contingency Table (Excluding 0 and 3 Persons): Number of Frames by Sorted Cluster and Number of Persons")
                display(observed_contingency_table_filtered)

                # Perform the Chi-Square Test of Independence on the filtered table
                # Check if the filtered table has more than one row and column
                if observed_contingency_table_filtered.shape[0] > 1 and observed_contingency_table_filtered.shape[1] > 1:
                    chi2, p, dof, expected = chi2_contingency(observed_contingency_table_filtered)

                    print("\nChi-Square Test of Independence Results:")
                    print(f"Chi-Square Statistic: {chi2:.4f}")
                    print(f"P-value: {p:.4f}")
                    print(f"Degrees of Freedom: {dof}")

                    print("\nExpected Contingency Table (from Chi-Square Test):")
                    # Convert the expected numpy array to a pandas DataFrame for better display
                    expected_contingency_df = pd.DataFrame(expected,
                                                           index=observed_contingency_table_filtered.index,
                                                           columns=observed_contingency_table_filtered.columns)
                    display(expected_contingency_df)

                    # Interpretation guidance
                    alpha = 0.05
                    print("\nInterpretation:")
                    if p < alpha:
                        print(f"Since the P-value ({p:.4f}) is less than the significance level ({alpha}), we reject the null hypothesis.")
                        print("There is a statistically significant association between the sorted cluster and the number of persons per frame (excluding 0 and 3 persons).")
                    else:
                        print(f"Since the P-value ({p:.4f}) is greater than or equal to the significance level ({alpha}), we fail to reject the null hypothesis.")
                        print("There is not enough evidence to suggest a statistically significant association between the sorted cluster and the number of persons per frame (excluding 0 and 3 persons).")
                else:
                    print("\nContingency table has less than 2 rows or columns after filtering. Cannot perform Chi-Square test.")

    else:
        print(f"Error: Required data for '{video_to_analyze}' not found. Please ensure 'Visual Data Analysis - Pose Variance' and 'K-Means Clustering' cells were run successfully for this video.")

elif not multiple_videos:
    print("Analysis can only be performed when 'multiple_videos' is True and 'livre-chega' is included in the analysis.")
else:
    print(f"Video '{video_to_analyze}' not found in the loaded data. Cannot generate analysis.")

In [ ]:
# @title Facial emotion recognition - Pie Charts per Video and Overall Combined

from collections import Counter # Useful for counting

# Assuming 'all_videos_data' is available when multiple_videos is True

# Dictionary to store the total count of each dominant emotion across ALL videos
overall_emotion_counts = Counter() # Using Counter is convenient for summing up counts

# Check if multiple_videos is True and all_videos_data is available
if multiple_videos:
    print("Analyzing facial emotion data (based on dominant emotion string) for ALL videos in 'all_videos_data'.")

    if 'all_videos_data' in locals():
        # Iterate through each video in the all_videos_data dictionary
        for video_name, video_data_df in all_videos_data.items():
            print(f"\n--- Processing video: {video_name} ---")

            # Dictionary to store the count of each dominant emotion for THIS video
            video_emotion_counts = Counter() # Use Counter for this video's counts too
            frame_count_with_fer = 0

            # Iterate through each row (frame) in the current video's DataFrame
            for index, row in video_data_df.iterrows():
                fers = row['fer']

                # Check if there is FER data for this frame
                if fers:
                    frame_count_with_fer += 1
                    # Process each detected face in this frame
                    for face_fer_data in fers:
                        # Check if the 'emotion' key exists and is a string
                        if 'emotion' in face_fer_data and isinstance(face_fer_data['emotion'], str):
                            dominant_emotion = face_fer_data['emotion']
                            # Increment the count for this emotion for THIS video
                            video_emotion_counts[dominant_emotion] += 1

            # --- Create and Display the Pie Chart for THIS Video ---

            # Check if any emotion data was collected for this video
            if video_emotion_counts:
                emotions = list(video_emotion_counts.keys())
                counts = list(video_emotion_counts.values())

                plt.figure(figsize=(10, 10)) # Set the figure size
                plt.pie(counts, labels=emotions, autopct='%1.1f%%', startangle=140) # autopct displays percentage, startangle rotates the start
                plt.title(f'Dominant Facial Emotion Distribution ({video_name})') # Set the title with video name
                plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.

                print(f"Analyzed {len(video_data_df)} frames for '{video_name}'. FER data found in {frame_count_with_fer} frames.")
                print(f"Dominant emotion counts for '{video_name}':")
                for emotion, count in video_emotion_counts.items():
                    print(f"{emotion}: {count}")

                # Show the plot for this video
                plt.show()

                # --- Accumulate counts for the overall chart ---
                # Add the counts from this video to the overall counts
                overall_emotion_counts.update(video_emotion_counts)


            else:
                print(f"No dominant emotion data found for video: {video_name}.")


        # --- Create and Display the FINAL Combined Pie Chart ---

        print("\n--- Creating Overall Combined Emotion Distribution Pie Chart ---")

        if overall_emotion_counts:
            overall_emotions = list(overall_emotion_counts.keys())
            overall_counts = list(overall_emotion_counts.values())

            plt.figure(figsize=(12, 12)) # Larger figure size for the combined chart
            plt.pie(overall_counts, labels=overall_emotions, autopct='%1.1f%%', startangle=140)
            plt.title('Overall Dominant Facial Emotion Distribution Across All Videos')
            plt.axis('equal')

            print("\nOverall dominant emotion counts across all videos:")
            for emotion, count in overall_emotion_counts.items():
                 print(f"{emotion}: {count}")

            # Show the combined plot
            plt.show()

        else:
            print("No overall dominant emotion data found across all videos.")


    else:
        print("Error: 'all_videos_data' not found. Please ensure multiple videos were loaded in a previous cell and 'multiple_videos' is True.")

elif 'data' in locals() and not multiple_videos:
     print("This code block is designed for multiple video analysis ('multiple_videos' is True).")
     print("To analyze a single video, ensure 'multiple_videos' is False and run the previous single video analysis code.")

else:
     print("Analysis skipped. Ensure 'multiple_videos' is set correctly and data ('data' or 'all_videos_data') is loaded.")

In [ ]:
# @title Visualize emotions and detected faces
#idx_to_class={0: 'Anger', 1: 'Contempt', 2: 'Disgust', 3: 'Fear', 4: 'Happiness', 5: 'Neutral', 6: 'Sadness', 7: 'Surprise'}

#fig = plt.figure()
#ax = fig.add_axes([0, 0, 1, 1])
#ax.imshow(img)

#width, height = img.size

#fers = data.iloc[frame]['fer']
#print(fers)

#for fer in fers:

#    x = fer['location'][0]
#    y = fer['location'][2]
#   width = fer['location'][1] - fer['location'][0]
#    height = fer['location'][3] - fer['location'][2]

#    print(fer['emotion'])

    # Create a rectangle patch
#    rect = patches.Rectangle((x, y), width, height, linewidth=1, edgecolor='g', facecolor='none')

    # Add patch to the image
#    ax.add_patch(rect)

#    ax.text(x, y, fer['emotion'],
#            verticalalignment='top',
#            color='g')

#plt.show()

## **- One Video -**

In [ ]:
# @title Load Video Data

video = "chega-be"

# Extract the data for the video
data = pd.read_pickle(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Features', video + '_visual.pkl'))

# Print first lines
print(f"Loading data for single video: {video}")
display(data.head())

# Display number of frames
print(f"\nNumber of frames: {len(data)}")

# A vector with the object position [x, y, width, height]
# A string with the object class (up to 80 possible classes)
# And the class probability score, which is indicative of the confidence of the detector

In [ ]:
# @title Access one Video and order it's frames

print(f"\n--- Processing video: {video_name} ---")

# Sort the DataFrame by the 'filename' column for the current video
# .sort_values(by='filename') sorts the DataFrame
# .reset_index(drop=True) resets the index after sorting and drops the old index
sorted_video_data = data.sort_values(by='filename').reset_index(drop=True)

# Print first lines to verify the order for the current video
print(f"Data after sorting by filename for {video_name}:")

# .head(5) gets the first 5 rows of the DataFrame
# ['filename'] selects the 'filename' column
# .tolist() converts the selected column to a list for easier printing
print(sorted_video_data['filename'].head(5).tolist())

# Print number of frames for the current video
print(f"\nNumber of frames for {video_name}: {len(sorted_video_data)}")

# Print info about the data
sorted_video_data.info()

# Print an example for each type of data
print("\n---Filename")
for i, row in sorted_video_data.iterrows():
    if row['filename']:  # Only print non-empty ones
        print(f"\n--- Frame {i} ---")
        print(row['filename'])
    if i >= 5:
        break

print("\n---Detections")
for i, row in sorted_video_data.iterrows():
    if row['detections']:  # Only print non-empty ones
        print(f"\n--- Frame {i} ---")
        print(row['detections'])
    if i >= 5:
        break

print("\n---Poses")
for i, row in sorted_video_data.iterrows():
    if row['poses']:  # Only print non-empty ones
        print(f"\n--- Frame {i} ---")
        print(row['poses'])
    if i >= 5:
        break

print("\n---Fer")
for i, row in sorted_video_data.iterrows():
    if row['fer']:  # Only print non-empty ones
        print(f"\n--- Frame {i} ---")
        print(row['fer'])
    if i >= 5:
        break


In [ ]:
# @title Create groups of Frames based on how much people are in the frame

grouped_frames = defaultdict(list)

for frame_index, fer in enumerate(sorted_video_data['fer']):
  if isinstance(fer, list):
    num_people = len(fer)
    grouped_frames[num_people].append(frame_index)
  elif isinstance(fer, dict):  # single face
      num_people = 1
  else:
    num_people = 0  # malformed or no detections

# Display results
for num_people, frames in grouped_frames.items():
    print(f"Frames with {num_people} people: {len(frames)} frames -> indices: {frames[:5]}{'...' if len(frames) > 5 else ''}")

In [ ]:
# @title Iterate over each group and extract features with tracklet assignment

def cosine_similarity(a, b):
  a = np.array(a)
  b = np.array(b)
  return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def iou(boxA, boxB):
  xA = max(boxA[0], boxB[0])
  yA = max(boxA[2], boxB[2])
  xB = min(boxA[1], boxB[1])
  yB = min(boxA[3], boxB[3])
  interW = max(0, xB - xA)
  interH = max(0, yB - yA)
  interArea = interW * interH
  boxAArea = max(0, (boxA[1] - boxA[0]) * (boxA[3] - boxA[2]))
  boxBArea = max(0, (boxB[1] - boxB[0]) * (boxB[3] - boxB[2]))
  union = boxAArea + boxBArea - interArea + 1e-8
  return interArea / union if union > 0 else 0

groupwise_data = {}

for num_people, frame_indices in grouped_frames.items():
  print(f"\n--- Processing group with {num_people} people ({len(frame_indices)} frames) ---")

  group_embeddings = []
  group_locations = []
  group_pose_vectors = []
  group_frame_indices_emb = []
  group_frame_indices_pose = []
  group_person_indices_in_frame = []
  group_tracklet_ids = []

  next_tracklet_id = 0
  last_frame_faces = []
  tracklet_id = 0

  for frame_index in frame_indices:
    fer_list = sorted_video_data.iloc[frame_index]['fer']
    pose_list = sorted_video_data.iloc[frame_index]['poses']

    if not isinstance(fer_list, list):
      continue

    current_faces = fer_list

    for i, face_data in enumerate(current_faces):
      if not (isinstance(face_data, dict) and 'embedding' in face_data and 'location' in face_data):
        continue

      emb = face_data['embedding']
      loc = face_data['location']

      if not (isinstance(emb, (list, np.ndarray)) and len(emb) > 0 and isinstance(loc, list) and len(loc) == 4):
        continue

      # Assign tracklet_id
      assigned = False
      for prev_face in last_frame_faces:
        prev_emb = prev_face.get('embedding')
        prev_box = prev_face.get('location')
        if prev_emb is None or prev_box is None:
          continue

        sim = cosine_similarity(emb, prev_emb)
        overlap = iou(loc, prev_box)
        if sim > 0.75 and overlap > 0.1:
          tracklet_id = prev_face['tracklet_id']
          assigned = True
          break

      if not assigned:
        tracklet_id = next_tracklet_id
        next_tracklet_id += 1

      face_data['tracklet_id'] = tracklet_id
      group_embeddings.append(emb)
      group_locations.append(loc)
      group_frame_indices_emb.append(frame_index)
      group_person_indices_in_frame.append(i)
      group_tracklet_ids.append(tracklet_id)

      last_frame_faces = [
        face for face in current_faces
        if isinstance(face, dict) and 'tracklet_id' in face
      ]

    if isinstance(pose_list, list):
      for pose_data in pose_list:
        group_pose_vectors.append(pose_data)
        group_frame_indices_pose.append(frame_index)

  groupwise_data[num_people] = {
    "embeddings": group_embeddings,
    "locations": group_locations,
    "frames_emb": group_frame_indices_emb,
    "frames_pose": group_frame_indices_pose,
    "person_indices": group_person_indices_in_frame,
    "poses": group_pose_vectors,
    "tracklet_ids": group_tracklet_ids
  }

  print(f"Extracted {len(group_embeddings)} embeddings for group with {num_people} people.")
  print(f"Extracted {len(group_locations)} locations for group with {num_people} people.")
  print(f"Extracted {len(group_frame_indices_emb)} frame values for group with {num_people} people.")
  print(f"Extracted {len(group_person_indices_in_frame)} person indices for group with {num_people} people.")
  print(f"Extracted {len(group_pose_vectors)} pose vectors for group with {num_people} people.")
  print(f"Assigned {next_tracklet_id} unique tracklet IDs.")

In [ ]:
# @title Visualising a Random Frame from Each Person Group

print("\n--- Visualising a Random Frame per Group ---")

for group_size, group_info in sorted(groupwise_data.items()):

  frame_indices = group_info["frames_emb"]
  print(f"Group with {group_size} people: {len(frame_indices)} frames")
  person_indices = group_info["person_indices"]
  print(f"  Person Indices in Frame: {person_indices}")
  locations = group_info["locations"]
  print(f"  Locations: {locations}")
  num_frames_to_display = min(1, len(frame_indices))  # Number of frames to display per group

  for num_frames_to_display in range(num_frames_to_display, 0, -1):

    if not frame_indices:
      print(f"Skipping group {group_size} (no frames)")
      continue

    # Pick a random entry from this group
    num_frames = min(50, len(frame_indices) - 1)
    rand_idx = random.randint(0, num_frames)
    frame_idx = frame_indices[rand_idx]
    person_idx_in_frame = person_indices[rand_idx]

    # Get frame metadata
    frame_row = sorted_video_data.iloc[frame_idx]
    filename = frame_row['filename']
    fer_list = frame_row.get('fer', [])

    image_path = os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Video_Frames', video, filename)

    try:
      img = Image.open(image_path)
    except FileNotFoundError:
      print(f"❌ Image not found: {image_path}")
      continue

    print(f"\n▶ Group with {group_size} people: Frame {frame_idx} - {filename}")
    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)

    # Draw each person's bounding box
    for i, face in enumerate(fer_list):
      if isinstance(face, dict) and 'location' in face:
        x1, x2, y1, y2 = face['location']
        # Calculate width and height
        width = x2 - x1
        height = y2 - y1

        rect = patches.Rectangle((x1, y1), width, height, linewidth=2, edgecolor='cyan', facecolor='none')
        ax.add_patch(rect)

        # Add text label with Person ID
        color = 'blue'
        ax.text(x1, y1, f"Person",
        verticalalignment='top', color='white',
        fontsize=10, weight='bold',
        bbox=dict(facecolor=color, alpha=0.6))

    ax.set_title(f"Random Frame from Group: {group_size} People")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.show()

print("\n✅ Finished visualising all groups.")

In [ ]:
# @title Use PCA for dimensionality reduction in all embeddings in different groups
print("\n--- Applying PCA to Embeddings in Each Group ---")

def embedding_pairwise_subtractions(embedding):
    """
    For a given 1D embedding vector, return all e_i - e_j for j > i.
    """
    diffs = []
    for i in range(len(embedding)):
        for j in range(i + 1, len(embedding)):
            diffs.append(embedding[i] - embedding[j])
    return np.array(diffs)

total_frames_in_video = len(sorted_video_data)
frame_to_visualize = np.random.randint(0, total_frames_in_video) # Randomly select a frame index

# Get frame information using the specified frame index and data
frame_id = data.iloc[frame_to_visualize]['filename']

# Construct the full path to the image file
image_path = os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Video_Frames', video, frame_id)
print(f"Image path: {image_path}")

# Open the image
try:
  img = Image.open(image_path)
except FileNotFoundError:
  print(f"Error: Image file not found at {image_path}. Please check the path and file name.")
  exit() # Stop execution if image is not found

width, height = img.size

# Adjust variance threshold if needed
variance_retained = 0.85
pca = PCA(n_components=variance_retained, svd_solver='full')

for group_size, group_info in groupwise_data.items():
  #
  embeddings = group_info.get("embeddings")
  if embeddings is None or len(embeddings) == 0:
      print(f"Skipping group {group_size} (no embeddings).")
      continue
  print(f"\n▶ Group {group_size} with {len(embeddings)} embeddings")
  embeddings_array = np.array(embeddings)
  reduced = pca.fit_transform(embeddings_array)
  reduced_embeddings = np.array(reduced)

  #
  pairwise_features = np.array([embedding_pairwise_subtractions(emb) for emb in embeddings])
  reduced_pw = pca.fit_transform(pairwise_features)
  reduced_pw_embeddings = np.array(reduced_pw)

  #
  poses = group_info.get("poses")
  if poses is None or len(poses) == 0:
      print(f"Skipping group {group_size} (no poses).")
      continue
  print(f"  Group {group_size} with {len(poses)} poses")
  num_poses = len(poses)
  poses_array = np.array(poses).reshape(num_poses, 165)
  print(f"  Shape of poses_array: {poses_array.shape}")
  reduced_poses = pca.fit_transform(poses_array)
  reduced_poses_array = np.array(reduced_poses)

  #Associate each pose to its embeddings
  extended_features = []

  locations = group_info.get("locations")     # Assuming list of dicts or arrays
  poses = group_info.get("poses")
  frames_pose = group_info.get("frames_pose")
  frames_emb = group_info.get("frames_emb")
  person_indices_pose = group_info.get("person_indices")
  person_indices_emb = group_info.get("person_indices")
  tracklet_ids = group_info.get("tracklet_ids")
  tracklet_ids = np.array(tracklet_ids).reshape(len(tracklet_ids), 1)

  # Initialise all extended embeddings with pose part as zeros
  pose_dim = reduced_poses_array.shape[1]
  # Store in group
  # Combine features
  print(f"reduced_embeddings shape: {len(reduced_embeddings)}")
  print(f"reduced_pw_embeddings shape: {len(reduced_pw_embeddings)}")
  print(f"tracklet_ids shape: {len(tracklet_ids)}")
  emb_features = np.hstack([reduced_embeddings, reduced_pw_embeddings])  # shape: (N, reduced_dim + 1)

  for emb_feature in emb_features:
    pose_pad = np.zeros(pose_dim)
    extended = np.concatenate([emb_feature, pose_pad])
    extended_features.append(extended)

  extended_features = np.array(extended_features)

  # Match each pose to its corresponding embedding if keypoint inside face box
  for pose, r_pose, f_pose in zip(poses, reduced_poses_array, frames_pose):
    pose = np.array(pose)  # Ensure pose is a NumPy array
    if pose.shape[1] < 2:
      continue  # Skip malformed pose

    x = pose[:, 0] * width
    y = pose[:, 1] * height

    for i, (emb, face, f_emb) in enumerate(zip(embeddings, locations, frames_emb)):
      # Optional: match frame & person index
      if f_pose != f_emb:
        continue

      # Extract bounding box
      x1, x2, y1, y2 = face['location'] if isinstance(face, dict) else face

      # Check if any keypoint lies inside the bounding box
      if np.any((x >= x1) & (x <= x2) & (y >= y1) & (y <= y2)):
        # Concatenate embedding and pose (flattened)
        extended_features[i, -pose_dim:] = r_pose
        break  # One match per pose

  # Save the final array
  group_info["reduced_features"] = extended_features # np.hstack([extended_features, tracklet_ids])

print("\n✅ PCA completed for all groups.")

In [ ]:
# @title  Using ISOMAP to separate all the embeddings in different possible persons (per group)

warnings.filterwarnings("ignore")

print("\n--- Performing Isomap for each group ---")

for group_size, group_info in groupwise_data.items():
  #if group_size != 3:
  #  continue
  features = group_info.get("reduced_features")

  if features is None or len(features) == 0:
    print(f"Skipping group {group_size} (empty features).")
    continue

  print(f"\n▶ Group {group_size} with {len(features)} instances")

  # n_neighbors = min(30, len(features) - 1)  # Ensure Isomap has valid neighbour count
  # max_n_neighbors = min(50, 30)  # Ensure Isomap has valid neighbour count
  # array_n_neighbors = np.arange(10, 60, 1)
  # array_n_neighbors = [int(np.sqrt(len(features)))]  # Wrap the single value in a list
  array_n_neighbors = [15]  # Wrap the single value in a list

  for n_neighbors in array_n_neighbors:
    print(f"  Trying n_neighbors={n_neighbors}")

    try:
      isomap = Isomap(n_components=2, n_neighbors=n_neighbors)
      print(f"  Running Isomap (n_neighbors={n_neighbors})...")
      embeddings_2d = isomap.fit_transform(features)
      print("  Done.")
      group_info["isomap_2d"] = embeddings_2d

      # Visualise
      plt.figure(figsize=(10, 8))
      plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=10, alpha=0.6)
      plt.title(f"Isomap: Group with {group_size} People per Frame")
      plt.xlabel("Isomap Component 1")
      plt.ylabel("Isomap Component 2")
      plt.grid(True, linestyle='--', alpha=0.5)
      plt.show()

      print(f"  2D shape: {embeddings_2d.shape}")
    except Exception as e:
      print(f"  ❌ Failed to run Isomap for group {group_size}: {e}")

In [ ]:
# @title Applying K-Means clustering on all groups for each projection mode

from sklearn.metrics import silhouette_score
from collections import Counter

# Define which projections to apply clustering on
modes = ['isomap_2d']

print("\n--- Applying K-Means clustering on all groups for each projection mode ---")

for mode in modes:
  print(f"\n=== MODE: {mode.upper()} ===")

  for group_size, group_info in groupwise_data.items():
    embeddings_2d = group_info.get(mode)

    if embeddings_2d is None or len(embeddings_2d) == 0:
      print(f"  Skipping group {group_size} (no data for {mode})")
      continue

    print(f"\n▶ Group {group_size} with {len(embeddings_2d)} points")

    # Number of clusters (can be dynamic)
    if group_size == 3:
      n_clusters = 3
    else:
      n_clusters = min(4, len(embeddings_2d))
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, max_iter=10000, init='k-means++')

    print(f"  Running K-Means with {n_clusters} clusters on {mode}...")
    clusters = kmeans.fit_predict(embeddings_2d)
    print("  Done.")

    # Save clusters
    group_info[f"{mode}_clusters"] = clusters

    # --- Visualise ---
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
      embeddings_2d[:, 0], embeddings_2d[:, 1],
      c=clusters, cmap='viridis', s=10, alpha=0.6
    )
    plt.title(f"K-Means Clustering on {mode.upper()} (Group Size: {group_size})")
    plt.xlabel(f"{mode} Component 1")
    plt.ylabel(f"{mode} Component 2")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(*scatter.legend_elements(), title="Cluster")
    plt.show()

    # Cluster sizes
    cluster_counts = Counter(clusters)
    print(f"  Cluster sizes:")
    for label in sorted(cluster_counts):
      print(f"    Cluster {label}: {cluster_counts[label]} points")

    # Silhouette score
    try:
      sil_score = silhouette_score(embeddings_2d, clusters)
    except ValueError:
      sil_score = 0.0  # If all points are in one cluster
    group_info[f"{mode}_silhouette"] = sil_score
    print(f"  ▶ Silhouette score: {sil_score:.3f} (estimated error: {1 - sil_score:.3f})")


In [ ]:
# @title Assigning person_id (cluster labels) back to sorted_video_data

print("\nAssigning person_id (cluster labels) back to sorted_video_data...")

successful_assignments = 0
mode = 'isomap_2d'

# Loop through all group sizes and their data
for group_size, group_data in groupwise_data.items():

  if not all(key in group_data for key in [f"{mode}_clusters", "frames_emb", "person_indices"]):
    print(f"Skipping group {group_size}: Missing keys.")
    continue

  labels = group_data[f"{mode}_clusters"]
  frame_indices = group_data["frames_emb"]
  person_indices = group_data["person_indices"]

  print(f"person_indices: {len(person_indices)}")

  if not (len(labels) == len(frame_indices) == len(person_indices)):
    print(f"Warning: Length mismatch in group {group_size}. Skipping.")
    continue

  successful_assignments = 0

  for i in range(len(labels)):
    frame_idx = frame_indices[i]
    person_idx = person_indices[i]
    label = labels[i]

    try:
      fer_list = sorted_video_data.iloc[frame_idx]['fer']
      if isinstance(fer_list, list) and person_idx < len(fer_list):
        face_dict = fer_list[person_idx]
        if isinstance(face_dict, dict):
          face_dict['person_id'] = int(label)
          successful_assignments += 1

          # If needed, write back the updated fer_list
          # sorted_video_data.at[frame_idx, 'fer'] = fer_list

    except Exception as e:
      print(f"Error at index {i}: {e}")

  print(f"Total successful assignments: {successful_assignments}")

print(f"✅ Successfully assigned {successful_assignments} person_ids across all groups.")


In [ ]:
# @title Matching clusters across groups using embedding distribution similarity
from sklearn.metrics import pairwise_distances
from scipy.optimize import linear_sum_assignment

print("\n--- Matching clusters across groups using embedding distribution similarity ---")

# Store global IDs to propagate across groups
global_id_counter = 0
cluster_to_global_id = {}  # (group_size, cluster_label) -> global_id

# Convert all cluster assignments into cluster-wise embeddings
cluster_embeddings_by_group = {}

for group_size, group_data in groupwise_data.items():
    labels = group_data.get("isomap_2d_clusters")
    embeddings = group_data.get("embeddings")

    if labels is None or embeddings is None:
        continue

    clusters = {}
    for idx, label in enumerate(labels):
        clusters.setdefault(label, []).append(embeddings[idx])
    cluster_embeddings_by_group[group_size] = clusters

# Sort group sizes to match smaller group sizes first
sorted_groups = sorted(cluster_embeddings_by_group.items(), key=lambda x: x[0])

# Initial matching using the first group as reference
reference_group_size, reference_clusters = sorted_groups[0]

for label, emb_list in reference_clusters.items():
    cluster_to_global_id[(reference_group_size, label)] = global_id_counter
    global_id_counter += 1

# Now match the rest of the groups
for i in range(1, len(sorted_groups)):
    current_group_size, current_clusters = sorted_groups[i]

    # Build cost matrix between current clusters and known global clusters
    cost_matrix = []
    current_labels = list(current_clusters.keys())
    reference_labels = list(cluster_to_global_id.keys())

    for cur_label in current_labels:
        cur_embs = np.vstack(current_clusters[cur_label])
        row = []
        for ref_key in reference_labels:
            ref_group, ref_label = ref_key
            ref_embs = np.vstack(cluster_embeddings_by_group[ref_group][ref_label])
            avg_dist = pairwise_distances(cur_embs, ref_embs).mean()
            row.append(avg_dist)
        cost_matrix.append(row)

    cost_matrix = np.array(cost_matrix)

    # Hungarian algorithm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    # Assign global IDs
    for i_row, i_col in zip(row_ind, col_ind):
        cur_label = current_labels[i_row]
        ref_key = reference_labels[i_col]
        global_id = cluster_to_global_id[ref_key]
        cluster_to_global_id[(current_group_size, cur_label)] = global_id

    # Assign new global IDs to unmatched clusters
    unmatched = set(current_labels) - {current_labels[i] for i in row_ind}
    for cur_label in unmatched:
        cluster_to_global_id[(current_group_size, cur_label)] = global_id_counter
        global_id_counter += 1

print(f"\n✅ Total global identities assigned: {len(set(cluster_to_global_id.values()))}")

In [ ]:
# @title Assigning global person_id (cluster labels) back to sorted_video_data
# print("\nAssigning global person_id (cluster labels) back to sorted_video_data...")
#
# successful_assignments = 0
# mode = 'isomap_2d'  # or whichever mode you're using
#
# for group_size, group_data in groupwise_data.items():
#
#     if not all(key in group_data for key in [f"{mode}_clusters", "frames_emb", "person_indices"]):
#         print(f"Skipping group {group_size}: Missing keys.")
#         continue
#
#     labels = group_data[f"{mode}_clusters"]
#     frame_indices = group_data["frames_emb"]
#     person_indices = group_data["person_indices"]
#
#     if not (len(labels) == len(frame_indices) == len(person_indices)):
#         print(f"Warning: Length mismatch in group {group_size}. Skipping.")
#         continue
#
#     for i in range(len(labels)):
#         frame_idx = frame_indices[i]
#         person_idx = person_indices[i]
#         local_label = labels[i]
#
#         global_id = cluster_to_global_id.get((group_size, local_label), -1)  # fallback ID if not found
#
#         try:
#             fer_list = sorted_video_data.iloc[frame_idx]['fer']
#             if isinstance(fer_list, list) and person_idx < len(fer_list):
#                 face_dict = fer_list[person_idx]
#                 if isinstance(face_dict, dict):
#                     face_dict['person_id'] = int(global_id)
#                     successful_assignments += 1
#         except Exception as e:
#             print(f"⚠️ Error at index {i} (frame {frame_idx}, person {person_idx}): {e}")
#
# print(f"✅ Successfully assigned {successful_assignments} global person_ids.")
#

In [ ]:
# @title Assign Global Person IDs & Visualise Them on 2D Embedding

from collections import Counter
from matplotlib import pyplot as plt
import matplotlib.patches as patches

mode = 'isomap_2d'  # Change to 'tsne_2d' or 'umap_2d' if needed

print("\nAssigning global person_id (cluster labels) back to sorted_video_data and visualising embeddings...")

successful_assignments = 0

for group_size, group_info in groupwise_data.items():

    if not all(key in group_info for key in [f"{mode}_clusters", "frames_emb", "person_indices", mode]):
        print(f"Skipping group {group_size}: Missing required keys.")
        continue

    labels = group_info[f"{mode}_clusters"]
    frame_indices = group_info["frames_emb"]
    person_indices = group_info["person_indices"]
    embeddings_2d = group_info[mode]

    if not (len(labels) == len(frame_indices) == len(person_indices) == len(embeddings_2d)):
        print(f"Warning: Length mismatch in group {group_size}. Skipping.")
        continue

    global_ids = []

    for i in range(len(labels)):
        frame_idx = frame_indices[i]
        person_idx = person_indices[i]
        local_label = labels[i]

        global_id = cluster_to_global_id.get((group_size, local_label), -1)  # fallback if missing
        global_ids.append(global_id)

        try:
            fer_list = sorted_video_data.iloc[frame_idx]['fer']
            if isinstance(fer_list, list) and person_idx < len(fer_list):
                face_dict = fer_list[person_idx]
                if isinstance(face_dict, dict):
                    face_dict['person_id'] = int(global_id)
                    successful_assignments += 1
        except Exception as e:
            print(f"⚠️ Error at index {i} (frame {frame_idx}, person {person_idx}): {e}")

    # Store the global_ids for visualisation
    group_info["global_ids"] = global_ids

    # --- Visualise ---
    print(f"\n▶ Visualising Group {group_size} with {len(embeddings_2d)} points")

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
        embeddings_2d[:, 0], embeddings_2d[:, 1],
        c=global_ids, cmap='tab10', s=10, alpha=0.6
    )
    plt.title(f"Global Person IDs on {mode.upper()} Embedding (Group Size: {group_size})")
    plt.xlabel(f"{mode} Component 1")
    plt.ylabel(f"{mode} Component 2")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(*scatter.legend_elements(), title="Person ID")
    plt.show()

    cluster_counts = Counter(global_ids)
    print(f"  Global ID sizes:")
    for label in sorted(cluster_counts):
        print(f"    Person {label}: {cluster_counts[label]} points")

print(f"\n✅ Successfully assigned {successful_assignments} global person_ids and visualised them.")

In [ ]:
# @title Visualize Person ID in Images (based on clustered facial embeddings)

# Choose video
# Assuming 'video' variable is set from a previous cell for single video mode
# If in multiple_videos mode, you might want to select a specific video from all_videos_data
# For this example, let's continue using the 'video' variable from single video analysis
# Ensure 'multiple_videos' is False and 'data' is loaded for single video analysis

if not multiple_videos and 'sorted_video_data' in locals():
    video_to_visualize = video # Use the video name defined for single video mode
    video_data = sorted_video_data # Use the sorted data for the single video
    frames_path = f"/content/drive/MyDrive/PBD/Project/Project_Data/Video_Frames/{video_to_visualize}"

    print(f"Visualizing assigned Person IDs for video: {video_to_visualize}")

    # Display the first 30 frames
    num_frames_to_display = min(1000, len(video_data))
    print(f"Displaying the first {num_frames_to_display} frames.")

    for frame_index in range(0, num_frames_to_display, 10):
        frame_row = video_data.iloc[frame_index]
        frame_id = frame_row['filename']
        fers_data = frame_row['fer'] # Get the Face Emotion Recognition (FER) data

        # Load image
        image_path = os.path.join(frames_path, frame_id)
        try:
            img = Image.open(image_path)
        except FileNotFoundError:
            print(f"❌ Image not found: {image_path}")
            continue

        # Set up plot
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(img)

        # Draw bounding boxes and display Person ID for detected faces
        if isinstance(fers_data, list): # Check if there's a list of faces
            for face_data in fers_data:
                # Check if face data is a dictionary and contains 'location' and 'person_id'
                if isinstance(face_data, dict) and 'location' in face_data and 'person_id' in face_data:
                    try:
                        # Extract location and person ID
                        x1, x2, y1, y2 = face_data['location']
                        person_id = face_data['person_id']

                        # Calculate width and height
                        width = x2 - x1
                        height = y2 - y1

                        # Create a rectangle patch (using a color map for different persons)
                        # Using a colormap helps distinguish persons visually
                        # You might need to adjust cmap and normalization based on the range of your person_ids
                        cmap = plt.get_cmap('tab10') # Example colormap
                        color = cmap(person_id % cmap.N) # Assign color based on person_id

                        rect = patches.Rectangle((x1, y1), width, height, linewidth=2, edgecolor=color, facecolor='none')
                        ax.add_patch(rect)

                        # Add text label with Person ID
                        ax.text(x1, y1, f"Person {person_id}",
                                verticalalignment='top', color='white',
                                fontsize=10, weight='bold',
                                bbox=dict(facecolor=color, alpha=0.6))

                    except (TypeError, ValueError, IndexError) as e:
                         print(f"⚠️ Skipped invalid face data format in frame {frame_id}: {face_data}. Error: {e}")


        # Formatting
        ax.set_title(f"{video_to_visualize.upper()} – Frame {frame_index} ({frame_id}) (Person IDs)", fontsize=12, weight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()

    print("\nVisualization complete.")

elif multiple_videos:
    print("This visualization is currently set up for single video analysis.")
    print("To run this for multiple videos, you would need to select a specific video from 'all_videos_data' and iterate through its frames.")

else:
    print("Analysis skipped. Ensure 'multiple_videos' is False and 'sorted_video_data' is available from previous single video analysis.")

In [ ]:
# @title Statistic
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import pandas as pd

# --- Tracking ---
person_frame_count = defaultdict(set)
person_emotion_count = defaultdict(Counter)

for frame_idx, row in sorted_video_data.iterrows():
    fer_list = row.get('fer', [])

    if isinstance(fer_list, list):
        for face in fer_list:
            if isinstance(face, dict) and 'person_id' in face:
                person_id = face['person_id']
                emotion = face.get('emotion', 'unknown')

                person_frame_count[person_id].add(frame_idx)
                person_emotion_count[person_id][emotion] += 1

# --- Convert to counts & sort ---
person_appearance_counts = {
    pid: len(frames) for pid, frames in person_frame_count.items()
}
sorted_appearance = sorted(person_appearance_counts.items(), key=lambda x: x[1], reverse=True)

# --- Emotion with most occurrences per person ---
dominant_emotion = {
    pid: emotions.most_common(1)[0][0] for pid, emotions in person_emotion_count.items()
}

# --- Console Output ---
print("\n Number of frames each person appears in (sorted):")
for pid, count in sorted_appearance:
    print(f"Person {pid}: {count} frames — Most frequent emotion: {dominant_emotion[pid]}")

# --- Bar Plot: Frame Appearances ---
df = pd.DataFrame(sorted_appearance, columns=["person_id", "frame_count"])
plt.figure(figsize=(10, 6))
plt.bar(df['person_id'].astype(str), df['frame_count'], color='steelblue')
plt.xticks(rotation=45)
plt.xlabel("Person ID")
plt.ylabel("Number of Frames")
plt.title("Frame Count per Person (Sorted)")
plt.tight_layout()
plt.show()

#
top_n = 5

for pid in sorted(person_emotion_count):
    emotion_counts = person_emotion_count[pid]
    top_emotions = emotion_counts.most_common(top_n)

    print(f"\n Person {pid} – Top {top_n} emotions:")
    for emo, cnt in top_emotions:
        print(f"   {emo}: {cnt}")

    # Bar plot
    emotions, counts = zip(*top_emotions)
    plt.figure(figsize=(6, 4))
    plt.bar(emotions, counts, color='orchid')
    plt.title(f"Top {top_n} Emotions – Person {pid}")
    plt.ylabel("Count")
    plt.xlabel("Emotion")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

# Aggregate all emotions across all people
global_emotions = Counter()
for counts in person_emotion_count.values():
    global_emotions.update(counts)

top_global = global_emotions.most_common(top_n)
print("\n🌍 Global Top 5 Emotions:")
for emo, cnt in top_global:
    print(f"   {emo}: {cnt}")

# Bar plot
emotions, counts = zip(*top_global)
plt.figure(figsize=(6, 4))
plt.bar(emotions, counts, color='teal')
plt.title(f"Global Top {top_n} Emotions")
plt.ylabel("Count")
plt.xlabel("Emotion")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

combined_emotions = Counter()
for pid in [1, 2]:
    combined_emotions.update(person_emotion_count.get(pid, Counter()))

top_combined = combined_emotions.most_common(top_n)
print("\n Top 5 Emotions (Person 1 + Person 2):")
for emo, cnt in top_combined:
    print(f"   {emo}: {cnt}")

# Bar plot
emotions, counts = zip(*top_combined)
plt.figure(figsize=(6, 4))
plt.bar(emotions, counts, color='coral')
plt.title(f"Top {top_n} Emotions – Person 1 & 2 Combined")
plt.ylabel("Count")
plt.xlabel("Emotion")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# **-- ALL VIDEOS People Identification --**

In [ ]:
# @title Run this 1st (auxiliary Functions)
def cosine_similarity(a, b):
  a = np.array(a)
  b = np.array(b)
  return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def iou(boxA, boxB):
  xA = max(boxA[0], boxB[0])
  yA = max(boxA[2], boxB[2])
  xB = min(boxA[1], boxB[1])
  yB = min(boxA[3], boxB[3])
  interW = max(0, xB - xA)
  interH = max(0, yB - yA)
  interArea = interW * interH
  boxAArea = max(0, (boxA[1] - boxA[0]) * (boxA[3] - boxA[2]))
  boxBArea = max(0, (boxB[1] - boxB[0]) * (boxB[3] - boxB[2]))
  union = boxAArea + boxBArea - interArea + 1e-8
  return interArea / union if union > 0 else 0

def embedding_pairwise_subtractions(embedding):
    """
    For a given 1D embedding vector, return all e_i - e_j for j > i.
    """
    diffs = []
    for i in range(len(embedding)):
        for j in range(i + 1, len(embedding)):
            diffs.append(embedding[i] - embedding[j])
    return np.array(diffs)

In [ ]:
# @title Distinguish between persons in the frames

# Final output: video -> all stats
all_video_stats = {}

# Path to the features folder
features_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features'
visual_files = [f for f in os.listdir(features_path) if f.endswith('_visual.pkl')]

# aux_asdadadssrdwa = 0
for file in visual_files:
  video = file.replace('_visual.pkl', '')
  # if aux_asdadadssrdwa == 1:
  #   break
  # else:
  #   aux_asdadadssrdwa = 1

  print("-----------------------------------------------------------------------------------------------------")
  print(f"Processing video: {video}")
  print("-----------------------------------------------------------------------------------------------------")

  # Extract the data for the video
  data = pd.read_pickle(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Features', video + '_visual.pkl'))
  sorted_video_data = data.sort_values(by='filename').reset_index(drop=True)

  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
  # Create groups of Frames based on how much people are in the frame #
  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

  grouped_frames = defaultdict(list)

  for frame_index, fer in enumerate(sorted_video_data['fer']):
    if isinstance(fer, list):
      num_people = len(fer)
    elif isinstance(fer, dict):  # single face
        num_people = 1
    else:
      num_people = 0  # malformed or no detections
    grouped_frames[num_people].append(frame_index)

  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
  # Iterate over each group and extract features with tracklet assignment #
  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

  groupwise_data = {}

  count_aux = 0

  grouped_frames = {k: v for k, v in grouped_frames.items() if k in [2, 3, 4]}

  for num_people, frame_indices in grouped_frames.items():
    group_embeddings = []
    group_locations = []
    group_pose_vectors = []
    group_frame_indices_emb = []
    group_frame_indices_pose = []
    group_person_indices_in_frame = []
    group_tracklet_ids = []

    for frame_index in frame_indices:
      fer_list = sorted_video_data.iloc[frame_index]['fer']
      pose_list = sorted_video_data.iloc[frame_index]['poses']

      if not isinstance(fer_list, list):
        continue

      current_faces = fer_list

      for i, face_data in enumerate(current_faces):
        if not (isinstance(face_data, dict) and 'embedding' in face_data and 'location' in face_data):
          continue

        emb = face_data['embedding']
        loc = face_data['location']

        if not (isinstance(emb, (list, np.ndarray)) and len(emb) > 0 and isinstance(loc, list) and len(loc) == 4):
          continue

        group_embeddings.append(emb)
        group_locations.append(loc)
        group_frame_indices_emb.append(frame_index)
        group_person_indices_in_frame.append(i)

        last_frame_faces = [
          face for face in current_faces
          if isinstance(face, dict) and 'tracklet_id' in face
        ]

      if isinstance(pose_list, list):
        for pose_data in pose_list:
          group_pose_vectors.append(pose_data)
          group_frame_indices_pose.append(frame_index)

    groupwise_data[num_people] = {
      "embeddings": group_embeddings,
      "locations": group_locations,
      "frames_emb": group_frame_indices_emb,
      "frames_pose": group_frame_indices_pose,
      "person_indices": group_person_indices_in_frame,
      "poses": group_pose_vectors,
      "tracklet_ids": group_tracklet_ids
    }

  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
  # Use PCA for dimensionality reduction in all embeddings in different groups  #
  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

  total_frames_in_video = len(sorted_video_data)
  frame_to_visualize = np.random.randint(0, total_frames_in_video) # Randomly select a frame index

  # Get frame information using the specified frame index and data
  frame_id = data.iloc[frame_to_visualize]['filename']

  # Construct the full path to the image file
  image_path = os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Video_Frames', video, frame_id)

  # Open the image
  try:
    img = Image.open(image_path)
  except FileNotFoundError:
    print(f"Error: Image file not found at {image_path}. Please check the path and file name.")
    exit() # Stop execution if image is not found

  width, height = img.size

  # Adjust variance threshold if needed
  variance_retained = 0.85
  pca = PCA(n_components=variance_retained, svd_solver='full')

  for group_size, group_info in groupwise_data.items():
    #
    embeddings = group_info.get("embeddings")
    if embeddings is None or len(embeddings) == 0:
        print(f"Skipping group {group_size} (no embeddings).")
        continue
    embeddings_array = np.array(embeddings)
    reduced = pca.fit_transform(embeddings_array)
    reduced_embeddings = np.array(reduced)

    #
    pairwise_features = np.array([embedding_pairwise_subtractions(emb) for emb in embeddings])
    reduced_pw = pca.fit_transform(pairwise_features)
    reduced_pw_embeddings = np.array(reduced_pw)

    #
    poses = group_info.get("poses")
    if poses is None or len(poses) == 0:
        print(f"Skipping group {group_size} (no poses).")
        continue
    num_poses = len(poses)
    poses_array = np.array(poses).reshape(num_poses, 165)
    reduced_poses = pca.fit_transform(poses_array)
    reduced_poses_array = np.array(reduced_poses)

    #Associate each pose to its embeddings
    extended_features = []

    locations = group_info.get("locations")     # Assuming list of dicts or arrays
    poses = group_info.get("poses")
    frames_pose = group_info.get("frames_pose")
    frames_emb = group_info.get("frames_emb")
    person_indices_pose = group_info.get("person_indices")
    person_indices_emb = group_info.get("person_indices")
    tracklet_ids = group_info.get("tracklet_ids")
    tracklet_ids = np.array(tracklet_ids).reshape(len(tracklet_ids), 1)

    # Initialise all extended embeddings with pose part as zeros
    pose_dim = reduced_poses_array.shape[1]
    # Store in group
    # Combine features
    emb_features = np.hstack([reduced_embeddings, reduced_pw_embeddings])  # shape: (N, reduced_dim + 1)

    for emb_feature in emb_features:
      pose_pad = np.zeros(pose_dim)
      extended = np.concatenate([emb_feature, pose_pad])
      extended_features.append(extended)

    extended_features = np.array(extended_features)

    # Match each pose to its corresponding embedding if keypoint inside face box
    for pose, r_pose, f_pose in zip(poses, reduced_poses_array, frames_pose):
      pose = np.array(pose)  # Ensure pose is a NumPy array
      if pose.shape[1] < 2:
        continue  # Skip malformed pose

      x = pose[:, 0] * width
      y = pose[:, 1] * height

      for i, (emb, face, f_emb) in enumerate(zip(embeddings, locations, frames_emb)):
        # Optional: match frame & person index
        if f_pose != f_emb:
          continue

        # Extract bounding box
        x1, x2, y1, y2 = face['location'] if isinstance(face, dict) else face

        # Check if any keypoint lies inside the bounding box
        if np.any((x >= x1) & (x <= x2) & (y >= y1) & (y <= y2)):
          # Concatenate embedding and pose (flattened)
          extended_features[i, -pose_dim:] = r_pose
          break  # One match per pose

    # Save the final array
    group_info["reduced_features"] = extended_features # np.hstack([extended_features, tracklet_ids])

  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
  # Using ISOMAP to separate all the embeddings in different possible persons (per group) #
  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

  warnings.filterwarnings("ignore")

  for group_size, group_info in groupwise_data.items():
    #if group_size != 3:
    #  continue
    features = group_info.get("reduced_features")

    if features is None or len(features) == 0:
      print(f"Skipping group {group_size} (empty features).")
      continue

    # n_neighbors = min(30, len(features) - 1)  # Ensure Isomap has valid neighbour count
    # max_n_neighbors = min(50, 30)  # Ensure Isomap has valid neighbour count
    # array_n_neighbors = np.arange(10, 60, 1)
    # array_n_neighbors = [int(np.sqrt(len(features)))]  # Wrap the single value in a list
    array_n_neighbors = [15]  # Wrap the single value in a list

    for n_neighbors in array_n_neighbors:
      try:
        isomap = Isomap(n_components=2, n_neighbors=n_neighbors)
        embeddings_2d = isomap.fit_transform(features)
        group_info["isomap_2d"] = embeddings_2d
        print(f"Successfully applied Isomap for group {group_size} with n_neighbors={n_neighbors}")

      except Exception as e:
        print(f"Failed to run Isomap for group {group_size}: {e}")

  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
  # Applying K-Means clustering on all groups for each projection mode  #
  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

  # Define which projections to apply clustering on
  modes = ['isomap_2d']

  for mode in modes:
    for group_size, group_info in groupwise_data.items():
      embeddings_2d = group_info.get(mode)

      if embeddings_2d is None or len(embeddings_2d) == 0:
        print(f"  Skipping group {group_size} (no data for {mode})")
        continue

      # Number of clusters (can be dynamic)
      if group_size == 3:
        n_clusters = 3
      else:
        n_clusters = min(4, len(embeddings_2d))
      kmeans = KMeans(n_clusters=n_clusters, random_state=42, max_iter=10000, init='k-means++')
      clusters = kmeans.fit_predict(embeddings_2d)

      # Save clusters
      group_info[f"{mode}_clusters"] = clusters

      # Cluster sizes
      cluster_counts = Counter(clusters)

      # Silhouette score
      try:
        sil_score = silhouette_score(embeddings_2d, clusters)
      except ValueError:
        sil_score = 0.0  # If all points are in one cluster
      group_info[f"{mode}_silhouette"] = sil_score

  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
  # Assigning person_id (cluster labels) back to sorted_video_data  #
  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

  successful_assignments = 0
  mode = 'isomap_2d'

  # Loop through all group sizes and their data
  for group_size, group_data in groupwise_data.items():

    if not all(key in group_data for key in [f"{mode}_clusters", "frames_emb", "person_indices"]):
      print(f"Skipping group {group_size}: Missing keys.")
      continue

    labels = group_data[f"{mode}_clusters"]
    frame_indices = group_data["frames_emb"]
    person_indices = group_data["person_indices"]

    if not (len(labels) == len(frame_indices) == len(person_indices)):
      print(f"Warning: Length mismatch in group {group_size}. Skipping.")
      continue

    successful_assignments = 0

    for i in range(len(labels)):
      frame_idx = frame_indices[i]
      person_idx = person_indices[i]
      label = labels[i]

      try:
        fer_list = sorted_video_data.iloc[frame_idx]['fer']
        if isinstance(fer_list, list) and person_idx < len(fer_list):
          face_dict = fer_list[person_idx]
          if isinstance(face_dict, dict):
            face_dict['person_id'] = int(label)
            successful_assignments += 1

      except Exception as e:
        print(f"Error at index {i}: {e}")

  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
  # Matching clusters across groups using embedding distribution similarity #
  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

  # Store global IDs to propagate across groups
  global_id_counter = 0
  cluster_to_global_id = {}  # (group_size, cluster_label) -> global_id

  # Convert all cluster assignments into cluster-wise embeddings
  cluster_embeddings_by_group = {}

  for group_size, group_data in groupwise_data.items():
    labels = group_data.get("isomap_2d_clusters")
    embeddings = group_data.get("embeddings")

    if labels is None or embeddings is None:
      continue

    clusters = {}
    for idx, label in enumerate(labels):
      clusters.setdefault(label, []).append(embeddings[idx])
    cluster_embeddings_by_group[group_size] = clusters

  # Sort group sizes to match smaller group sizes first
  sorted_groups = sorted(cluster_embeddings_by_group.items(), key=lambda x: x[0])

  if not sorted_groups:
    print("❌ No groups found in cluster_embeddings_by_group. Skipping global ID assignment.")
    continue  # or return, depending on where this block is

  # Initial matching using the first group as reference
  reference_group_size, reference_clusters = sorted_groups[0]

  for label, emb_list in reference_clusters.items():
      cluster_to_global_id[(reference_group_size, label)] = global_id_counter
      global_id_counter += 1

  # Now match the rest of the groups
  for i in range(1, len(sorted_groups)):
      current_group_size, current_clusters = sorted_groups[i]

      # Build cost matrix between current clusters and known global clusters
      cost_matrix = []
      current_labels = list(current_clusters.keys())
      reference_labels = list(cluster_to_global_id.keys())

      for cur_label in current_labels:
          cur_embs = np.vstack(current_clusters[cur_label])
          row = []
          for ref_key in reference_labels:
              ref_group, ref_label = ref_key
              ref_embs = np.vstack(cluster_embeddings_by_group[ref_group][ref_label])
              avg_dist = pairwise_distances(cur_embs, ref_embs).mean()
              row.append(avg_dist)
          cost_matrix.append(row)

      cost_matrix = np.array(cost_matrix)

      # Hungarian algorithm
      row_ind, col_ind = linear_sum_assignment(cost_matrix)

      # Assign global IDs
      for i_row, i_col in zip(row_ind, col_ind):
          cur_label = current_labels[i_row]
          ref_key = reference_labels[i_col]
          global_id = cluster_to_global_id[ref_key]
          cluster_to_global_id[(current_group_size, cur_label)] = global_id

      # Assign new global IDs to unmatched clusters
      unmatched = set(current_labels) - {current_labels[i] for i in row_ind}
      for cur_label in unmatched:
          cluster_to_global_id[(current_group_size, cur_label)] = global_id_counter
          global_id_counter += 1

  # # # # # # # # # # # # # # #
  # Assign Global Person IDs  #
  # # # # # # # # # # # # # # #

  successful_assignments = 0
  mode = 'isomap_2d'

  for group_size, group_info in groupwise_data.items():

      if not all(key in group_info for key in [f"{mode}_clusters", "frames_emb", "person_indices", mode]):
          print(f"Skipping group {group_size}: Missing required keys.")
          continue

      labels = group_info[f"{mode}_clusters"]
      frame_indices = group_info["frames_emb"]
      person_indices = group_info["person_indices"]
      embeddings_2d = group_info[mode]

      if not (len(labels) == len(frame_indices) == len(person_indices) == len(embeddings_2d)):
          print(f"Warning: Length mismatch in group {group_size}. Skipping.")
          continue

      global_ids = []

      for i in range(len(labels)):
          frame_idx = frame_indices[i]
          person_idx = person_indices[i]
          local_label = labels[i]

          global_id = cluster_to_global_id.get((group_size, local_label), -1)  # fallback if missing
          global_ids.append(global_id)

          try:
              fer_list = sorted_video_data.iloc[frame_idx]['fer']
              if isinstance(fer_list, list) and person_idx < len(fer_list):
                  face_dict = fer_list[person_idx]
                  if isinstance(face_dict, dict):
                      embedding_2d = group_info["isomap_2d"][i]
                      face_dict['person_id'] = int(global_id)
                      face_dict['embedding_2d'] = embedding_2d
                      successful_assignments += 1
          except Exception as e:
              print(f" Error at index {i} (frame {frame_idx}, person {person_idx}): {e}")

      # Store the global_ids for visualisation
      group_info["global_ids"] = global_ids

      cluster_counts = Counter(global_ids)

  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
  # Save all the obtained data obtained from the treatment of the given features  #
  # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

  person_frame_count = defaultdict(set)
  person_emotion_count = defaultdict(Counter)
  frame_info = {}

  for frame_idx, row in sorted_video_data.iterrows():
      fer_list = row.get('fer', [])
      frame_people = []

      if isinstance(fer_list, list):
          for face in fer_list:
              if isinstance(face, dict) and 'person_id' in face:
                  pid = face['person_id']
                  embedding = face_dict['embedding_2d']
                  emotion = face.get('emotion', 'unknown')
                  cluster = face.get('cluster', None)
                  location = face.get('location', None)

                  # Count frames and emotions
                  person_frame_count[pid].add(frame_idx)
                  person_emotion_count[pid][emotion] += 1

                  # Store per-face data for this frame
                  frame_people.append({
                      "person_id": pid,
                      "embedding": embedding,
                      "emotion": emotion,
                      "cluster": cluster,
                      "location": location,
                  })

      frame_info[frame_idx] = frame_people

  # Save everything per video
  all_video_stats[video] = {
      "person_frame_count": person_frame_count,
      "person_emotion_count": person_emotion_count,
      "frame_info": frame_info
  }

  groupwise_data = {}

print("--- Finished assigning global person IDs. ---")

In [ ]:
# @title Save The data obtained for all videos
save_dir = "/content/drive/MyDrive/PBD/Project/All_video_stats"  # Replace with your actual folder path
os.makedirs(save_dir, exist_ok=True)

with open(os.path.join(save_dir, "all_video_stats.pkl"), 'wb') as f:
    pickle.dump(all_video_stats, f)

# Save each video's stats
for video_name, stats in all_video_stats.items():
    filename = f"{video_name}_stats.pkl"
    filepath = os.path.join(save_dir, filename)

    with open(filepath, 'wb') as f:
        pickle.dump(stats, f)

    print(f"Saved: {filename}")

In [ ]:
# @title Load The data obtained for all videos
with open('/content/drive/MyDrive/PBD/Project/All_video_stats/all_video_stats.pkl', 'rb') as f:
    all_video_stats = pickle.load(f)

In [ ]:
# @title Assign a role to each person (Moderator/Party name/Gesture)

import itertools

def total_dispersion(embeddings_list):
  mean_emb = np.mean(np.vstack(embeddings_list), axis=0)
  return sum(np.linalg.norm(emb - mean_emb) ** 2 for emb in embeddings_list)

def assign_roles_bruteforce(all_video_stats, party):
  video_candidates = {}

  for video, stats in all_video_stats.items():
    frame_info = stats.get("frame_info", {})
    frame_counts = stats.get("person_frame_count", {})
    name = video.lower()

    video_parties = name.replace(".pkl", "").replace("_visual", "").split("-")
    video_parties = [p.upper() for p in video_parties]

    if party not in video_parties:
      continue

    if not frame_counts:
      continue

    # Assign fixed roles: Moderator and Gestural Language
    sorted_counts = sorted(frame_counts.items(), key=lambda x: len(x[1]))
    least_pid = sorted_counts[0][0]
    most_pid = sorted_counts[-1][0]

    stats.setdefault("role_assignment", {})
    stats["role_assignment"][least_pid] = "Moderator"
    stats["role_assignment"][most_pid] = "Gestural Language"

    # Only consider the remaining person_ids for party role
    assigned_pids = set(stats.get("role_assignment", {}).keys())
    remaining_pids = [pid for pid in frame_counts if pid not in assigned_pids]

    embeddings_by_person = defaultdict(list)
    for frame in frame_info.values():
      for person in frame:
        pid = person.get("person_id")
        embedding = person.get("embedding")
        if embedding is not None and pid in remaining_pids:
            embeddings_by_person[pid].append(np.array(embedding))

    avg_embs = {
      pid: np.mean(np.vstack(embs), axis=0)
      for pid, embs in embeddings_by_person.items()
      if len(embs) > 0
    }

    if len(avg_embs) == 1:  # Only one unassigned person remains
      only_pid = list(avg_embs.keys())[0]

      # Find the remaining unassigned party in this video
      assigned_roles = set(stats["role_assignment"].values())
      print(assigned_roles)
      remaining_roles = set(video_parties) - assigned_roles
      print(remaining_roles)
      if len(remaining_roles) == 1:
        remaining_party = remaining_roles.pop()
        stats["role_assignment"][only_pid] = remaining_party
    elif len(avg_embs) == 2:
      video_candidates[video] = avg_embs

  if not video_candidates:
    print(f"No valid videos found for party {party}")
    return

  videos = list(video_candidates.keys())
  candidates_lists = [list(video_candidates[v].keys()) for v in videos]

  best_score = float('inf')
  best_assignment = None

  for assignment in itertools.product(*candidates_lists):
    embeddings_list = [video_candidates[v][pid] for v, pid in zip(videos, assignment)]
    score = total_dispersion(embeddings_list)
    if score < best_score:
      best_score = score
      best_assignment = assignment

  if best_assignment:
    for v, pid in zip(videos, best_assignment):
      all_video_stats[v].setdefault("role_assignment", {})[pid] = party

  # === Assign leftover roles if some person_ids are still unassigned ===
  for video in video_candidates:
    stats = all_video_stats[video]
    assigned_roles = set(stats["role_assignment"].values())
    frame_counts = stats.get("person_frame_count", {})
    all_pids = set(frame_counts.keys())
    assigned_pids = set(stats["role_assignment"].keys())
    unassigned_pids = all_pids - assigned_pids

    name = video.lower()
    video_parties = name.replace(".pkl", "").replace("_visual", "").replace(".pkl", "").split("-")
    video_parties = [p.upper() for p in video_parties]
    unassigned_roles = set(video_parties) - assigned_roles

    for pid, role in zip(unassigned_pids, unassigned_roles):
      stats["role_assignment"][pid] = role

# Automatically extract unique parties from all video names
party_set = set()
for video in all_video_stats:
  name = video.lower()
  video_parties = name.replace(".pkl", "").replace("_visual", "").split("-")
  party_set.update(p.upper() for p in video_parties)

# Assign party roles across videos
for party in sorted(party_set):
  assign_roles_bruteforce(all_video_stats, party)

print("\nSummary of Assigned Roles for All Videos\n" + "-"*50)

for video, stats in all_video_stats.items():
  print(f"\nVideo: {video}")
  role_assignment = stats.get("role_assignment", {})
  frame_counts = stats.get("person_frame_count", {})

  if not role_assignment:
    print("No roles assigned.")
    continue

  for pid, role in role_assignment.items():
    count = len(frame_counts.get(pid, []))
    print(f"Person {pid:<3} → {role:<20} | {count} frames")


In [ ]:
# @title Visual Data Analysis (Emotions by Role)

top_n = 5  # Number of top emotions to show

for video_name, stats in all_video_stats.items():
  print(f"\n🎞️ Video: {video_name}")

  role_map = stats.get("role_assignment", {})
  person_frame_count = stats["person_frame_count"]
  person_emotion_count = stats["person_emotion_count"]

  # Map roles to IDs (fallback to 'Person X' if no role assigned)
  def get_role(pid):
    return role_map.get(pid, f"Person {pid}")

  # --- Convert to counts & sort ---
  person_appearance_counts = {
    get_role(pid): len(frames) for pid, frames in person_frame_count.items()
  }
  sorted_appearance = sorted(person_appearance_counts.items(), key=lambda x: x[1], reverse=True)

  # --- Emotion with most occurrences per person ---
  dominant_emotion = {
    get_role(pid): emotions.most_common(1)[0][0] for pid, emotions in person_emotion_count.items()
  }

  # --- Console Output ---
  print("\n📊 Number of frames each person appears in (sorted):")
  for role, count in sorted_appearance:
    print(f"  {role:<20} — {count:>4} frames — Most frequent emotion: {dominant_emotion[role]}")

  # --- Bar Plot: Frame Appearances ---
  df = pd.DataFrame(sorted_appearance, columns=["role", "frame_count"])
  plt.figure(figsize=(10, 6))
  plt.bar(df['role'], df['frame_count'], color='steelblue')
  plt.xticks(rotation=45)
  plt.xlabel("Role")
  plt.ylabel("Number of Frames")
  plt.title(f"{video_name} – Frame Count per Role")
  plt.tight_layout()
  plt.show()

In [ ]:
# @title Top 5 Emotions for each Party

from collections import defaultdict

top_n = 5  # Number of top emotions to show

role_emotion_totals = defaultdict(Counter)

for stats in all_video_stats.values():
    role_map = stats.get("role_assignment", {})
    person_emotion_count = stats.get("person_emotion_count", {})

    for pid, emotions in person_emotion_count.items():
        role = role_map.get(pid, f"Person {pid}")
        role_emotion_totals[role].update(emotions)

# Plot emotion distribution for each role
for role, emotions in role_emotion_totals.items():
    top_emotions = emotions.most_common(top_n)
    labels, counts = zip(*top_emotions)

    plt.figure(figsize=(6, 4))
    plt.bar(labels, counts, color='slateblue')
    plt.title(f"Top {top_n} Emotions for Role: {role}")
    plt.xlabel("Emotion")
    plt.ylabel("Count")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# @title Normalized Frame Counts per Party Across All Videos (Excluding Moderator & Gestural Language)

import matplotlib.pyplot as plt
from collections import defaultdict

# Aggregate normalized frame counts per party across all videos
party_frame_totals = defaultdict(float)
total_frames_all_videos = 0

for video_name, stats in all_video_stats.items():
    role_map = stats.get("role_assignment", {})
    person_frame_count = stats.get("person_frame_count", {})

    total_frames = sum(len(frames) for frames in person_frame_count.values())
    if total_frames == 0:
        continue

    total_frames_all_videos += total_frames

    for pid, frames in person_frame_count.items():
        role = role_map.get(pid, f"Person {pid}")
        if role in ("Moderator", "Gestural Language"):
            continue
        party_frame_totals[role] += len(frames)

# Normalize aggregated counts by total frames across all videos
for party in party_frame_totals:
    party_frame_totals[party] /= total_frames_all_videos

# Prepare data for plotting
parties = list(party_frame_totals.keys())
norm_counts = [party_frame_totals[party] for party in parties]

# Sort parties by normalized counts descending
sorted_parties_counts = sorted(party_frame_totals.items(), key=lambda x: x[1], reverse=True)
parties, norm_counts = zip(*sorted_parties_counts)

plt.figure(figsize=(12, 6))
plt.bar(parties, norm_counts, color='mediumseagreen')
plt.title("Normalized Frame Counts per Party Across All Videos (Excluding Moderator & Gestural Language)")
plt.xlabel("Party")
plt.ylabel("Normalized Frame Count")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# -- **Audio & Speech** --

In [ ]:
# @title List all Pickle (.pkl) files

pklfiles = [f for f in os.listdir('/content/drive/MyDrive/PBD/Project/Project_Data/Features') if os.path.isfile(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Features', f)) and f.endswith('_audio.pkl')]
print(pklfiles)

In [ ]:
# @title Load one
video = 'chega-be'
data_audio = pd.read_pickle(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Features', video + '_audio.pkl'))

# Print first lines
data_audio.head()

In [ ]:
# @title Check feature names
print(data_audio.keys())

In [ ]:
# @title Align with the video frame

print(data_audio['time stamp'][19:25])
print(data_audio['duration'][19:25])

In [ ]:
# @title Associate Frames with Audio Snippets (using data_audio)

# Ensure necessary imports are present (should be from previous cells)
# import pandas as pd
# import os
# import re # For extracting frame numbers

print("\n--- Associating Frames with Audio Snippets (using data_audio) ---")

# Assume 'data' DataFrame (visual data) is available and sorted by filename for single-video analysis.
# Assume 'data_audio' DataFrame is available containing audio snippet information.
# Assume 'video' variable is available holding the name of the single video.

# Dictionary to store frame indices associated with each audio snippet (using index from data_audio)
audio_frame_associations = {}

# Ensure 'data' and 'data_audio' are loaded for single video analysis
try:
    current_video_data = data # Assuming 'data' holds the DataFrame for the visual data
    current_audio_data = data_audio # Assuming 'data_audio' holds the DataFrame for the audio data
    video_name = video       # Assuming 'video' holds the name of the single video
    print(f"Loaded data for video: {video_name}")
except NameError:
    print("Error: 'data', 'data_audio', or 'video' variable not found. Ensure single video data was loaded in a previous cell.")
    # exit() # Stop execution if data isn't available

# Assume a constant frame rate for the video (e.g., 30 frames per second)
# You might need to adjust this based on your video's actual frame rate.
frame_rate = 1 # !!! Adjust this if your video has a different frame rate !!!
frame_duration_ms = (1 / frame_rate) * 1000 # Duration of each frame in milliseconds

# Iterate through each row in the data_audio DataFrame
for audio_index, row in current_audio_data.iterrows():
    audio_filename = audio_index # Assuming 'filename' column in data_audio holds the snippet filename
    duration_ms = round(row['duration']*1000) # Assuming 'duration_ms' column in data_audio holds the duration

    # --- Determine the start time/frame of the audio snippet ---
    # This part still depends on how your audio snippets were created and how 'data_audio' is structured.
    # If 'data_audio' has a 'start_time_seconds' or similar column, use that.
    # If snippets are sequential, you might need to calculate the cumulative duration of previous snippets.
    # If the filename contains time information, parse it.

    # Placeholder: Assuming 'data_audio' has a 'start_time_seconds' column
    # If your data_audio structure is different, **you need to modify this part.**
    try:
        start_time_seconds = row['time stamp'] # !!! Adjust column name if needed !!!
        start_frame_index = int(start_time_seconds * frame_rate)
    except KeyError:
        print(f"Error: 'start_time_seconds' column not found in data_audio. Cannot determine start frame for {audio_filename}.")
        # You need to add logic here to determine the start frame based on your data_audio structure.
        # For now, skipping this audio snippet.
        continue # Skip to the next audio snippet if start time cannot be determined


    # Calculate the number of frames covered by the audio snippet
    num_frames_in_snippet = int(duration_ms / frame_duration_ms)

    # Determine the end frame index (exclusive)
    end_frame_index = start_frame_index + num_frames_in_snippet

    # Get the indices of the frames from the visual data DataFrame
    # that fall within this time range.
    # Ensure frame indices are within the bounds of your visual data DataFrame
    associated_frame_indices = list(range(start_frame_index, min(end_frame_index, len(current_video_data))))

    # Associate the frame indices with the index from the data_audio DataFrame
    # This allows you to easily link back to the audio data.
    audio_frame_associations[audio_index] = associated_frame_indices

    print(f"Audio Index: {audio_index}, File: {audio_filename}, Duration: {duration_ms:.2f} ms, Associated Frames: {len(associated_frame_indices)} frames (indices: {associated_frame_indices[:5]}{'...' if len(associated_frame_indices) > 5 else ''})")


print("\n--- Finished associating frames with audio snippets. ---")

# The 'audio_frame_associations' dictionary now contains indices from data_audio as keys
# and lists of corresponding frame indices from the visual data as values.
# You can use the index from audio_frame_associations to look up the audio data in data_audio.

In [ ]:
# @title Check features names and values

print(data_audio.iloc[19])

In [ ]:
# @title Let's analyse all the pitch values for this specific video

pitch_values = data_audio['meanF0Hz']

# Create a box plot of all pitch values
plt.figure(figsize=(8, 6))
plt.boxplot(pitch_values, showfliers=True)
plt.title('Box Plot of Mean Pitch (meanF0Hz)')
plt.ylabel('Pitch (Hz)')
plt.show()

q33, q66 = np.percentile(pitch_values, [33, 66])

def pitch_to_emotion_bin(pitch):
    if pitch < q33:
        return "low"
    elif pitch <= q66:
        return "neutral"
    else:
        return "high"

pitch_categories = [pitch_to_emotion_bin(p) for p in pitch_values]

print("\nFirst 25 pitch categories:")
print(pitch_categories[:25]) # Print the first few categories to see the result

In [ ]:
# @title Associate each audio snip to an orator (Agglomerative Clustering)

from scipy.io import wavfile
from IPython.display import Audio

# 1. --- Load your data ---
# data_audio = pd.read_csv('your_data.csv')

# 2. --- Calculate end times ---
data_audio['end_time'] = data_audio['time stamp'] + data_audio['duration']

# 3. --- Select features for clustering ---
feature_columns = ['meanF0Hz', 'stdevF0Hz', 'HNR', 'localJitter', 'rapJitter', 'avgFormant', 'mff']

# If speak_embeddings is a column with vectors (list or array per row), expand it:
if 'speak_embeddings' in data_audio.columns:
    embeddings = pd.DataFrame(data_audio['speak_embeddings'].tolist()).add_prefix('embed_')
    data_audio = pd.concat([data_audio, embeddings], axis=1)
    feature_columns += list(embeddings.columns)

# 4. --- Normalize features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data_audio[feature_columns])

# 5. --- Cluster into 3 speakers using Agglomerative Clustering ---
agglo = AgglomerativeClustering(n_clusters=3)
data_audio['predicted_speaker'] = agglo.fit_predict(X_scaled)

# 6. --- Build IntervalTree to find overlaps ---
tree = IntervalTree()
for i, row in data_audio.iterrows():
    tree[row['time stamp']:row['end_time']] = i

# 7. --- Determine overlapping segments ---
data_audio['num_overlapping'] = data_audio.apply(
    lambda row: len(tree[row['time stamp']:row['end_time']]), axis=1
)
data_audio['multiple_speakers'] = data_audio['num_overlapping'] > 1

# 8. --- Visualize speaker timeline ---

for i in range(len(data_audio)):
    plt.figure(figsize=(12, 5))
    j = 0
    for j, row in data_audio.iterrows():
        speaker_id = row['predicted_speaker']
        color = 'red' if i == j else 'blue'

        plt.hlines(y=speaker_id,
                   xmin=row['time stamp'],
                   xmax=row['end_time'],
                   linewidth=3,
                   color=color)
        j += 1
    plt.xlabel("Time (seconds)")
    plt.ylabel("Speaker ID")
    plt.title("Speaker Activity Timeline (Agglomerative Clustering)")
    plt.yticks(sorted(data_audio['predicted_speaker'].unique()))
    plt.tight_layout()
    plt.show()
    samplerate, audio = wavfile.read(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Audio', video + '.wav'))

    sample = audio[int(data_audio.iloc[i]['time stamp']*samplerate):int(data_audio.iloc[i]['time stamp']+data_audio.iloc[i]['duration'])*samplerate,0]

    display(Audio(sample,rate=samplerate))



# 9. --- Optional: Display processed DataFrame ---
data_audio.head()


In [ ]:
# @title Visualise multiple speakers

# Define custom legend entries
red_line = mlines.Line2D([], [], color='red', linewidth=3, label='Multiple Speakers')
blue_line = mlines.Line2D([], [], color='blue', linewidth=3, label='Single Speaker')

plt.figure(figsize=(12, 5))
for j, row in data_audio.iterrows():
    speaker_id = row['predicted_speaker']
    color = 'red' if row['multiple_speakers'] else 'blue'

    plt.hlines(y=speaker_id,
                xmin=row['time stamp'],
                xmax=row['end_time'],
                linewidth=3,
                color=color)
    j += 1
plt.xlabel("Time (seconds)")
plt.ylabel("Speaker ID")
plt.title("Speaker Count Timeline (Agglomerative Clustering)")
plt.legend(handles=[red_line, blue_line], bbox_to_anchor=(0.9, 0.8), loc='center')
plt.tight_layout()
plt.show()

blue_num = mlines.Line2D([], [], color='blue', linewidth=3, label='1 Speaker')
green_num = mlines.Line2D([], [], color='green', linewidth=3, label='2 Speakers')
yellow_num = mlines.Line2D([], [], color=(252/255, 186/255, 3/255) , linewidth=3, label='3 Speakers')

plt.figure(figsize=(12, 5))
for j, row in data_audio.iterrows():
    speaker_id = row['predicted_speaker']
    match row['num_overlapping']:
        case 1:
            color = 'blue'
        case 2:
            color = 'green'
        case 3:
            color = (252/255, 186/255, 3/255)
        case _:
            color = 'red'

    plt.hlines(y=speaker_id,
                xmin=row['time stamp'],
                xmax=row['end_time'],
                linewidth=3,
                color=color)
    j += 1
plt.xlabel("Time (seconds)")
plt.ylabel("Speaker ID")
plt.title("Amount of Speakers Timeline (Agglomerative Clustering)")
plt.legend(handles=[blue_num, green_num, yellow_num], bbox_to_anchor=(0.9, 0.8), loc='center')
plt.tight_layout()
plt.show()

In [ ]:
# @title Pitch association

samplerate, audio = wavfile.read(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Audio', video + '.wav'))

# If stereo, take one channel
if len(audio.shape) == 2:
    audio = audio[:, 0]

# Create time axis in seconds
time = np.linspace(0, len(audio) / samplerate, num=len(audio))

# Plot
plt.figure(figsize=(12, 4))
plt.plot(time, audio)
plt.title('Waveform')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

In [ ]:
# @title Load Audio Data and Box Plots for meanF0Hz (Per Video and Overall)

# @markdown referir intenções futuras

# @markdown Que depois queremos consguir dividir isto entre pessoas comparando
# @markdown com outras coisas como os embeedings de quem poderá estar a falar

# Ensure necessary imports are present
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

# Assuming your audio analysis data is stored in a similar structure to visual data,
# likely in pickle files within a specific folder for each video.

# Path to the audio features folder
audio_features_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features' # Assuming audio features are in the same 'Features' folder
audio_file_suffix = '_audio.pkl' # Assuming audio analysis files end with '_audio.pkl'

print("Loading audio analysis data...")

# Dictionary to store all audio data
all_audio_data = {}

# List all audio pickle files
audio_files = [f for f in os.listdir(audio_features_path) if f.endswith(audio_file_suffix)]

# Define the correct column name for pitch
pitch_column_name = 'meanF0Hz'

if audio_files:
    for file in audio_files:
        video_name = file.replace(audio_file_suffix, '')
        file_path = os.path.join(audio_features_path, file)

        try:
            audio_df = pd.read_pickle(file_path)
            all_audio_data[video_name] = audio_df
            print(f"✓ Loaded audio data for {video_name} with {len(audio_df)} entries.")
        except Exception as e:
            print(f"✗ Failed to load audio data for {video_name}: {e}")

    print(f"\nTotal videos with audio data loaded: {len(all_audio_data)}")

    # --- Create Box Plot for EACH Video's meanF0Hz Distribution ---

    print(f"\nCreating box plots for {pitch_column_name} distribution per video...")

    all_pitch_values_overall = [] # List to store all pitch values for the final overall plot

    # Iterate through each video's audio data
    for video_name, audio_data_df in all_audio_data.items():
        print(f"\n--- Processing video: {video_name} ---")

        # Assuming the pitch values are in the column named by pitch_column_name
        if pitch_column_name in audio_data_df.columns:
            # Get the pitch values for this video, dropping None/NaN
            video_pitch_values = audio_data_df[pitch_column_name].dropna().tolist()

            if video_pitch_values:
                print(f"Creating box plot for {video_name}...")
                plt.figure(figsize=(8, 6)) # Set the figure size
                plt.boxplot(video_pitch_values, showfliers=True)
                plt.title(f'Distribution of Pitch Values ({pitch_column_name}) for {video_name}')
                plt.ylabel('Pitch Value (Hz)')
                plt.xticks([1], [video_name]) # Label the x-axis with the video name
                plt.grid(axis='y', linestyle='--', alpha=0.7)
                plt.show()

                # Add this video's pitch values to the overall list
                all_pitch_values_overall.extend(video_pitch_values)
                print(f"Aggregated {len(video_pitch_values)} pitch values from {video_name} for overall plot.")

            else:
                print(f"No valid pitch values ({pitch_column_name}) found for {video_name} to create a box plot.")
        else:
            print(f"Warning: '{pitch_column_name}' column not found in audio data for {video_name}. Skipping box plot for this video.")

    # --- Create FINAL Overall Box Plot of Aggregated meanF0Hz Values ---

    print(f"\n--- Creating FINAL overall box plot of {pitch_column_name} distribution ---")

    if all_pitch_values_overall:
        plt.figure(figsize=(10, 7)) # Slightly larger figure for the overall plot
        plt.boxplot(all_pitch_values_overall, showfliers=True)
        plt.title(f'Overall Distribution of Pitch Values ({pitch_column_name}) Across All Videos')
        plt.ylabel('Pitch Value (Hz)')
        plt.xticks([1], ['All Videos Combined']) # Clear label for the combined plot
        plt.grid(axis='y', linestyle='--', alpha=0.7)

        print(f"Total aggregated pitch values ({pitch_column_name}) for overall plot: {len(all_pitch_values_overall)}")

        # Show the overall plot
        plt.show()

    else:
        print(f"\nNo valid pitch values ({pitch_column_name}) found across all videos for the overall box plot.")


else:
    print(f"No audio analysis files ending with '{audio_file_suffix}' found in the specified features path.")

In [ ]:
# @title Audio features across all videos

if 'all_audio_data' not in locals() or not isinstance(all_audio_data, dict) or not all_audio_data:
    print("Error: 'all_audio_data' is not defined, not a dictionary, or is empty.")
    print("Please ensure the audio data loading script (the first script) has been run successfully.")
    # Optionally, you could stop execution here or attempt to load data.
    # For now, we'll proceed assuming it might be populated by a prior cell in a notebook.
    # If it's truly missing, the script will gracefully exit later.
    audio_features_to_plot = [] # Initialize as empty if data is not loaded
else:
    # --- Dynamically define audio_features_to_plot from the loaded data ---
    try:
        # Get a sample DataFrame to infer columns (assuming all DataFrames have similar columns)
        sample_video_name = next(iter(all_audio_data))
        sample_df = all_audio_data[sample_video_name]

        # Select only numeric columns as potential features
        if isinstance(sample_df, pd.DataFrame):
            audio_features_to_plot = sample_df.select_dtypes(include=np.number).columns.tolist()
        else:
            print(f"Warning: Data for '{sample_video_name}' is not a DataFrame. Cannot infer features.")
            audio_features_to_plot = []

        # Optional: You can manually exclude columns if needed
        # For example, if 'meanF0Hz' (pitch_column_name) was extensively plotted
        # by the first script and you don't want its *comparative* plot here:
        # pitch_column_name = 'meanF0Hz' # Assuming this was defined in the first script
        # if pitch_column_name in audio_features_to_plot:
        #     audio_features_to_plot.remove(pitch_column_name)
        # However, the comparative plot generated here (one box per video on the same axes)
        # is different from the per-video plots or the single aggregated plot from the first script,
        # so it might still be useful to include it.

        if not audio_features_to_plot:
            print("No numeric features found in the data to plot or data is not in expected format.")
        else:
            print(f"Successfully identified features for plotting: {audio_features_to_plot}")

    except StopIteration:
        print("Error: 'all_audio_data' is empty. Cannot infer features.")
        audio_features_to_plot = [] # Ensure it's empty
    except Exception as e:
        print(f"An error occurred during feature identification: {e}")
        audio_features_to_plot = [] # Ensure it's empty


# Proceed only if features were identified and all_audio_data is confirmed to be populated
if 'all_audio_data' in locals() and isinstance(all_audio_data, dict) and all_audio_data and audio_features_to_plot:

    print(f"\nCreating plots for each identified feature across all videos...")

    # Define which features are formants to plot as bar charts
    formant_features = [
        'f1_mean', 'f2_mean', 'f3_mean', 'f4_mean',
        'f1_median', 'f2_median', 'f3_median', 'f4_median'
    ]

    # Get the list of video names (which represent the parties/debates)
    video_names = sorted(all_audio_data.keys())

    # Iterate through each identified audio feature
    for feature in audio_features_to_plot:
        print(f"\n--- Processing feature: {feature} ---")

        data_to_plot_this_feature = []
        labels_for_plot = []

        # Collect data for this feature from ALL videos
        for video_name in video_names:
            if video_name in all_audio_data and isinstance(all_audio_data[video_name], pd.DataFrame):
                audio_data_df = all_audio_data[video_name]

                if feature in audio_data_df.columns:
                    # Get the data for this feature, dropping None/NaN, and ensuring it's not empty
                    feature_values = audio_data_df[feature].dropna().tolist()
                    if feature_values: # Only add if there's actual data
                        data_to_plot_this_feature.append(feature_values)
                        labels_for_plot.append(video_name)
                    # else:
                        # print(f"No valid (non-NaN) data for '{feature}' in '{video_name}'.")
                # else:
                    # print(f"Feature '{feature}' not found in columns of '{video_name}'.")
            # else:
                # print(f"No data or not a DataFrame for video '{video_name}'.")

        # Only plot if there is valid data from at least one video
        if data_to_plot_this_feature and labels_for_plot:
            plt.figure(figsize=(max(10, len(labels_for_plot) * 1.0), 8)) # Adjusted width factor for readability

            # ========== Box Plot (default) ==========
            if feature not in formant_features:
                print(f"Creating box plot for '{feature}' across videos...")
                plt.boxplot(data_to_plot_this_feature, showfliers=False) # showfliers=False was in the original
                plt.title(f'Distribution of {feature} Across Videos')
                plt.ylabel('Value')
                plt.xticks(range(1, len(labels_for_plot) + 1), labels_for_plot, rotation=45, ha='right')

            # ========== Bar Plot (for formants) ==========
            else:
                print(f"Creating bar plot for '{feature}' (formant) across videos...")

                # Calculate means and stds, being careful about empty lists (if a video had no valid data)
                means = [np.mean(values) if values else np.nan for values in data_to_plot_this_feature]
                stds = [np.std(values) if values else np.nan for values in data_to_plot_this_feature]

                # Filter out any NaNs that resulted from empty value lists for a video
                valid_indices = [i for i, m in enumerate(means) if not np.isnan(m)]

                if valid_indices:
                    valid_labels = [labels_for_plot[i] for i in valid_indices]
                    valid_means = [means[i] for i in valid_indices]
                    valid_stds = [stds[i] for i in valid_indices]

                    plt.bar(valid_labels, valid_means, yerr=valid_stds, capsize=5, color='skyblue', edgecolor='black')
                    plt.title(f'Mean ± Std of {feature} Across Videos')
                    plt.ylabel('Value (e.g., Hz for formants)') # General Y-axis label
                    plt.xticks(rotation=45, ha='right')
                else:
                    print(f"No valid aggregated data (means/stds) to plot for formant feature '{feature}'.")
                    # Need to close the figure if nothing is plotted to avoid empty figure showing
                    plt.close()
                    continue # Skip to next feature

            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout() # Adjust layout to prevent labels from overlapping
            plt.show()

        else:
            print(f"No valid data found for feature '{feature}' across any video to create a plot.")
            # If plt.figure() was called but no plot made, ensure it's closed if you have a conditional plot path
            # In this structure, plt.figure is called only if data_to_plot_this_feature is non-empty.

elif not ('all_audio_data' in locals() and isinstance(all_audio_data, dict) and all_audio_data):
    print("Script execution halted: 'all_audio_data' was not properly loaded or is empty.")
elif not audio_features_to_plot:
    print("Script execution halted: No audio features were identified for plotting.")

In [ ]:
# @title Speech Features analysis (speechrate, articulationrate, npause)
# Features to plot
rate_features = ['speechrate', 'articulationrate']
pause_feature = 'npause'

num_videos = len(all_audio_data)
fig, axes = plt.subplots(num_videos, 1, figsize=(12, 4 * num_videos), sharex=False)

# Ensure axes is an array even for a single video
if num_videos == 1:
    axes = [axes]

for i, (video, df) in enumerate(all_audio_data.items()):
    # Check if required features and time/duration information are present
    if all(f in df.columns for f in rate_features + [pause_feature]) and 'time stamp' in df.columns and 'duration' in df.columns:
        times = df['time stamp']
        sr = df['speechrate']
        ar = df['articulationrate']
        pauses = df['npause']
        durations = df['duration']

        ax = axes[i]
        ax2 = ax.twinx()  # Create a second y-axis for npause

        # Plotting speech rate and articulation rate as horizontal lines representing duration
        for j in range(len(times)):
            start_time = times.iloc[j]
            end_time = times.iloc[j] + durations.iloc[j]

            ax.hlines(sr.iloc[j], start_time, end_time, color='tab:blue', linewidth=3, alpha=0.7, label='Speech Rate' if j == 0 else "")
            ax.hlines(ar.iloc[j], start_time, end_time, color='tab:red', linewidth=3, alpha=0.7, label='Articulation Rate' if j == 0 else "")


        # Bar plot for npause (on second axis) - using the time stamp as the position and duration as width
        ax2.bar(times, pauses, width=durations, alpha=0.4, color='tab:orange', label='Number of Pauses', align='edge') # 'edge' aligns the left edge of the bar with the time stamp

        # Titles and labels
        ax.set_title(f"{video} — Speaking Metrics")
        ax.set_ylabel("Rate (syllables/sec)")
        ax2.set_ylabel("Number of Pauses")

        # Set x-axis limits based on the min time stamp and max end time (time stamp + duration)
        ax.set_xlim([df['time stamp'].min(), (df['time stamp'] + df['duration']).max()])


        # Grid and legends
        ax.grid(True, linestyle='--', alpha=0.6)
        # Combine legends from both axes
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    else:
        axes[i].set_title(f"{video} — Required features not available")
        axes[i].set_ylabel("No data")

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

In [ ]:
# @title Load Audio Data and Box Plot for Harmonic to Noise Ratio (HNR) Across All Videos

# Ensure necessary imports are present
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

# Assuming your audio analysis data is stored in a similar structure to visual data,
# likely in pickle files within a specific folder for each video.

# Path to the audio features folder
audio_features_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features' # Assuming audio features are in the same 'Features' folder
audio_file_suffix = '_audio.pkl' # Assuming audio analysis files end with '_audio.pkl'

print("Loading audio analysis data...")

# Dictionary to store all audio data
all_audio_data = {}

# List all audio pickle files
audio_files = [f for f in os.listdir(audio_features_path) if f.endswith(audio_file_suffix)]

# Define the specific feature to plot
feature_to_plot = 'HNR'

if audio_files:
    for file in audio_files:
        video_name = file.replace(audio_file_suffix, '')
        file_path = os.path.join(audio_features_path, file)

        try:
            audio_df = pd.read_pickle(file_path)
            all_audio_data[video_name] = audio_df
            print(f"✓ Loaded audio data for {video_name} with {len(audio_df)} entries.")
        except Exception as e:
            print(f"✗ Failed to load audio data for {video_name}: {e}")

    print(f"\nTotal videos with audio data loaded: {len(all_audio_data)}")

    # --- Create Box Plot for HNR Across All Videos ---

    print(f"\nCreating box plot for {feature_to_plot} across all videos...")

    data_to_plot_this_feature = []
    labels_for_plot = [] # Use video names as labels

    # Get the list of video names (which represent the parties/debates)
    video_names = list(all_audio_data.keys())

    # Collect data for the specific feature (HNR) from ALL videos
    for video_name in video_names:
        audio_data_df = all_audio_data[video_name]

        if feature_to_plot in audio_data_df.columns:
            # Get the data for HNR from the current video, dropping None/NaN
            feature_values = audio_data_df[feature_to_plot].dropna().tolist()
            if feature_values:
                data_to_plot_this_feature.append(feature_values)
                labels_for_plot.append(video_name) # Add the video name as a label for this data set
            else:
                 print(f"Warning: Feature '{feature_to_plot}' for video '{video_name}' contains only NaN/None values after dropna(). Skipping for this video in this plot.")
        else:
             print(f"Warning: Feature '{feature_to_plot}' not found in audio data for {video_name}. Skipping for this video in this plot.")


    # --- Create and Display the Box Plot for HNR Across ALL Videos ---

    if data_to_plot_this_feature:
        print(f"Creating box plot for '{feature_to_plot}' across videos...")
        # Adjust figure size - width needs to accommodate more box plots
        plt.figure(figsize=(max(10, len(labels_for_plot) * 0.8), 8)) # Dynamic width based on number of videos
        plt.boxplot(data_to_plot_this_feature, showfliers=False) # showfliers=False might make the plot cleaner
        plt.title(f'Distribution of {feature_to_plot} Across Videos')
        plt.ylabel(feature_to_plot) # Label the y-axis with the feature name
        # Set x-ticks with video names and rotate
        plt.xticks(range(1, len(labels_for_plot) + 1), labels_for_plot, rotation=45, ha='right')
        plt.grid(axis='y', linestyle='--', alpha=0.7) # Add a horizontal grid
        plt.tight_layout() # Adjust layout to prevent labels overlapping
        plt.show()

    else:
        print(f"No valid data found for feature '{feature_to_plot}' across any video to create a box plot.")


else:
    print(f"No audio analysis files ending with '{audio_file_suffix}' found in the specified features path.")

In [ ]:
# @title Lets listen
from scipy.io import wavfile
from IPython.display import Audio

samplerate, audio = wavfile.read(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Audio', video + '.wav'))

sample = audio[int(data_audio.iloc[19]['time stamp']*samplerate):int(data_audio.iloc[19]['time stamp']+data_audio.iloc[19]['duration'])*samplerate,0]

display(Audio(sample,rate=samplerate))

In [ ]:
# @title Load all audio data

audio_features_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features'
audio_file_suffix = '_audio.pkl'

all_audio_data = {}

for file in pklfiles:
    file_path = os.path.join(audio_features_path, file)
    try:
        df = pd.read_pickle(file_path)
        all_audio_data[file] = df
        print(f"✓ Loaded {file}") # Add a confirmation message
    except FileNotFoundError:
        print(f"✗ Error: File not found at {file_path}")
    except Exception as e:
        print(f"✗ Failed to load {file}: {e}")

print(f"\nTotal audio dataframes loaded: {len(all_audio_data)}")

In [ ]:
# @title Match parties to their videos
# List of parties
parties = ["ad", "be", "cdu", "chega", "il", "livre", "pan", "ps"]
party_videos = {party: [f for f in all_audio_data if party in f] for party in parties}

for party in party_videos:
    print(f"{party}: {party_videos[party]}")

In [ ]:
# @title Associate each audio snip to an orator (Agglomerative Clustering) for All Videos

# Assuming 'all_audio_data' is a list of pandas DataFrames,
# where each DataFrame contains audio features for a single video.
# This list should have been populated by a previous cell.

if 'all_audio_data' not in locals() or not isinstance(all_audio_data, dict) or not all_audio_data:
    print("Error: 'all_audio_data' is not defined, not a list, or is empty.")
    print("Please ensure the audio data loading script has been run successfully.")
else:
    # --- Define features for clustering ---
    # Define this outside the loop as it should be consistent for all videos
    feature_columns_base = ['meanF0Hz', 'stdevF0Hz', 'HNR', 'localJitter', 'rapJitter', 'avgFormant', 'mff']

    # We will store the processed dataframes in a new list or overwrite the original
    processed_audio_data = {}


    # Iterate through each DataFrame in the list
    for i, key in enumerate(all_audio_data):
        print(f"\n--- Processing video {i+1}/{len(all_audio_data)} ---")

        # Make a copy to avoid modifying the original DataFrame in the list directly
        # if you intend to keep the original data_audio list unchanged.
        # If you are fine modifying the DataFrames in all_audio_data, you can skip this.
        # data_audio_processed = data_audio.copy()
        # Using the original data_audio variable for simplicity here,
        # but be mindful if you need the original data later.

        data_audio = all_audio_data[key]

        # 2. --- Calculate end times ---
        if 'time stamp' in data_audio.columns and 'duration' in data_audio.columns:
            data_audio['end_time'] = data_audio['time stamp'] + data_audio['duration']
        else:
            print(f"Warning: 'time stamp' or 'duration' not found in DataFrame {i}. Skipping end time calculation.")
            # Depending on your needs, you might skip this DataFrame or handle it differently.
            # For now, we'll continue and the IntervalTree step will likely have issues.
            # A more robust solution would be to skip processing for this DataFrame.


        # 3. --- Select features for clustering ---
        feature_columns = feature_columns_base[:] # Start with the base features

        # Check and add speak_embeddings if present
        if 'speak_embeddings' in data_audio.columns and data_audio['speak_embeddings'].iloc[0] is not None:
             try:
                # Ensure embeddings are lists/arrays before expanding
                embeddings_valid = data_audio['speak_embeddings'].apply(lambda x: isinstance(x, (list, np.ndarray)))
                if embeddings_valid.any():
                    embeddings_df = pd.DataFrame(data_audio.loc[embeddings_valid, 'speak_embeddings'].tolist()).add_prefix('embed_')
                    # Align indices before concatenating
                    embeddings_df.index = data_audio.loc[embeddings_valid, 'speak_embeddings'].index
                    data_audio = pd.concat([data_audio, embeddings_df], axis=1)
                    feature_columns += list(embeddings_df.columns)
                    print(f"Added {len(list(embeddings_df.columns))} speak_embedding features.")
                else:
                     print(f"Warning: 'speak_embeddings' column found, but no valid embeddings (list/array) in DataFrame {i}.")
             except Exception as e:
                 print(f"Error expanding 'speak_embeddings' in DataFrame {i}: {e}")


        # Filter feature_columns to only include those actually present and numeric
        available_numeric_features = data_audio.select_dtypes(include=np.number).columns.tolist()
        features_for_clustering = [f for f in feature_columns if f in available_numeric_features]

        if not features_for_clustering:
            print(f"Error: No valid numeric features found for clustering in DataFrame {i}. Skipping clustering for this video.")
            processed_audio_data[key] = data_audio # Append the original or partially processed df
            continue # Skip to the next DataFrame

        print(f"Features used for clustering: {features_for_clustering}")

        # Handle potential NaNs or infinite values before scaling
        X = data_audio[features_for_clustering].replace([np.inf, -np.inf], np.nan).dropna()

        if X.empty:
            print(f"Warning: No valid data points remaining for clustering after handling NaNs/Infs in DataFrame {i}. Skipping clustering.")
            data_audio['predicted_speaker'] = -1 # Assign a placeholder for skipped rows
            data_audio['num_overlapping'] = 0
            data_audio['multiple_speakers'] = False
            processed_audio_data[key] = data_audio # Append the original or partially processed df
            continue # Skip to the next DataFrame


        # Store the original index to map results back
        original_index = X.index

        # 4. --- Normalize features ---
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # 5. --- Cluster into 3 speakers using Agglomerative Clustering ---
        # Check if there are enough samples for clustering (at least n_clusters)
        n_clusters = 3 # You can make this a parameter if needed
        if X_scaled.shape[0] < n_clusters:
            print(f"Warning: Not enough data points ({X_scaled.shape[0]}) for {n_clusters} clusters in DataFrame {i}. Skipping clustering.")
            # Assign a placeholder for skipped rows, aligning with original_index
            data_audio.loc[original_index, 'predicted_speaker'] = -1 # Use -1 to indicate no cluster
            # Initialize other columns for consistency
            if 'num_overlapping' not in data_audio.columns:
                 data_audio['num_overlapping'] = 0
            if 'multiple_speakers' not in data_audio.columns:
                 data_audio['multiple_speakers'] = False

        else:
            agglo = AgglomerativeClustering(n_clusters=n_clusters)
            # Assign the cluster labels back to the original DataFrame, aligning by index
            data_audio.loc[original_index, 'predicted_speaker'] = agglo.fit_predict(X_scaled)
            # For rows that had NaNs and were dropped from X, their 'predicted_speaker' will be NaN.
            # You might want to fill these with a placeholder like -1:
            data_audio['predicted_speaker'] = data_audio['predicted_speaker'].fillna(-1).astype(int)


        # 6. --- Build IntervalTree to find overlaps ---
        # Ensure 'time stamp' and 'end_time' exist before building the tree
        if 'time stamp' in data_audio.columns and 'end_time' in data_audio.columns:
            tree = IntervalTree()
            # Only add intervals for rows with valid time stamps and durations
            for idx, row in data_audio.dropna(subset=['time stamp', 'end_time']).iterrows():
                try:
                    tree[row['time stamp']:row['end_time']] = idx
                except ValueError as e:
                     print(f"Warning: Invalid interval [{row['time stamp']}:{row['end_time']}] for index {idx} in DataFrame {i}. Skipping this interval.")
                     # This could happen if start_time > end_time or if times are not numeric.


            # 7. --- Determine overlapping segments ---
            if not tree.is_empty():
                data_audio['num_overlapping'] = data_audio.apply(
                    lambda row: len(tree[row['time stamp']:row['end_time']]) if pd.notnull(row['time stamp']) and pd.notnull(row['end_time']) else 0,
                    axis=1
                )
                data_audio['multiple_speakers'] = data_audio['num_overlapping'] > 1
            else:
                 print(f"Warning: IntervalTree is empty for DataFrame {i}. Skipping overlap calculation.")
                 # Initialize columns if not already done in the skipping section
                 if 'num_overlapping' not in data_audio.columns:
                     data_audio['num_overlapping'] = 0
                 if 'multiple_speakers' not in data_audio.columns:
                     data_audio['multiple_speakers'] = False
                 # If time_stamp or end_time were missing, these would already be initialized
        else:
             print(f"Warning: 'time stamp' or 'end_time' missing in DataFrame {i}. Skipping overlap calculation.")
             # Initialize columns if not already done in the skipping section
             if 'num_overlapping' not in data_audio.columns:
                 data_audio['num_overlapping'] = 0
             if 'multiple_speakers' not in data_audio.columns:
                 data_audio['multiple_speakers'] = False


        # Append the processed DataFrame to the new list
        processed_audio_data[key] = data_audio

        plt.figure(figsize=(12, 5))
        for _, row in data_audio.iterrows():
            speaker_id = row['predicted_speaker']
            color ='blue'

            plt.hlines(y=speaker_id,
                      xmin=row['time stamp'],
                      xmax=row['end_time'],
                      linewidth=3,
                      color=color)
        plt.xlabel("Time (seconds)")
        plt.ylabel("Speaker ID")
        plt.title("Speaker Activity Timeline (Agglomerative Clustering)")
        plt.yticks(sorted(data_audio['predicted_speaker'].unique()))
        plt.tight_layout()
        plt.show()

        # Optional: Display the head of the processed DataFrame for this video
        print(f"Processed DataFrame head for video {key}:")
        display(data_audio.head())


    print("\n--- Finished processing all videos ---")

    # Now 'processed_audio_data' contains the DataFrames with 'predicted_speaker',
    # 'num_overlapping', and 'multiple_speakers' columns added.

    # You can now proceed with visualizations or further analysis using 'processed_audio_data'.
    # For example, to access the processed data for the first video:
    # first_video_processed_df = processed_audio_data[0]

In [ ]:
# @title Calculate Average Speaking Time Percentage Per Party
# Track known orator embeddings by party
known_orator_embeddings = defaultdict(list)

def get_avg_embeddings_by_speaker(df):
    """Return a dict: {speaker_id: avg_embedding} from a DataFrame."""
    embed_cols = [col for col in df.columns if col.startswith("embed_")]
    speaker_ids = df['predicted_speaker'].unique()
    return {
        speaker_id: df[df['predicted_speaker'] == speaker_id][embed_cols].mean().values
        for speaker_id in speaker_ids
    }

def identify_party_orator_ids_cross_video(processed_audio_data, party_name):
    """
    For a given party, compare all speakers across all videos they appear in,
    and return the most likely speaker ID per video based on cosine similarity.

    Returns:
        party_orator_map = {filename: predicted_speaker_id}
    """
    from sklearn.metrics.pairwise import cosine_similarity

    # Step 1: Get all videos for the party
    relevant_files = party_videos[party_name]
    print(f"Party '{party_name}' relevant videos: {relevant_files}")
    if len(relevant_files) < 2:
        print(f"Not enough videos to compare for party {party_name}")
        return {}

    # Step 2: Extract average embeddings per speaker for each video
    file_to_speaker_embeddings = {}
    for fname in relevant_files:
        df = processed_audio_data[fname]
        file_to_speaker_embeddings[fname] = get_avg_embeddings_by_speaker(df)

    # Step 3: For each file, find the speaker with max average similarity to others
    party_orator_map = {}

    for current_file, current_embeddings in file_to_speaker_embeddings.items():
        speaker_scores = {}

        for speaker_id, emb in current_embeddings.items():
            similarities = []

            for other_file, other_embeddings in file_to_speaker_embeddings.items():
                if other_file == current_file:
                    continue

                for other_emb in other_embeddings.values():
                    sim = cosine_similarity([emb], [other_emb])[0][0]
                    similarities.append(sim)

            if similarities:
                speaker_scores[speaker_id] = np.mean(similarities)

        if speaker_scores:
            best_speaker = max(speaker_scores, key=speaker_scores.get)
            party_orator_map[current_file] = best_speaker
            print(f"{current_file}: Selected speaker {best_speaker} for party '{party_name}' (avg sim: {speaker_scores[best_speaker]:.3f})")

    return party_orator_map

# Storage for results
party_speaking_times = defaultdict(float)
party_total_debate_times = defaultdict(float)

if isinstance(processed_audio_data, dict) and processed_audio_data:
    print("Calculating average speaking time percentage per party...")

    # parties_to_analyze = sorted(list(set([file.split('-')[0] for file in processed_audio_data if '-' in file])))
    # print(f"Analyzing speaking time for parties: {parties_to_analyze}")

    for party_name in party_videos:
        for filename, data_audio_processed in processed_audio_data.items():
            if party_name not in filename:
                continue
            try:
                # parts = filename.replace('_audio.pkl', '').split('-')
                # party_name = parts[0] if parts else 'unknown_party'
                # debate_name = '-'.join(parts[1:]) if len(parts) > 1 else 'unknown_debate'
                debate_name = filename.replace('_audio.pkl', '')

                if party_name in parties:
                    print(f"\nProcessing debate: {debate_name} for party: {party_name}")

                    party_orators_result = identify_party_orator_ids_cross_video(processed_audio_data, party_name)
                    party_orator_id = party_orators_result[filename]

                    if party_orator_id is not None:
                        print(f"Identified party orator with predicted_speaker ID: {party_orator_id}")

                        party_orator_data = data_audio_processed[data_audio_processed['predicted_speaker'] == party_orator_id]
                        speaking_time_this_debate = party_orator_data['duration'].sum()
                        party_speaking_times[party_name] += speaking_time_this_debate
                        print(f"Speaking time for {party_name} in {debate_name}: {speaking_time_this_debate:.2f} seconds")

                    else:
                        print(f"Could not identify party orator for {party_name} in {debate_name}. Skipping...")

                    total_debate_time_this_debate = data_audio_processed['duration'].sum()
                    party_total_debate_times[party_name] += total_debate_time_this_debate
                    print(f"Total debate time for {debate_name}: {total_debate_time_this_debate:.2f} seconds")

            except Exception as e:
                print(f"Error processing file '{filename}': {e}")

    print("\n--- Results ---")
    party_speaking_percentages = {}

    for party in parties:
        total_speaking_time = party_speaking_times[party]
        total_debate_time = party_total_debate_times[party]

        if total_debate_time > 0:
            speaking_percentage = (total_speaking_time / total_debate_time) * 100
            party_speaking_percentages[party] = speaking_percentage
            print(f"{party}: {speaking_percentage:.2f}% ({total_speaking_time:.2f}s / {total_debate_time:.2f}s)")
        else:
            print(f"{party}: No usable debate time.")
            party_speaking_percentages[party] = 0.0

    speaking_percentage_df = pd.DataFrame(
        list(party_speaking_percentages.items()),
        columns=['Party', 'Speaking Percentage (%)']
    ).sort_values(by='Speaking Percentage (%)', ascending=False).reset_index(drop=True)

    print("\nSpeaking Percentage Summary:")
    display(speaking_percentage_df)

else:
    print("Script halted: 'processed_audio_data' must be a non-empty dictionary.")

In [ ]:
# Build this once, after running your speaking time calculation loop:
orator_lookup = {}  # {(audio_key, speaker_id): party_name}

for party_name in party_videos:
    party_orators_result = identify_party_orator_ids_cross_video(processed_audio_data, party_name)
    for audio_key, speaker_id in party_orators_result.items():
        orator_lookup[(audio_key, speaker_id)] = party_name


# --**Audio Multiple Videos**--

In [ ]:
# @title Load all audio data

audio_features_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features'
audio_file_suffix = '_audio.pkl'

all_audio_data = {}

for file in pklfiles:
    file_path = os.path.join(audio_features_path, file)
    try:
        df = pd.read_pickle(file_path)
        all_audio_data[file] = df
        print(f"✓ Loaded {file}") # Add a confirmation message
    except FileNotFoundError:
        print(f"✗ Error: File not found at {file_path}")
    except Exception as e:
        print(f"✗ Failed to load {file}: {e}")

print(f"\nTotal audio dataframes loaded: {len(all_audio_data)}")

In [ ]:
# @title Match parties to their videos
# List of parties
parties = ["ad", "be", "cdu", "chega", "il", "livre", "pan", "ps"]
party_videos = {party: [f for f in all_audio_data if party in f] for party in parties}

for party in party_videos:
    print(f"{party}: {party_videos[party]}")

In [ ]:
# @title Associate each audio snip to an orator (Agglomerative Clustering) for All Videos

# Assuming 'all_audio_data' is a list of pandas DataFrames,
# where each DataFrame contains audio features for a single video.
# This list should have been populated by a previous cell.

if 'all_audio_data' not in locals() or not isinstance(all_audio_data, dict) or not all_audio_data:
    print("Error: 'all_audio_data' is not defined, not a list, or is empty.")
    print("Please ensure the audio data loading script has been run successfully.")
else:
    # --- Define features for clustering ---
    # Define this outside the loop as it should be consistent for all videos
    feature_columns_base = ['meanF0Hz', 'stdevF0Hz', 'HNR', 'localJitter', 'rapJitter', 'avgFormant', 'mff']

    # We will store the processed dataframes in a new list or overwrite the original
    processed_audio_data = {}


    # Iterate through each DataFrame in the list
    for i, key in enumerate(all_audio_data):
        print(f"\n--- Processing video {i+1}/{len(all_audio_data)} ---")

        # Make a copy to avoid modifying the original DataFrame in the list directly
        # if you intend to keep the original data_audio list unchanged.
        # If you are fine modifying the DataFrames in all_audio_data, you can skip this.
        # data_audio_processed = data_audio.copy()
        # Using the original data_audio variable for simplicity here,
        # but be mindful if you need the original data later.

        data_audio = all_audio_data[key]

        # 2. --- Calculate end times ---
        if 'time stamp' in data_audio.columns and 'duration' in data_audio.columns:
            data_audio['end_time'] = data_audio['time stamp'] + data_audio['duration']
        else:
            print(f"Warning: 'time stamp' or 'duration' not found in DataFrame {i}. Skipping end time calculation.")
            # Depending on your needs, you might skip this DataFrame or handle it differently.
            # For now, we'll continue and the IntervalTree step will likely have issues.
            # A more robust solution would be to skip processing for this DataFrame.


        # 3. --- Select features for clustering ---
        feature_columns = feature_columns_base[:] # Start with the base features

        # Check and add speak_embeddings if present
        if 'speak_embeddings' in data_audio.columns and data_audio['speak_embeddings'].iloc[0] is not None:
             try:
                # Ensure embeddings are lists/arrays before expanding
                embeddings_valid = data_audio['speak_embeddings'].apply(lambda x: isinstance(x, (list, np.ndarray)))
                if embeddings_valid.any():
                    embeddings_df = pd.DataFrame(data_audio.loc[embeddings_valid, 'speak_embeddings'].tolist()).add_prefix('embed_')
                    # Align indices before concatenating
                    embeddings_df.index = data_audio.loc[embeddings_valid, 'speak_embeddings'].index
                    data_audio = pd.concat([data_audio, embeddings_df], axis=1)
                    feature_columns += list(embeddings_df.columns)
                    print(f"Added {len(list(embeddings_df.columns))} speak_embedding features.")
                else:
                     print(f"Warning: 'speak_embeddings' column found, but no valid embeddings (list/array) in DataFrame {i}.")
             except Exception as e:
                 print(f"Error expanding 'speak_embeddings' in DataFrame {i}: {e}")


        # Filter feature_columns to only include those actually present and numeric
        available_numeric_features = data_audio.select_dtypes(include=np.number).columns.tolist()
        features_for_clustering = [f for f in feature_columns if f in available_numeric_features]

        if not features_for_clustering:
            print(f"Error: No valid numeric features found for clustering in DataFrame {i}. Skipping clustering for this video.")
            processed_audio_data[key] = data_audio # Append the original or partially processed df
            continue # Skip to the next DataFrame

        print(f"Features used for clustering: {features_for_clustering}")

        # Handle potential NaNs or infinite values before scaling
        X = data_audio[features_for_clustering].replace([np.inf, -np.inf], np.nan).dropna()

        if X.empty:
            print(f"Warning: No valid data points remaining for clustering after handling NaNs/Infs in DataFrame {i}. Skipping clustering.")
            data_audio['predicted_speaker'] = -1 # Assign a placeholder for skipped rows
            data_audio['num_overlapping'] = 0
            data_audio['multiple_speakers'] = False
            processed_audio_data[key] = data_audio # Append the original or partially processed df
            continue # Skip to the next DataFrame


        # Store the original index to map results back
        original_index = X.index

        # 4. --- Normalize features ---
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # 5. --- Cluster into 3 speakers using Agglomerative Clustering ---
        # Check if there are enough samples for clustering (at least n_clusters)
        n_clusters = 3 # You can make this a parameter if needed
        if X_scaled.shape[0] < n_clusters:
            print(f"Warning: Not enough data points ({X_scaled.shape[0]}) for {n_clusters} clusters in DataFrame {i}. Skipping clustering.")
            # Assign a placeholder for skipped rows, aligning with original_index
            data_audio.loc[original_index, 'predicted_speaker'] = -1 # Use -1 to indicate no cluster
            # Initialize other columns for consistency
            if 'num_overlapping' not in data_audio.columns:
                 data_audio['num_overlapping'] = 0
            if 'multiple_speakers' not in data_audio.columns:
                 data_audio['multiple_speakers'] = False

        else:
            agglo = AgglomerativeClustering(n_clusters=n_clusters)
            # Assign the cluster labels back to the original DataFrame, aligning by index
            data_audio.loc[original_index, 'predicted_speaker'] = agglo.fit_predict(X_scaled)
            # For rows that had NaNs and were dropped from X, their 'predicted_speaker' will be NaN.
            # You might want to fill these with a placeholder like -1:
            data_audio['predicted_speaker'] = data_audio['predicted_speaker'].fillna(-1).astype(int)


        # 6. --- Build IntervalTree to find overlaps ---
        # Ensure 'time stamp' and 'end_time' exist before building the tree
        if 'time stamp' in data_audio.columns and 'end_time' in data_audio.columns:
            tree = IntervalTree()
            # Only add intervals for rows with valid time stamps and durations
            for idx, row in data_audio.dropna(subset=['time stamp', 'end_time']).iterrows():
                try:
                    tree[row['time stamp']:row['end_time']] = idx
                except ValueError as e:
                     print(f"Warning: Invalid interval [{row['time stamp']}:{row['end_time']}] for index {idx} in DataFrame {i}. Skipping this interval.")
                     # This could happen if start_time > end_time or if times are not numeric.


            # 7. --- Determine overlapping segments ---
            if not tree.is_empty():
                data_audio['num_overlapping'] = data_audio.apply(
                    lambda row: len(tree[row['time stamp']:row['end_time']]) if pd.notnull(row['time stamp']) and pd.notnull(row['end_time']) else 0,
                    axis=1
                )
                data_audio['multiple_speakers'] = data_audio['num_overlapping'] > 1
            else:
                 print(f"Warning: IntervalTree is empty for DataFrame {i}. Skipping overlap calculation.")
                 # Initialize columns if not already done in the skipping section
                 if 'num_overlapping' not in data_audio.columns:
                     data_audio['num_overlapping'] = 0
                 if 'multiple_speakers' not in data_audio.columns:
                     data_audio['multiple_speakers'] = False
                 # If time_stamp or end_time were missing, these would already be initialized
        else:
             print(f"Warning: 'time stamp' or 'end_time' missing in DataFrame {i}. Skipping overlap calculation.")
             # Initialize columns if not already done in the skipping section
             if 'num_overlapping' not in data_audio.columns:
                 data_audio['num_overlapping'] = 0
             if 'multiple_speakers' not in data_audio.columns:
                 data_audio['multiple_speakers'] = False


        # Append the processed DataFrame to the new list
        processed_audio_data[key] = data_audio

        plt.figure(figsize=(12, 5))
        for _, row in data_audio.iterrows():
            speaker_id = row['predicted_speaker']
            color ='blue'

            plt.hlines(y=speaker_id,
                      xmin=row['time stamp'],
                      xmax=row['end_time'],
                      linewidth=3,
                      color=color)
        plt.xlabel("Time (seconds)")
        plt.ylabel("Speaker ID")
        plt.title("Speaker Activity Timeline (Agglomerative Clustering)")
        plt.yticks(sorted(data_audio['predicted_speaker'].unique()))
        plt.tight_layout()
        plt.show()

        # Optional: Display the head of the processed DataFrame for this video
        print(f"Processed DataFrame head for video {key}:")
        display(data_audio.head())


    print("\n--- Finished processing all videos ---")

    # Now 'processed_audio_data' contains the DataFrames with 'predicted_speaker',
    # 'num_overlapping', and 'multiple_speakers' columns added.

    # You can now proceed with visualizations or further analysis using 'processed_audio_data'.
    # For example, to access the processed data for the first video:
    # first_video_processed_df = processed_audio_data[0]

In [ ]:
# @title Calculate Average Speaking Time Percentage Per Party
# Track known orator embeddings by party
known_orator_embeddings = defaultdict(list)

def get_avg_embeddings_by_speaker(df):
    """Return a dict: {speaker_id: avg_embedding} from a DataFrame."""
    embed_cols = [col for col in df.columns if col.startswith("embed_")]
    speaker_ids = df['predicted_speaker'].unique()
    return {
        speaker_id: df[df['predicted_speaker'] == speaker_id][embed_cols].mean().values
        for speaker_id in speaker_ids
    }

def identify_party_orator_ids_cross_video(processed_audio_data, party_name):
    """
    For a given party, compare all speakers across all videos they appear in,
    and return the most likely speaker ID per video based on cosine similarity.

    Returns:
        party_orator_map = {filename: predicted_speaker_id}
    """
    from sklearn.metrics.pairwise import cosine_similarity

    # Step 1: Get all videos for the party
    relevant_files = party_videos[party_name]
    print(f"Party '{party_name}' relevant videos: {relevant_files}")
    if len(relevant_files) < 2:
        print(f"Not enough videos to compare for party {party_name}")
        return {}

    # Step 2: Extract average embeddings per speaker for each video
    file_to_speaker_embeddings = {}
    for fname in relevant_files:
        df = processed_audio_data[fname]
        file_to_speaker_embeddings[fname] = get_avg_embeddings_by_speaker(df)

    # Step 3: For each file, find the speaker with max average similarity to others
    party_orator_map = {}

    for current_file, current_embeddings in file_to_speaker_embeddings.items():
        speaker_scores = {}

        for speaker_id, emb in current_embeddings.items():
            similarities = []

            for other_file, other_embeddings in file_to_speaker_embeddings.items():
                if other_file == current_file:
                    continue

                for other_emb in other_embeddings.values():
                    sim = cosine_similarity([emb], [other_emb])[0][0]
                    similarities.append(sim)

            if similarities:
                speaker_scores[speaker_id] = np.mean(similarities)

        if speaker_scores:
            best_speaker = max(speaker_scores, key=speaker_scores.get)
            party_orator_map[current_file] = best_speaker
            print(f"{current_file}: Selected speaker {best_speaker} for party '{party_name}' (avg sim: {speaker_scores[best_speaker]:.3f})")

    return party_orator_map

# Storage for results
party_speaking_times = defaultdict(float)
party_total_debate_times = defaultdict(float)
party_orator_per_video = defaultdict(lambda: defaultdict(int))

if isinstance(processed_audio_data, dict) and processed_audio_data:
    print("Calculating average speaking time percentage per party...")

    # parties_to_analyze = sorted(list(set([file.split('-')[0] for file in processed_audio_data if '-' in file])))
    # print(f"Analyzing speaking time for parties: {parties_to_analyze}")

    for party_name in party_videos:
        for filename, data_audio_processed in processed_audio_data.items():
            if party_name not in filename:
                continue
            try:
                # parts = filename.replace('_audio.pkl', '').split('-')
                # party_name = parts[0] if parts else 'unknown_party'
                # debate_name = '-'.join(parts[1:]) if len(parts) > 1 else 'unknown_debate'
                debate_name = filename.replace('_audio.pkl', '')

                if party_name in parties:
                    print(f"\nProcessing debate: {debate_name} for party: {party_name}")

                    party_orators_result = identify_party_orator_ids_cross_video(processed_audio_data, party_name)
                    party_orator_id = party_orators_result[filename]
                    party_orator_per_video[party_name][filename] = party_orator_id

                    if party_orator_id is not None:
                        print(f"Identified party orator with predicted_speaker ID: {party_orator_id}")

                        party_orator_data = data_audio_processed[data_audio_processed['predicted_speaker'] == party_orator_id]
                        speaking_time_this_debate = party_orator_data['duration'].sum()
                        party_speaking_times[party_name] += speaking_time_this_debate
                        print(f"Speaking time for {party_name} in {debate_name}: {speaking_time_this_debate:.2f} seconds")

                    else:
                        print(f"Could not identify party orator for {party_name} in {debate_name}. Skipping...")

                    total_debate_time_this_debate = data_audio_processed['duration'].sum()
                    party_total_debate_times[party_name] += total_debate_time_this_debate
                    print(f"Total debate time for {debate_name}: {total_debate_time_this_debate:.2f} seconds")

            except Exception as e:
                print(f"Error processing file '{filename}': {e}")

    print("\n--- Results ---")
    party_speaking_percentages = {}

    for party in parties:
        total_speaking_time = party_speaking_times[party]
        total_debate_time = party_total_debate_times[party]

        if total_debate_time > 0:
            speaking_percentage = (total_speaking_time / total_debate_time) * 100
            party_speaking_percentages[party] = speaking_percentage
            print(f"{party}: {speaking_percentage:.2f}% ({total_speaking_time:.2f}s / {total_debate_time:.2f}s)")
        else:
            print(f"{party}: No usable debate time.")
            party_speaking_percentages[party] = 0.0

    speaking_percentage_df = pd.DataFrame(
        list(party_speaking_percentages.items()),
        columns=['Party', 'Speaking Percentage (%)']
    ).sort_values(by='Speaking Percentage (%)', ascending=False).reset_index(drop=True)

    print("\nSpeaking Percentage Summary:")
    display(speaking_percentage_df)

else:
    print("Script halted: 'processed_audio_data' must be a non-empty dictionary.")

In [ ]:
# Build this once, after running your speaking time calculation loop:
orator_lookup = {}  # {(audio_key, speaker_id): party_name}

for party_name in party_videos:
    party_orators_result = identify_party_orator_ids_cross_video(processed_audio_data, party_name)
    for audio_key, speaker_id in party_orators_result.items():
        orator_lookup[(audio_key, speaker_id)] = party_name


## -- **Video + Audio** --

#  **- Emotions & Timber -**

In [ ]:
# @title Sort all video data

all_sorted_video_data = {}

for video_name in all_videos_data:
    print(f"\n--- Processing video_name: {video_name} ---")

    df = all_videos_data[video_name]

    all_sorted_video_data[video_name] = df.sort_values(by='filename').reset_index(drop=True)

    # Print first lines to verify the order for the current video
    print(f"Data after sorting by filename for {video_name}:")

    # .head(5) gets the first 5 rows of the DataFrame
    # ['filename'] selects the 'filename' column
    # .tolist() converts the selected column to a list for easier printing
    print(all_sorted_video_data[video_name]['filename'].head(5).tolist())

    # Print number of frames for the current video
    print(f"\nNumber of frames for {video_name}: {len(all_sorted_video_data[video_name])}")

In [ ]:
# @title For each video relate the audio snips to the frames

import math

for audio_name in processed_audio_data:
    frames_list = []
    video_name = audio_name.replace('_audio.pkl', '')

    for i, snip in processed_audio_data[audio_name].iterrows():
        start_time = math.ceil(snip["time stamp"])
        end_time = math.floor(snip["time stamp"] + snip["duration"])

        relevant_frames = all_sorted_video_data[video_name].loc[
            (all_sorted_video_data[video_name].index >= start_time) &
            (all_sorted_video_data[video_name].index <= end_time),
            "filename"
        ].tolist()

        frames_list.append(relevant_frames)

    # Add the full list as a new column
    processed_audio_data[audio_name][f"frames_{video_name}"] = frames_list

print(processed_audio_data)

In [ ]:
# @title Match audio snips to each orator

party_snips = defaultdict(list)

for party in party_videos:
    for filename, data_audio_processed in processed_audio_data.items():
        if party not in filename:
            continue
        # Get speaker ID for this party in this video
        speaker_id = party_orator_per_video[party][filename]

        # Filter rows for the party orator and where multiple_speakers == False
        filtered_snips = data_audio_processed[
            (data_audio_processed["predicted_speaker"] == speaker_id) &
            (~data_audio_processed["multiple_speakers"])  # or .notna() if it's sometimes missing
        ]

        party_snips[party].append(filtered_snips)

print(party_snips)

In [ ]:
# @title Load video stats from drive if the variable all_video_stats is empty

with open('/content/drive/MyDrive/PBD/Project/All_video_stats/all_video_stats.pkl', 'rb') as f:
    all_video_stats = pickle.load(f)

In [ ]:
# @title Fetching emotions embeddings for the images

for party in party_snips:
    party_debates = [ # testar
        col.replace("frames_", "")
        for i in range(len(party_snips[party]))
        for col in party_snips[party][i].columns
        if "frames_" in col
    ]

    # print(f"Analysing: {party}")
    for video, stats in all_video_stats.items():
        if video not in party_debates:
            continue
        idx = party_debates.index(video)

        role_assignment = stats.get("role_assignment", {})  # dict | key: int | value: string (role)
        frames_info = stats.get("frame_info", {})          # frame[idx] , idx is int
                                                            # "person_id": pid,
                                                            # "embedding": embedding,
                                                            # "emotion": emotion,
                                                            # "cluster": cluster,
                                                            # "location": location,
        frame_counts = stats.get("person_frame_count", {})  # total number of persons in a frame

        video_all_snip_embeddings = []
        video_all_snip_embeddings_avg = []
        video_all_snip_most_common_emotion = []
        for snip_frames in party_snips[party][idx][f"frames_{video}"]:

            snip_frames_idxs = [int(name.replace("img", "").replace(".jpeg","")) for name in snip_frames]

            pid = -1
            # Get id of current party in this video
            for _pid in role_assignment:
                if role_assignment[_pid].lower() == party:
                    pid = _pid
                    break

            # Get the embedding emotion
            snip_emotions = []
            snip_embeddings = []
            for frame in snip_frames_idxs:
                for fr_ps in range(len(frames_info[frame])):
                    if frames_info[frame][fr_ps]["person_id"] == pid:
                        snip_embeddings.append(frames_info[frame][fr_ps]["embedding"])
                        snip_emotions.append(frames_info[frame][fr_ps]["emotion"])
                        # aux = frames_info[frame][fr_ps]["emotion"]
                        # print(f"emotion: {aux}")
            try:
                # print(snip_emotions)
                most_common_emotion = Counter(snip_emotions).most_common(1)[0][0]
            except:
                most_common_emotion = "None"

            video_all_snip_most_common_emotion.append(most_common_emotion)
            video_all_snip_embeddings.append(snip_embeddings)
            # non_empty_embeddings = [emb for emb in video_all_snip_embeddings if len(emb) > 0]
            # if non_empty_embeddings:
            #     video_all_snip_embeddings_avg.append(np.mean(non_empty_embeddings, axis=0))
            # else:
            #     video_all_snip_embeddings_avg.append(np.zeros(512))

        party_snips[party][idx] = party_snips[party][idx].copy()
        party_snips[party][idx].loc[:, "most_common_emotion"] = video_all_snip_most_common_emotion
        party_snips[party][idx].loc[:, "emotion_embeddings"] = pd.Series(video_all_snip_embeddings)
        party_snips[party][idx].loc[:, "emotion_embeddings_avg"] = pd.Series(video_all_snip_embeddings_avg)



In [ ]:
# @title Ready audio type data

final_data = []

# Audio features to extract
audio_feature_columns = [
    'meanF0Hz', 'stdevF0Hz', 'HNR', 'localJitter', 'localabsoluteJitter', 'rapJitter',
    'ppq5Jitter', 'ddpJitter', 'localShimmer', 'localdbShimmer', 'apq3Shimmer',
    'apq5Shimmer', 'apq11Shimmer', 'ddaShimmer', 'f1_mean', 'f2_mean', 'f3_mean',
    'f4_mean', 'f1_median', 'f2_median', 'f3_median', 'f4_median', 'npause',
    'speechrate', 'articulationrate', 'asd', 'pF', 'fdisp', 'avgFormant', 'mff'
] + [f'embed_{i}' for i in range(512)]  # make sure embed_0 to embed_511 exist in your data

# Build model-ready data
for party in party_snips:
    for i, snip_df in enumerate(party_snips[party]):
        if 'most_common_emotion' not in snip_df or 'emotion_embeddings' not in snip_df:
            continue

        for j in range(len(snip_df)):
            audio_feats = snip_df[audio_feature_columns].iloc[j].to_dict()

            # Get emotion embedding and compute mean
            emb_list = snip_df["emotion_embeddings"].iloc[j]
            if isinstance(emb_list, list) and len(emb_list) > 0:
                avg_embedding = np.mean(emb_list, axis=0)
            else:
                avg_embedding = np.zeros(512)

            row = {
                **audio_feats,
                "emotion_emb": avg_embedding,
                "most_common_emotion": snip_df["most_common_emotion"].iloc[j]
            }
            final_data.append(row)

# Create final DataFrame
df_model = pd.DataFrame(final_data)

# Split embedding into separate columns (emotion_emb_0, emotion_emb_1, ...)
embeddings_expanded = pd.DataFrame(df_model["emotion_emb"].tolist(), index=df_model.index)
embeddings_expanded.columns = [f"emotion_emb_{i}" for i in range(embeddings_expanded.shape[1])]

# Merge
df_model = pd.concat([df_model.drop(columns=["emotion_emb"]), embeddings_expanded], axis=1)

print("✅ Final model-ready DataFrame shape:", df_model.shape)


In [ ]:
# 1. Predict most_common_emotion from audio features
audio_X = df_model[audio_feature_columns]
y = df_model["most_common_emotion"]

print(audio_X)
print(y)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(audio_X, y, test_size=0.2, random_state=42)

# Train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test)
print("🔍 Classification Report (Audio Features → Emotion):\n")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
# 2. Reduce and visualize emotion embeddings
emotion_cols = [col for col in df_model.columns if col.startswith("emotion_emb_")]
emotion_X = df_model[emotion_cols]

# Reduce to 2D
pca = PCA(n_components=2)
emotion_pca = pca.fit_transform(emotion_X)
emotion_pca_df = pd.DataFrame(emotion_pca, columns=["PC1", "PC2"])
emotion_pca_df["emotion"] = y.values

# Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(data=emotion_pca_df, x="PC1", y="PC2", hue="emotion", palette="Set2", alpha=0.7)
plt.title("PCA of Emotion Embeddings colored by Emotion")
plt.show()


In [ ]:
# 3. Correlation (optional insight)
combined_features = df_model[audio_feature_columns + emotion_cols]
corr_matrix = combined_features.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title("Correlation between Audio Features and Emotion Embeddings")
plt.show()


In [ ]:
# @title Sort all video data

all_sorted_video_data = {}

for video_name in all_videos_data:
    print(f"\n--- Processing video_name: {video_name} ---")

    df = all_videos_data[video_name]

    all_sorted_video_data[video_name] = df.sort_values(by='filename').reset_index(drop=True)

    # Print first lines to verify the order for the current video
    print(f"Data after sorting by filename for {video_name}:")

    # .head(5) gets the first 5 rows of the DataFrame
    # ['filename'] selects the 'filename' column
    # .tolist() converts the selected column to a list for easier printing
    print(all_sorted_video_data[video_name]['filename'].head(5).tolist())

    # Print number of frames for the current video
    print(f"\nNumber of frames for {video_name}: {len(all_sorted_video_data[video_name])}")

In [ ]:
# @title For each video relate the audio snips to the frames

import math

for audio_name in processed_audio_data:
    frames_list = []
    video_name = audio_name.replace('_audio.pkl', '')

    for i, snip in processed_audio_data[audio_name].iterrows():
        start_time = math.ceil(snip["time stamp"])
        end_time = math.floor(snip["time stamp"] + snip["duration"])

        relevant_frames = all_sorted_video_data[video_name].loc[
            (all_sorted_video_data[video_name].index >= start_time) &
            (all_sorted_video_data[video_name].index <= end_time),
            "filename"
        ].tolist()

        frames_list.append(relevant_frames)

    # Add the full list as a new column
    processed_audio_data[audio_name][f"frames_{video_name}"] = frames_list

print(processed_audio_data)

In [ ]:
# @title Match audio snips to each orator

party_snips = defaultdict(list)

for party in party_videos:
    for filename, data_audio_processed in processed_audio_data.items():
        if party not in filename:
            continue
        # Get speaker ID for this party in this video
        speaker_id = party_orator_per_video[party][filename]

        # Filter rows for the party orator and where multiple_speakers == False
        filtered_snips = data_audio_processed[
            (data_audio_processed["predicted_speaker"] == speaker_id) &
            (~data_audio_processed["multiple_speakers"])  # or .notna() if it's sometimes missing
        ]

        party_snips[party].append(filtered_snips)

print(party_snips)

In [ ]:
# @title Load video stats from drive if the variable all_video_stats is empty

with open('/content/drive/MyDrive/PBD/Project/All_video_stats/all_video_stats.pkl', 'rb') as f:
    all_video_stats = pickle.load(f)

In [ ]:
# @title Fetching emotions embeddings for the images

for party in party_snips:
    party_debates = [ # testar
        col.replace("frames_", "")
        for i in range(len(party_snips[party]))
        for col in party_snips[party][i].columns
        if "frames_" in col
    ]

    print(f"Analysing: {party}")
    for video, stats in all_video_stats.items():
        if video not in party_debates:
            continue
        idx = party_debates.index(video)

        role_assignment = stats.get("role_assignment", {})  # dict | key: int | value: string (role)
        frames_info = stats.get("frame_info", {})          # frame[idx] , idx is int
                                                            # "person_id": pid,
                                                            # "embedding": embedding,
                                                            # "emotion": emotion,
                                                            # "cluster": cluster,
                                                            # "location": location,
        frame_counts = stats.get("person_frame_count", {})  # total number of persons in a frame

        video_all_snip_embeddings = []
        video_all_snip_embeddings_avg = []
        video_all_snip_most_common_emotion = []
        for snip_frames in party_snips[party][idx][f"frames_{video}"]:

            snip_frames_idxs = [int(name.replace("img", "").replace(".jpeg","")) for name in snip_frames]

            pid = -1
            # Get id of current party in this video
            for _pid in role_assignment:
                if role_assignment[_pid] == party:
                    pid = _pid
                    break

            # Get the embedding emotion
            snip_emotions = []
            snip_embeddings = []
            for frame in snip_frames_idxs:
                for fr_ps in range(len(frames_info[frame])):
                    if frames_info[frame][fr_ps]["person_id"] == pid:
                        snip_embeddings.append(frames_info[frame]["embedding"])
                        snip_emotions.append(frames_info[frame]["emotion"])
            try:
                most_common_emotion = Counter(snip_emotions).most_common(1)[0][0]
            except:
                most_common_emotion = "None"

            video_all_snip_most_common_emotion.append(most_common_emotion)
            video_all_snip_embeddings.append(snip_embeddings)
            non_empty_embeddings = [emb for emb in video_all_snip_embeddings if len(emb) > 0]
            if non_empty_embeddings:
                video_all_snip_embeddings_avg.append(np.mean(non_empty_embeddings, axis=0))
            else:
                video_all_snip_embeddings_avg.append(np.zeros(512))

        party_snips[party][idx] = party_snips[party][idx].copy()
        party_snips[party][idx].loc[:, "most_common_emotion"] = video_all_snip_most_common_emotion
        party_snips[party][idx].loc[:, "emotion_embeddings"] = pd.Series(video_all_snip_embeddings)
        party_snips[party][idx].loc[:, "emotion_embeddings_avg"] = pd.Series(video_all_snip_embeddings_avg)



In [ ]:
# @title Prepare data

audio_features = [
    'meanF0Hz', 'stdevF0Hz', 'HNR', 'localJitter',
    'localabsoluteJitter', 'rapJitter', 'ppq5Jitter', 'ddpJitter',
    'localShimmer', 'localdbShimmer', 'apq3Shimmer', 'apq5Shimmer',
    'apq11Shimmer', 'ddaShimmer', 'f1_mean', 'f2_mean', 'f3_mean',
    'f4_mean', 'f1_median', 'f2_median', 'f3_median', 'f4_median', 'npause',
    'speechrate', 'articulationrate', 'asd'
] + [f'embed_{i}' for i in range(0, 512)] + [
    'pF', 'fdisp', 'avgFormant', 'mff'
]

variance_to_keep = 0.95

for party in party_snips:
    party_debates = [ # testar
        col.replace("frames_", "")
        for i in range(len(party_snips[party]))
        for col in party_snips[party][i].columns
        if "frames_" in col
    ]
    for idx in range(len(party_debates)):
        df = party_snips[party][idx]

        if df[audio_features].isnull().any(axis=None):
            print(f"Skipping {party_debates[idx]} due to missing values.")
            continue

        # Extract audio feature matrix
        X = df[audio_features]

        # Scale the data
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Fit PCA to retain the desired variance
        pca = PCA(n_components=variance_to_keep)
        X_pca = pca.fit_transform(X_scaled)

        # Store PCA results in the DataFrame or separately
        for comp_idx in range(X_pca.shape[1]):
            df[f"pca_audio_{comp_idx+1}"] = X_pca[:, comp_idx]

        print(f"{party} – {party_debates[idx]}: retained {X_pca.shape[1]} components to keep {variance_to_keep*100:.0f}% variance.")



# -- **Text** --

In [ ]:
# @title List all Pickle (.pkl) files that end with '_speech.pkl'

pklfiles = [f for f in os.listdir('/content/drive/MyDrive/PBD/Project/Project_Data/Features') if os.path.isfile(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Features', f)) and f.endswith('_speech.pkl')]
print(pklfiles)

In [ ]:
# @title Load one
video = 'chega-be'
data_speech = pd.read_pickle(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Features', video + '_speech.pkl'))

# Print first lines
data_speech.head()

In [ ]:
# @title Dictionary thecnical info
# --- CONFIGURATION ---
dictionary_text_pkl_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features/dictionary_text.pkl'
pt_dic_path = '/content/drive/MyDrive/PBD/Project/pt_PT.dic'

# --- LOAD TEXT DICTIONARY ---
try:
    data_dictionary = pd.read_pickle(dictionary_text_pkl_path)

    # --- Inspect the structure ---

    print("1. Data type of loaded object:")
    print(type(data_dictionary))

    print("\n2. First 5 rows of the DataFrame:")
    display(data_dictionary.head()) # Use display in Colab for better formatting

    print("\n3. Column names:")
    print(data_dictionary.columns)

    print("\n4. Data types of columns:")
    print(data_dictionary.dtypes)

    print("\n5. DataFrame Info:")
    data_dictionary.info()

except FileNotFoundError:
    print(f"Error: The file {dictionary_text_pkl_path} was not found.")
except Exception as e:
    print(f"An error occurred while loading or inspecting the pickle file: {e}")

# The load_dic_file function is not directly related to the structure of the data_dictionary pickle file,
# but I'll include it here as it was in your original code.
def load_dic_file(path):
    words = set()
    try:
        with open(path, encoding='utf-8', errors='ignore') as f:
            for line in f:
                word = line.strip().lower()
                if '/' in word:
                    word = word.split('/')[0]
                if word:
                    words.add(word)
    except FileNotFoundError:
        print(f"Error: The file {path} was not found.")
        raise
    except Exception as e:
        print(f"An error occurred while loading the dictionary file {path}: {e}")
        raise
    # print(f"Loaded {len(words)} unique words from {path} using utf-8 encoding.") # Uncomment if you want to see this output
    return words

# You can load the pt_dictionary separately if needed for other operations
# pt_dictionary = load_dic_file(dic_path='/content/drive/MyDrive/PBD/Project/pt_PT.dic')

In [ ]:
# @title Finding Words in the Transcripts that are not in the Dictionary
# --- CONFIGURATION ---
dictionary_text_pkl_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features/dictionary_text.pkl'
pt_dic_path = '/content/drive/MyDrive/PBD/Project/pt_PT.dic'

# Frequency is in the first column, which can be accessed by integer index 0
FREQ_COL_INDEX = 0

# --- LOAD TEXT DICTIONARY ---
try:
    df = pd.read_pickle(dictionary_text_pkl_path)
    print(f"Loaded DataFrame with {len(df)} rows and {len(df.columns)} columns.")
except FileNotFoundError:
    print(f"Error: The file {dictionary_text_pkl_path} was not found.")
    raise
except Exception as e:
    print(f"An unexpected error occurred loading dictionary_text.pkl: {e}")
    raise

# --- NORMALISE WORDS (from the index) ---
def normalise(word):
    # Lowercase, remove punctuation from start/end
    return re.sub(r'^\W+|\W+$', '', str(word).lower())

# Apply normalisation to the index and assign it back
df.index = df.index.to_series().apply(normalise)

# Remove entries where the normalized index (word) is empty
df = df[df.index != '']

# Sum frequencies for same words that appeared in different ways
df = df.groupby(df.index).sum()


def load_dic_file(path):
    words = set()
    try:
        with open(path, encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip().lower()
                if not line or line.startswith('['):  # skip empty or malformed lines
                    continue

                # Extract word before any whitespace or [CAT=...] block
                # Match patterns like: "word", "word/p", "word/variant [CAT=...]"
                word_part = re.split(r'\s|\[', line)[0]  # take the first token before space or [
                word = word_part.split('/')[0]  # remove any /suffix

                if word and not word.isdigit():
                    words.add(word)

    except FileNotFoundError:
        print(f"Error: The file {path} was not found.")
        raise
    except Exception as e:
        print(f"An error occurred while loading the dictionary file {path}: {e}")
        raise

    print(f"Loaded {len(words)} unique words from {path} using utf-8 encoding.\n")
    return words

pt_dict = load_dic_file(pt_dic_path)

# --- DEMONSTRATE WORD COMPARISON (for inspection) ---
print("\n--- Demonstrating Word Comparison ---")
# Get a few sample words from the DataFrame index
# Adjust the number of samples as needed
sample_words = df.index.unique().tolist()[:10] # Take first 10 unique words

for word in sample_words:
    is_in_dict = word in pt_dict # This is the core check
    print(f"Checking word '{word}': {'Found in Portuguese dictionary' if is_in_dict else 'NOT found in Portuguese dictionary'}")

# Optionally, check a few words you *expect* to be in the dictionary
# Example: Check if 'casa' (house) is in the dictionary
expected_word = 'casa'
is_expected_word_in_dict = expected_word in pt_dict
print(f"Checking word '{expected_word}': {'Found in Portuguese dictionary' if is_expected_word_in_dict else 'NOT found in Portuguese dictionary'}")
print("------------------------------------\n")


# --- FILTER & PRINT UNKNOWN WORDS ---

# Filter words in the DataFrame index that are not in the Portuguese dictionary
df_not_in_dict = df.loc[~df.index.isin(pt_dict)]

# Filter words in the DataFrame index that *are* in the Portuguese dictionary
df_in_dict = df.loc[df.index.isin(pt_dict)]

# Print the number of words common to both
print(f"Found {len(df_in_dict)} words that are in the Portuguese dictionary.\n")

print(f"Found {len(df_not_in_dict)} words not in Portuguese dictionary.\n")

# Check if df_not_in_dict is not empty before sorting and iterating
if not df_not_in_dict.empty:
    # Sort by the frequency column in descending order
    sorted_unknown_words = df_not_in_dict.sort_values(by=df_not_in_dict.columns[FREQ_COL_INDEX], ascending=False)

    # Select the top 100 most common unknown words
    # top_100_unknown_words = sorted_unknown_words.head(100)

    print("Most common 100 words not in Portuguese dictionary (sorted by frequency):")
    # Iterate through the top 100 words
    for word, row in sorted_unknown_words.iterrows():
      if row.iloc[FREQ_COL_INDEX] > 1:
        # 'word' is the index value (the normalized word)
        # row.iloc[FREQ_COL_INDEX] accesses the frequency value by integer index
        print(f"{word} (freq: {row.iloc[FREQ_COL_INDEX]})")
else:
    print("All words found in the Portuguese dictionary.")

In [ ]:
# @title Before loading the model, ensure it is downloaded
!python -m spacy download pt_core_news_lg

In [ ]:
# @title Using a model to keep only words normaly used in Portuguese News

import spacy
nlp = spacy.load("pt_core_news_lg")

named_entities = set()
for word in df_not_in_dict.index:
    doc = nlp(word)
    for ent in doc.ents:
        named_entities.add(ent.text.lower())

df_filtered_or_named = df_not_in_dict[(df_not_in_dict.index.isin(named_entities))]

print(f"Number of words before filtering or named entity resolution: {len(df_not_in_dict)}")
print(f"Number of words after filtering or named entity resolution: {len(df_filtered_or_named)}")

In [ ]:
# @title Visualization of the words kept by the model

# Check if df_filtered_or_named is not empty before sorting and iterating
if not df_filtered_or_named.empty:
    # Sort by the frequency column in descending order
    sorted_unknown_words = df_filtered_or_named.sort_values(by=df_filtered_or_named.columns[FREQ_COL_INDEX], ascending=False)

    # Select the top 100 most common unknown words
    # top_100_unknown_words = sorted_unknown_words.head(100)

    print("Most common 100 words not in Portuguese dictionary (sorted by frequency):")
    # Iterate through the top 100 words
    for word, row in sorted_unknown_words.iterrows():
      if row.iloc[FREQ_COL_INDEX] > 1:
        # 'word' is the index value (the normalized word)
        # row.iloc[FREQ_COL_INDEX] accesses the frequency value by integer index
        print(f"{word} (freq: {row.iloc[FREQ_COL_INDEX]})")
else:
    print("All words found in the Portuguese dictionary.")

In [ ]:
# @title Group the 2 dictonaries and make one big dictonary <br> in order to help in the creation of the embeedings

# 1. Get the set of words from the original Portuguese dictionary
base_dict_words = pt_dict  # already a set

# 2. Extract the index from df_filtered_or_named (named entities found in transcripts)
named_entity_words = set(df_filtered_or_named.index)

# 3. Combine both sets into one big dictionary
final_dict = base_dict_words.union(named_entity_words)

print(f"\n✅ Final dictionary created:")
print(f"- {len(base_dict_words)} words from pt_PT.dic")
print(f"- {len(named_entity_words)} named entities from transcripts")
print(f"- {len(final_dict)} total unique words in the final dictionary\n")

for word in final_dict:
  print(word)

In [ ]:
# @title Full Script to Process All Videos

# --- CONFIGURATION ---
features_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features'
speech_files = [f for f in os.listdir(features_path) if f.endswith('_speech.pkl')]
stop_words_pt = set(stopwords.words('portuguese'))

# --- Functions ---
def remove_stopwords(text):
    """
    Lowercases and tokenises the text, removes Portuguese stopwords and punctuation.
    """
    words = word_tokenize(text.lower())  # Tokenize and lowercase
    filtered_words = [
        word for word in words
        if word not in stop_words_pt and word not in string.punctuation
    ]
    return " ".join(filtered_words)

def filter_by_dictionary(text, dictionary):
    """
    Keeps only words that are in the provided dictionary set.
    """
    words = text.split()
    filtered_words = [word for word in words if word in dictionary]
    return " ".join(filtered_words)

# --- PROCESSING ---

total_counter = Counter()
file_counts = {}

for filename in speech_files:
    path = os.path.join(features_path, filename)
    try:
        df = pd.read_pickle(path)
        all_words = []

        for transcript in df['transcript'].dropna():
            cleaned = remove_stopwords(transcript)               # Remove stopwords & punctuation
            words = cleaned.split()                              # Tokenise cleaned result
            filtered_words = filter_by_dictionary(" ".join(words), final_dict)  # Apply dictionary filter
            all_words.extend(filtered_words.split())             # Add filtered tokens to list

        video_id = filename.replace('_speech.pkl', '')
        count = Counter(all_words)
        file_counts[video_id] = count
        total_counter.update(count)

        print(f"✅ Processed {video_id}: {len(count)} unique words, {sum(count.values())} total tokens")

    except Exception as e:
        print(f"⚠️ Failed to process {filename}: {e}")

# --- FINAL OUTPUT ---

print("\n--- Overall Word Frequencies Across All Videos ---")
for word, freq in total_counter.most_common(50):  # top 50
    print(f"{word}: {freq}")

In [ ]:
# @title Per-Video Bar-Plots with the **number of times a word was used** <br> All-Videos Bar-Plot at the end

# --- PER-VIDEO BAR PLOTS ---

for video_id, counter in file_counts.items():
    top_words = counter.most_common(20)
    words, freqs = zip(*top_words)

    plt.figure(figsize=(10, 5))
    plt.barh(words, freqs, color='cornflowerblue')
    plt.gca().invert_yaxis()
    plt.title(f"Top 20 Words – {video_id}")
    plt.xlabel("Frequency")
    plt.tight_layout()
    plt.show()

# --- OVERALL (ALL VIDEOS) BAR PLOT ---

top_total = total_counter.most_common(20)
words, freqs = zip(*top_total)

plt.figure(figsize=(10, 5))
plt.barh(words, freqs, color='seagreen')
plt.gca().invert_yaxis()
plt.title("Top 20 Words – All Videos Combined")
plt.xlabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# @title Per-Video Bar-Plots with the **embeeding value for all words** <br> All-Videos Bar-Plot at the end

# Download stopwords if you haven't already
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

def remove_stopwords(text):
    words = nltk.word_tokenize(text.lower()) # Tokenize and convert to lowercase
    filtered_words = [word for word in words if word not in stop_words_pt and word not in string.punctuation]
    return " ".join(filtered_words)

def filter_by_dictionary(text, dictionary):
    words = text.split() # Assuming words are space-separated after stop word removal
    filtered_words = [word for word in words if word in dictionary]
    return " ".join(filtered_words)

transcript = data_speech['transcript'].iloc[0]

# 1. Remove Most used words without a meaning (stop words)
stop_words_pt = set(stopwords.words('portuguese'))
processed_transcript = remove_stopwords(transcript)

# Apply the function (using the placeholder dictionary)
filtered_transcript = filter_by_dictionary(processed_transcript, final_dict)
print("\nTranscript after filtering by dictionary:")
print(filtered_transcript)

# 3. Create embeddings based on all the words in the video
# - TF-IDF (Term Frequency-Inverse Document Frequency)
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming you have a list of all transcripts from your video in `data_speech['transcript']`
all_transcripts = data_speech['transcript'].tolist()

# Apply preprocessing (removing stop words and punctuation) to all transcripts
processed_all_transcripts = [remove_stopwords(t) for t in all_transcripts]

# If you have a dictionary filtering function, apply it here too
processed_all_transcripts = [filter_by_dictionary(t, final_dict) for t in processed_all_transcripts]

# Create a TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(vocabulary=list(final_dict))

# Fit the vectorizer to your corpus and transform the transcripts into embeddings
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_all_transcripts)

# tfidf_matrix now contains the TF-IDF embeddings for each transcript
print("\nShape of the TF-IDF matrix (number of transcripts, number of unique words):")
print(tfidf_matrix.shape)

# You can access the embeddings for a specific transcript (e.g., the first one)
print("\nTF-IDF embedding for the first transcript:")
print(tfidf_matrix[0])

ft_names = tfidf_vectorizer.get_feature_names_out()
flat_tfidf = tfidf_matrix[0].toarray().flatten()
tfidf_series = pd.Series(flat_tfidf, index=ft_names)
sorted_tfidf = tfidf_series.sort_values(ascending=False)

print(sorted_tfidf.head(10))

# The feature names correspond to the words in your vocabulary
# print("\nVocabulary (words):")
# print(tfidf_vectorizer.get_feature_names_out())

In [ ]:
# @title Per-Video Bar-Plots with the **embeeding value for all words** <br> All-Videos Bar-Plot at the end

stop_words_pt = set(nltk.corpus.stopwords.words('portuguese'))

# --- Configuration ---
features_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features'
speech_files = [f for f in os.listdir(features_path) if f.endswith('_speech.pkl')]

# --- Preprocessing functions ---
def remove_stopwords(text):
    words = nltk.word_tokenize(text.lower())
    return " ".join([
        word for word in words
        if word not in stop_words_pt and word not in string.punctuation
    ])

def filter_by_dictionary(text, dictionary):
    return " ".join([
        word for word in text.split() if word in dictionary
    ])

# --- Storage ---
all_documents = []  # each entry is a transcript
doc_labels = []     # map each transcript to a video
per_video_text = defaultdict(list)

# --- Process and clean all videos ---
for filename in speech_files:
    path = os.path.join(features_path, filename)
    try:
        df = pd.read_pickle(path)
        video_id = filename.replace('_speech.pkl', '')

        for t in df['transcript'].dropna():
            cleaned = remove_stopwords(t)
            filtered = filter_by_dictionary(cleaned, final_dict)
            if filtered.strip():  # skip empty
                all_documents.append(filtered)
                doc_labels.append(video_id)
                per_video_text[video_id].append(filtered)

        print(f"✅ Cleaned {video_id}: {len(per_video_text[video_id])} transcripts")

    except Exception as e:
        print(f"⚠️ Failed to process {filename}: {e}")

# --- Per-video TF-IDF analysis ---
for video_id, texts in per_video_text.items():
    vectorizer = TfidfVectorizer(vocabulary=list(final_dict))
    tfidf_matrix = vectorizer.fit_transform(texts)
    summed = tfidf_matrix.sum(axis=0).A1  # sum across transcripts
    words = vectorizer.get_feature_names_out()

    tfidf_scores = pd.Series(summed, index=words).sort_values(ascending=False)
    top_words = tfidf_scores.head(20)

    # Plot
    plt.figure(figsize=(10, 5))
    plt.barh(top_words.index[::-1], top_words.values[::-1], color='cornflowerblue')
    plt.title(f"Top 20 TF-IDF Words – {video_id}")
    plt.xlabel("TF-IDF Score")
    plt.tight_layout()
    plt.show()

# --- Global TF-IDF across all videos ---
global_vectorizer = TfidfVectorizer(vocabulary=list(final_dict))
global_matrix = global_vectorizer.fit_transform(all_documents)
global_summed = global_matrix.sum(axis=0).A1
global_words = global_vectorizer.get_feature_names_out()

global_scores = pd.Series(global_summed, index=global_words).sort_values(ascending=False)
top_global_words = global_scores.head(20)

# Plot
plt.figure(figsize=(10, 5))
plt.barh(top_global_words.index[::-1], top_global_words.values[::-1], color='seagreen')
plt.title("Top 20 TF-IDF Words – All Videos Combined")
plt.xlabel("TF-IDF Score")
plt.tight_layout()
plt.show()

In [ ]:
# @title How to explore the dictionary and the transcripts? - Try using Bag-of-Words
#Cool demo at https://colab.research.google.com/github/littlecolumns/ds4j-notebooks/blob/master/text-analysis/notebooks/Counting%20words%20with%20scikit-learn's%20CountVectorizer.ipynb
from sklearn.feature_extraction.text import CountVectorizer

corpus = [transcript]

vectorizer = CountVectorizer(vocabulary = data_dictionary.index)
count_matrix = vectorizer.transform(corpus)
bow = vectorizer.get_feature_names_out()
count_array = count_matrix.toarray()

print(count_array.shape)
print(data_dictionary.index.shape)

words = np.where(np.squeeze(count_array) != 0)

#See words that were found
print(bow[words])

In [ ]:
# @title For a full video
texts = data_speech['transcript']

corpus = []

for text in texts:
  corpus.append(text)

count_matrix = vectorizer.transform(corpus)
bow = vectorizer.get_feature_names_out()
count_array = count_matrix.toarray()

print(count_array.shape)
print(data_dictionary.index.shape)

for i in range(count_array.shape[0]):
  words = np.where(np.squeeze(count_array[i,:]) != 0)

  print(bow[words])

# -- **Transcript IDs for one video** --

In [ ]:
# @title Put previously audio person id in the same text segments

# 1. Make sure you have 'end_time' columns
data_audio['end_time'] = data_audio['time stamp'] + data_audio['duration']
data_speech['end_time'] = data_speech['timestamp'] + data_speech['duration']

# 2. New column for the speaker with thresholding
data_speech['audio_predicted_speaker'] = -1
data_speech['speaker_confidence'] = 0.0

CONFIDENCE_THRESHOLD = 0.7  # 70% majority

audio_starts = data_audio['time stamp'].values
audio_ends = data_audio['end_time'].values
audio_speakers = data_audio['predicted_speaker'].values

for i, row in data_speech.iterrows():
    t0, t1 = row['timestamp'], row['end_time']
    overlaps = (audio_starts < t1) & (audio_ends > t0)
    overlapping_speakers = audio_speakers[overlaps]
    if len(overlapping_speakers) == 0:
        continue
    # Find most frequent speaker and confidence
    vals, counts = np.unique(overlapping_speakers, return_counts=True)
    max_idx = np.argmax(counts)
    majority_speaker = vals[max_idx]
    confidence = counts[max_idx] / np.sum(counts)
    data_speech.at[i, 'speaker_confidence'] = confidence
    if confidence >= CONFIDENCE_THRESHOLD:
        data_speech.at[i, 'audio_predicted_speaker'] = majority_speaker
    else:
        data_speech.at[i, 'audio_predicted_speaker'] = -1  # or 'ambiguous' if you prefer

# Example: View results
print(data_speech[['timestamp', 'duration', 'audio_predicted_speaker', 'speaker_confidence', 'transcript']].head(10))

In [ ]:
# @title Timeline plot of speaker

plt.figure(figsize=(12, 3))

segment_time = data_speech['timestamp'] + data_speech['duration'] / 2

# Confident segments
confident_mask = data_speech['audio_predicted_speaker'] != -1
plt.step(
    segment_time[confident_mask],
    data_speech.loc[confident_mask, 'audio_predicted_speaker'],
    where='post', color='blue', label='Confident assignment'
)

# Ambiguous segments: Draw a red line ONLY during each ambiguous segment
ambiguous_mask = data_speech['audio_predicted_speaker'] == -1
for idx, row in data_speech[ambiguous_mask].iterrows():
    t_start = row['timestamp']
    t_end = row['timestamp'] + row['duration']
    plt.hlines(
        y=-0.5,
        xmin=t_start,
        xmax=t_end,
        color='red',
        linewidth=4,
        alpha=0.7,
        label='Ambiguous assignment' if idx == data_speech[ambiguous_mask].index[0] else ""  # only once in legend
    )

plt.xlabel("Time (seconds)")
plt.ylabel("Predicted Speaker")
plt.title("Transcript Segment Speaker Timeline (Confidence Threshold, Time-Based)")
plt.yticks(list(sorted(data_speech['audio_predicted_speaker'].unique())) + [-0.5])
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# @title Number of words spoken by each speaker

# Only count words in segments with high confidence in speaker assignment
confident_speech = data_speech[data_speech['speaker_confidence'] >= CONFIDENCE_THRESHOLD]

# Compute word counts per speaker
words_by_speaker = confident_speech.groupby('audio_predicted_speaker')['word_count'].sum()

print("Number of words spoken by each (confident) speaker:")
print(words_by_speaker)

In [ ]:
# @title Texts segments spoken by a speaker

CONFIDENCE_THRESHOLD = 0.7

for speaker in sorted(data_speech['audio_predicted_speaker'].unique()):
    if speaker == -1:
        print("== Ambiguous Speaker ==")
    else:
        print(f"== Speaker {speaker} ==")
    for _, row in data_speech[
        (data_speech['audio_predicted_speaker'] == speaker) &
        (data_speech['speaker_confidence'] >= CONFIDENCE_THRESHOLD)
    ].iterrows():
        print(row['transcript'])
    print()

In [ ]:
# @title Timeplot of speakers

CONFIDENCE_THRESHOLD = 0.7

mask = (data_speech['audio_predicted_speaker'] == -1) | (data_speech['speaker_confidence'] < CONFIDENCE_THRESHOLD)
print("== Ambiguous or Low Confidence Segments ==")
for _, row in data_speech[mask].iterrows():
    print(f"[Speaker {row['audio_predicted_speaker']}, {row['speaker_confidence']:.2f}] [{row['timestamp']:.1f}s - {row['timestamp']+row['duration']:.1f}s] {row['transcript']}\n")

In [ ]:
# @title Texts segments spoken by speakers


from termcolor import colored

print("\nSpeaker-labeled Transcript (Ambiguous in Red):\n")
for _, row in data_speech.iterrows():
    speaker = row['audio_predicted_speaker']
    text = row['transcript']
    conf = row.get('speaker_confidence', 1.0)
    if speaker == -1 or conf < 0.7:
        print(colored(f"[Ambiguous, conf={conf:.2f}]", 'red'), text)
    elif speaker == 0:
        print(colored(f"[Speaker 0, conf={conf:.2f}]", 'blue'), text)
    elif speaker == 1:
        print(colored(f"[Speaker 1, conf={conf:.2f}]", 'green'), text)
    elif speaker == 2:
        print(colored(f"[Speaker 2, conf={conf:.2f}]", 'magenta'), text)
    else:
        print(colored(f"[Speaker {speaker}, conf={conf:.2f}]", 'yellow'), text)

In [ ]:
# @title Histogram of duration of segments

plt.figure(figsize=(7,3))
plt.hist(data_speech['duration'], bins=20, color='steelblue', edgecolor='black')
plt.xlabel("Segment Duration (s)")
plt.ylabel("Count")
plt.title("Histogram of Segment Durations")
plt.show()

In [ ]:
# @title Number os speaking turns per speaker


turn_counts = data_speech['audio_predicted_speaker'].value_counts().sort_index()
turn_counts.index = ['Ambiguous' if idx == -1 else f"Speaker {idx}" for idx in turn_counts.index]
turn_counts.plot(kind='bar', color=['red' if idx=='Ambiguous' else 'blue' for idx in turn_counts.index])
plt.ylabel("Number of Segments (Turns)")
plt.title("Number of Speaking Turns per Speaker")
plt.show()

# -- **Transcript IDs for all video** --

In [ ]:
# @title Load all text (transcript) data

text_features_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features'
text_file_suffix = '_speech.pkl'  # Change to '_speech.pkl' if that's your suffix!

# List all relevant .pkl files
textfiles = [
    f for f in os.listdir(text_features_path)
    if os.path.isfile(os.path.join(text_features_path, f)) and f.endswith(text_file_suffix)
]

all_speech_data = {}

for file in textfiles:
    file_path = os.path.join(text_features_path, file)
    try:
        df = pd.read_pickle(file_path)
        all_speech_data[file] = df
        print(f"✓ Loaded {file}")
    except FileNotFoundError:
        print(f"✗ Error: File not found at {file_path}")
    except Exception as e:
        print(f"✗ Failed to load {file}: {e}")

print(f"\nTotal text dataframes loaded: {len(all_speech_data)}")

In [ ]:
# @title Match parties to their text files
parties = ["ad", "be", "cdu", "chega", "il", "livre", "pan", "ps"]

# Use the keys from all_speech_data (which are your text file names)
party_texts = {party: [f for f in all_speech_data if party in f] for party in parties}

for party in party_texts:
    print(f"{party}: {party_texts[party]}")

In [ ]:
CONFIDENCE_THRESHOLD = 0.7  # 70% required to assign, else ambiguous

for audio_key in processed_audio_data.keys():
    # Convert '_audio.pkl' to '_speech.pkl'
    speech_key = audio_key.replace('_audio.pkl', '_speech.pkl')
    if speech_key not in all_speech_data:
        print(f"Skipping {audio_key}: No corresponding speech data ({speech_key}) found.")
        continue

    print(f"\n--- Aligning speakers for debate/video: {audio_key} ---")
    data_audio = processed_audio_data[audio_key]
    data_speech = all_speech_data[speech_key]

    # 1. Make sure you have 'end_time' columns
    data_audio['end_time'] = data_audio['time stamp'] + data_audio['duration']
    data_speech['end_time'] = data_speech['timestamp'] + data_speech['duration']

    # 2. New columns for speaker with thresholding
    data_speech['audio_predicted_speaker'] = -1
    data_speech['speaker_confidence'] = 0.0

    audio_starts = data_audio['time stamp'].values
    audio_ends = data_audio['end_time'].values
    audio_speakers = data_audio['predicted_speaker'].values

    for i, row in data_speech.iterrows():
        t0, t1 = row['timestamp'], row['end_time']
        overlaps = (audio_starts < t1) & (audio_ends > t0)
        overlapping_speakers = audio_speakers[overlaps]
        if len(overlapping_speakers) == 0:
            continue
        vals, counts = np.unique(overlapping_speakers, return_counts=True)
        max_idx = np.argmax(counts)
        majority_speaker = vals[max_idx]
        confidence = counts[max_idx] / np.sum(counts)
        data_speech.at[i, 'speaker_confidence'] = confidence
        if confidence >= CONFIDENCE_THRESHOLD:
            # Use party name if found, else fallback to speaker ID or ambiguous
            party_name = orator_lookup.get((audio_key, majority_speaker), -1)
            data_speech.at[i, 'audio_predicted_speaker'] = party_name
        else:
            data_speech.at[i, 'audio_predicted_speaker'] = -1  # or "ambiguous"

    # (Optional) Save or update the aligned dataframe
    all_speech_data[speech_key] = data_speech

    print(data_speech[['timestamp', 'duration', 'audio_predicted_speaker', 'speaker_confidence', 'transcript']].head(5))
    print(f"Speaker segment distribution:\n{data_speech['audio_predicted_speaker'].value_counts()}\n")

print("✅ Speaker alignment done for all debates/videos.")

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import nltk
import string

# nltk.download('stopwords')
# nltk.download('punkt')

stop_words_pt = set(nltk.corpus.stopwords.words('portuguese'))

def remove_stopwords(text):
    words = nltk.word_tokenize(text.lower())
    filtered_words = [word for word in words if word not in stop_words_pt and word not in string.punctuation]
    return filtered_words  # List

def filter_by_dictionary(words, dictionary):
    return [word for word in words if word in dictionary]

global_counts_by_party = {}

for speech_key, data_speech in all_speech_data.items():
    print(f"\n▶ Processing video: {speech_key}")

    # Use unique *party labels* (skipping ambiguous)
    party_labels = sorted([p for p in data_speech['audio_predicted_speaker'].unique() if p != -1 and p != '-1'])

    for party_label in party_labels:
        segments = data_speech[data_speech['audio_predicted_speaker'] == party_label]['transcript'].dropna()
        if segments.empty:
            continue
        full_text = " ".join(segments)
        words = remove_stopwords(full_text)
        filtered = filter_by_dictionary(words, final_dict)
        counter = Counter(filtered)

        # Add to global collection
        global_counts_by_party.setdefault(party_label, []).append(counter)

        # --- Per-video bar plot for this party ---
        top_words = counter.most_common(20)
        if not top_words: continue
        words_, freqs_ = zip(*top_words)
        plt.figure(figsize=(10, 5))
        plt.barh(words_, freqs_, color='cornflowerblue')
        plt.gca().invert_yaxis()
        plt.title(f"Top 20 Words – Party '{party_label}' – {speech_key.replace('_speech.pkl','')}")
        plt.xlabel("Frequency")
        plt.tight_layout()
        plt.show()

# --- OVERALL (ALL VIDEOS) BAR PLOT PER PARTY ---
for party_label, counters in global_counts_by_party.items():
    total_counter = Counter()
    for c in counters:
        total_counter.update(c)
    top_total = total_counter.most_common(20)
    if not top_total: continue
    words_, freqs_ = zip(*top_total)
    plt.figure(figsize=(10, 5))
    plt.barh(words_, freqs_, color='seagreen')
    plt.gca().invert_yaxis()
    plt.title(f"Top 20 Words – Party '{party_label}' – All Videos Combined")
    plt.xlabel("Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
import nltk
import string
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

# nltk.download('stopwords')
# nltk.download('punkt')

stop_words_pt = set(nltk.corpus.stopwords.words('portuguese'))

def remove_stopwords(text):
    words = nltk.word_tokenize(text.lower())
    filtered_words = [word for word in words if word not in stop_words_pt and word not in string.punctuation]
    return " ".join(filtered_words)

def filter_by_dictionary(text, dictionary):
    words = text.split()
    filtered_words = [word for word in words if word in dictionary]
    return " ".join(filtered_words)

# FINAL GLOBAL PLOT COLLECTION
all_party_corpus = {}

for speech_key, data_speech in all_speech_data.items():
    print(f"\n▶ Processing video: {speech_key}")
    if 'audio_predicted_speaker' not in data_speech.columns:
        print(f"audio_predicted_speaker missing for {speech_key}, skipping.")
        continue

    # Remove ambiguous (-1 or '-1') if you only want certain parties
    party_labels = sorted([p for p in data_speech['audio_predicted_speaker'].unique() if p != -1 and p != '-1'])

    for party_label in party_labels:
        # Concatenate all segments spoken by this party
        party_segs = data_speech[data_speech['audio_predicted_speaker'] == party_label]['transcript'].dropna()
        if len(party_segs) == 0:
            continue
        full_text = " ".join(party_segs)
        cleaned = filter_by_dictionary(remove_stopwords(full_text), final_dict)

        # Save for all-videos global plot
        all_party_corpus.setdefault(party_label, []).append(cleaned)

        # TF-IDF (just for this party in this debate)
        vectorizer = TfidfVectorizer(vocabulary=list(final_dict))
        tfidf = vectorizer.fit_transform([cleaned])
        ft_names = vectorizer.get_feature_names_out()
        tfidf_series = pd.Series(tfidf.toarray().flatten(), index=ft_names)
        tfidf_sorted = tfidf_series.sort_values(ascending=False)
        # Only plot top 10 words
        tfidf_top = tfidf_sorted.head(10)

        plt.figure(figsize=(8,2))
        tfidf_top.plot(kind='bar')
        plt.title(f"Top 10 TF-IDF Words - Party '{party_label}'\n{speech_key.replace('_speech.pkl','')}")
        plt.xlabel("Word")
        plt.ylabel("TF-IDF Score")
        plt.tight_layout()
        plt.show()

# --- ALL VIDEOS: Global per-party plot (sum across all videos) ---
for party_label, text_list in all_party_corpus.items():
    # Combine all cleaned segments from all videos for this party
    full_text = " ".join(text_list)
    vectorizer = TfidfVectorizer(vocabulary=list(final_dict))
    tfidf = vectorizer.fit_transform([full_text])
    ft_names = vectorizer.get_feature_names_out()
    tfidf_series = pd.Series(tfidf.toarray().flatten(), index=ft_names)
    tfidf_sorted = tfidf_series.sort_values(ascending=False)
    tfidf_top = tfidf_sorted.head(10)

    plt.figure(figsize=(8,2))
    tfidf_top.plot(kind='bar')
    plt.title(f"Global Top 10 TF-IDF Words - Party '{party_label}'")
    plt.xlabel("Word")
    plt.ylabel("TF-IDF Score")
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import nltk

def assign_word_timestamps_and_party(df, video_name):
    """
    For each segment, assign a timestamp for each word, party label, and segment meta.
    Assumes 'audio_predicted_speaker' contains the party label, not the numeric ID.
    """
    if 'transcript' not in df.columns:
        print(f"Video {video_name}: No 'transcript' column, skipping.")
        return pd.DataFrame()
    words_records = []
    for idx, row in df.iterrows():
        transcript = row['transcript']
        if pd.isnull(transcript) or row['duration'] <= 0:
            continue
        words = nltk.word_tokenize(transcript)
        n_words = len(words)
        if n_words == 0:
            continue
        start = row['timestamp']
        duration = row['duration']
        per_word = duration / n_words
        for i, word in enumerate(words):
            word_start = start + i * per_word
            word_end = word_start + per_word
            words_records.append({
                "word": word,
                "segment_idx": idx,
                "segment_start": start,
                "segment_end": row['end_time'],
                "word_start": word_start,
                "word_end": word_end,
                "party": row.get('audio_predicted_speaker', -1),  # clearer naming!
                "speaker_confidence": row.get('speaker_confidence', 0.0),
                "video": video_name
            })
    return pd.DataFrame(words_records)

# Build word-level dataframe for ALL videos
all_words = []

for speech_key, data_speech in all_speech_data.items():
    word_df = assign_word_timestamps_and_party(data_speech, speech_key)
    if word_df.empty:
        continue

    # Find corresponding audio features
    audio_key = speech_key.replace('_speech.pkl', '_audio.pkl')
    if audio_key in processed_audio_data:
        audio_df = processed_audio_data[audio_key]

        keep_audio_cols = ['time stamp', 'speechrate', 'articulationrate', 'npause']
        audio_df_small = audio_df[keep_audio_cols].rename(columns={'time stamp': 'timestamp'})

        # Merge: word_df['segment_start'] == audio_df_small['timestamp']
        word_df = word_df.merge(audio_df_small, left_on='segment_start', right_on='timestamp', how='left')
        word_df = word_df.drop(columns=['timestamp'])

    all_words.append(word_df)

# Combine all into a single dataframe
all_words_df = pd.concat(all_words, ignore_index=True)

In [ ]:
print(all_words_df.head())

In [ ]:
# @title Associate all emotions with the respective most commun words used in each frame

from pprint import PrettyPrinter
from collections import defaultdict, Counter

# To store words spoken under each emotion category
emotion_word_map = defaultdict(Counter)

aux = 0
# Iterate through all rows in the words dataframe
for _, row in all_words_df.iterrows():
  ''' if aux > 1000:
    break '''
  aux += 1
  pid = row['segment_idx']
  video = row['video'].replace('_speech.pkl', '')
  party = row['party']
  if party == -1:
    continue
  else:
    party = party.upper()
  word = row['word'].lower()
  if word not in final_dict:
    if word != None:
      # print(f"word: {word}")
      continue
  frame = int(row['word_start'])

  # print(f"video: {video}")
  # print(f"word: {word}")
  # print(f"party: {party}")
  # print(f"frame: {frame}")
  stats = all_video_stats.get(video)
  # print(f"stats: {stats}")
  if not stats:
    continue

  frame_people = stats.get("frame_info", {}).get(frame, [])
  emotion = None

  for person_data in frame_people:
    if person_data.get("person_id") == pid:
      emotion = person_data.get("emotion")
      #if emotion != None:
      #  print(f"Found emotion for PID {pid}: {emotion}")
      break

  # Only count if emotion is valid
  if emotion:
    emotion_word_map[emotion][word] += 1

  #for emotion, words in emotion_word_map.items():
  #  print(f"\n🔹 Emotion: {emotion}")
  #  for word, count in words.most_common(10):
  #    print(f"  {word}: {count}")


In [ ]:
print(f"\nTotal words processed: {aux}")
for emotion, words in emotion_word_map.items():
  print(f"\n🔹 Emotion: {emotion}")
  for word, count in words.most_common(50):
    print(f"  {word}: {count}")

In [ ]:
import matplotlib.pyplot as plt

for debate in all_words_df['video'].unique():
    print(f"debate: {debate}")
    df = all_words_df[all_words_df['video'] == debate]
    plt.figure(figsize=(14, 2))

    # Use 'party' column if you renamed, otherwise keep 'audio_predicted_speaker'
    party_col = 'party' if 'party' in df.columns else 'audio_predicted_speaker'

    parties = [p for p in df[party_col].unique() if p != -1 and p != '-1']
    colors = plt.cm.get_cmap('tab10', len(parties))

    for i, party in enumerate(parties):
        party_df = df[df[party_col] == party]
        plt.scatter(party_df['word_start'], [1]*len(party_df),
                    label=f"Party {party}", c=[colors(i)], s=18, alpha=0.8)
    plt.title(f"Word Timeline for {debate}")
    plt.xlabel("Time (s)")
    plt.yticks([])
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for debate in all_words_df['video'].unique():
    df = all_words_df[all_words_df['video'] == debate]
    # Use 'party' if you renamed, otherwise fallback to 'audio_predicted_speaker'
    party_col = 'party' if 'party' in df.columns else 'audio_predicted_speaker'

    # Remove ambiguous
    df_plot = df[df[party_col].isin([-1, '-1']) == False]
    avg_rates = df_plot.groupby(party_col)['speechrate'].mean().reset_index()
    plt.figure(figsize=(6, 3))
    sns.barplot(data=avg_rates, x=party_col, y='speechrate', palette="tab10")
    plt.title(f"Average Speechrate by Party — {debate.replace('_speech.pkl', '')}")
    plt.ylabel("Speechrate (syllables/sec)")
    plt.xlabel("Party")
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

for debate in all_words_df['video'].unique():
    df = all_words_df[all_words_df['video'] == debate]
    # Use 'party' if available, else fallback
    party_col = 'party' if 'party' in df.columns else 'audio_predicted_speaker'
    # Remove ambiguous labels
    valid_df = df[~df[party_col].isin([-1, '-1'])]
    word_counts = valid_df[party_col].value_counts().sort_index()

    plt.figure(figsize=(6, 3))
    word_counts.plot(kind='bar', color='skyblue')
    plt.title(f"Number of Words Spoken by Party — {debate.replace('_speech.pkl', '')}")
    plt.xlabel("Party")
    plt.ylabel("Word Count")
    plt.tight_layout()
    plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for debate in all_words_df['video'].unique():
    df = all_words_df[all_words_df['video'] == debate]
    # Use 'party' if renamed, else fallback to 'audio_predicted_speaker'
    party_col = 'party' if 'party' in df.columns else 'audio_predicted_speaker'
    # Exclude ambiguous values
    df_plot = df[~df[party_col].isin([-1, '-1'])]
    plt.figure(figsize=(7, 3))
    sns.boxplot(x=party_col, y='npause', data=df_plot, palette='tab10')
    plt.title(f"Pause Distribution by Party — {debate.replace('_speech.pkl','')}")
    plt.xlabel("Party")
    plt.ylabel("Number of Pauses (per segment)")
    plt.tight_layout()
    plt.show()


# -- **Pulsometer** --

In [ ]:
# @title Load the single file
data_pulsometer = pd.read_pickle(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Features', 'pulsometer_text.pkl'))

# Print first lines
data_pulsometer.head()

In [ ]:
# Load one
data_grades = pd.read_pickle(os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Features', 'grades_text.pkl'))

# Print first lines
data_grades.head()


print("\nVisualizing the scores across different categories using a bar plot:")

scores = data_grades.iloc[0]

# To sort in descending order (highest score first):
sorted_scores_descending = scores.sort_values(ascending=False)

# The index of this Series will be the column names (the categories)
categories = sorted_scores_descending.index

# The values of this Series are the scores for each category
score_values = sorted_scores_descending.values

# --- Create a Bar Plot to compare scores across categories ---

plt.figure(figsize=(12, 7)) # Set a suitable figure size

plt.bar(categories, score_values, color='teal')

plt.xlabel("Category (Political Party)") # Label for the x-axis
plt.ylabel("Ranking/Grade") # Label for the y-axis (adjust based on what the scores represent)
plt.title("Comparison of Rankings Across Categories") # Title of the plot
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability

# Add the score values on top of the bars for clarity
for i, value in enumerate(score_values):
    plt.text(i, value + 0.1, f'{value:.1f}', ha='center', va='bottom') # Position text slightly above the bar

plt.grid(axis='y', linestyle='--', alpha=0.7) # Add a horizontal grid
plt.tight_layout() # Adjust layout
plt.show() # Display the plot


# --- Create a Single Box Plot of All Scores ---

plt.figure(figsize=(6, 8)) # Set a suitable figure size

plt.boxplot(score_values, showfliers=True) # showfliers=True displays individual data points if there are any outliers

plt.title("Overall Distribution of Rankings/Grades (All Categories Combined)") # Title of the plot
plt.ylabel("Ranking/Grade Value") # Label for the y-axis (adjust based on what the scores represent)

# For a single box plot representing an aggregate, the x-axis often has a generic label
plt.xticks([1], ['All Scores'])

plt.grid(axis='y', linestyle='--', alpha=0.7) # Add a horizontal grid for readability

plt.tight_layout() # Adjust layout
plt.show() # Display the plot


In [ ]:
# @title Display Scores next to the Frame count in each party

import matplotlib.pyplot as plt
from collections import defaultdict
import numpy as np

# --- Step 1: Normalised Frame Count Collection ---

party_frame_totals = defaultdict(float)
total_frames_all_videos = 0

for video_name, stats in all_video_stats.items():
    role_map = stats.get("role_assignment", {})
    person_frame_count = stats.get("person_frame_count", {})

    total_frames = sum(len(frames) for frames in person_frame_count.values())
    if total_frames == 0:
        continue

    total_frames_all_videos += total_frames

    for pid, frames in person_frame_count.items():
        role = role_map.get(pid, f"Person {pid}")
        if role in ("Moderator", "Gestural Language"):
            continue
        party_frame_totals[role] += len(frames)

# Normalize frame counts
for party in party_frame_totals:
    party_frame_totals[party] /= total_frames_all_videos

# --- Step 2: Map scores correctly (strip 'ranking_global_' prefix) ---

scores = data_grades.iloc[0]
score_map = {k.replace("ranking_global_", "").upper(): v for k, v in scores.items()}

# --- Step 3: Align and Normalize Both Sets by Their Max ---

common_parties = sorted(set(party_frame_totals) & set(score_map))
frame_values = [party_frame_totals[p] for p in common_parties]
score_values = [score_map[p] for p in common_parties]

max_frame = max(frame_values)
max_score = max(score_values)

norm_frame_values = [v / max_frame for v in frame_values]
norm_score_values = [v / max_score for v in score_values]

# Sort by frame value (can adjust to score instead if needed)
sorted_data = sorted(
    zip(common_parties, norm_frame_values, norm_score_values),
    key=lambda x: x[1],
    reverse=True
)

parties, norm_frame_values, norm_score_values = zip(*sorted_data)

# --- Step 4: Plot ---

x = np.arange(len(parties))
width = 0.35

plt.figure(figsize=(14, 7))
bars1 = plt.bar(x - width/2, norm_frame_values, width, label='Normalised Frame Count', color='mediumseagreen')
bars2 = plt.bar(x + width/2, norm_score_values, width, label='Normalised Score', color='coral')

plt.xlabel("Party")
plt.ylabel("Normalised Value")
plt.title("Normalised Frame Count and Score per Party (0–1 Scale)")
plt.xticks(x, parties, rotation=45, ha='right')

# Annotate bars
for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.02, f'{height:.2f}', ha='center', va='bottom')

for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.02, f'{height:.2f}', ha='center', va='bottom')

plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.ylim(0, 1.15)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# --- Extract Scores and Top 3 Emotions (excluding 'neutral' and 'surprise') ---

# 1. Standardise party names from scores
scores = data_grades.iloc[0]
score_map = {k.replace("ranking_global_", "").upper(): v for k, v in scores.items()}

# 2. Extract top valid emotions per party
excluded_emotions = {"neutral", "surprise"}
top_emotions = {}

for role, emotions in role_emotion_totals.items():
    party = role.upper()
    if party in score_map:
        filtered = [(e, c) for e, c in emotions.items() if e not in excluded_emotions]
        top3 = sorted(filtered, key=lambda x: x[1], reverse=True)[:3]
        top_emotions[party] = [e for e, _ in top3]

# 3. Sort by score
sorted_data = sorted(score_map.items(), key=lambda x: x[1], reverse=True)
parties, score_values = zip(*sorted_data)
emotion_labels = [top_emotions.get(party, []) for party in parties]

# --- Plot ---

plt.figure(figsize=(12, 7))
bars = plt.bar(parties, score_values, color='mediumorchid')

plt.xlabel("Party")
plt.ylabel("Score")
plt.title("Scores per Party with Top 3 Emotions (excluding 'neutral' and 'surprise')")
plt.xticks(rotation=45, ha='right')

# Add score values slightly *inside* the bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height - 0.3, f"{score_values[i]:.1f}",
             ha='center', va='top', fontsize=10, color='white', fontweight='bold')

    # Add top 3 emotions stacked above the bar
    for j, emo in enumerate(reversed(emotion_labels[i])):
        plt.text(bar.get_x() + bar.get_width()/2, height + 0.2 + j * 0.25, emo,
                 ha='center', va='bottom', fontsize=9, color='black')

plt.ylim(0, max(score_values) + 1.5)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:

import moviepy.video.io.ImageSequenceClip
image_folder=os.path.join('/content/drive/MyDrive/PBD/Project/Project_Data/Video_Frames', video)
fps=1
image_files = [os.path.join(image_folder,img)
               for img in os.listdir(image_folder)
               if img.endswith(".jpeg")]
clip = moviepy.video.io.ImageSequenceClip.ImageSequenceClip(image_files, fps=fps)
clip.write_videofile('my_video.mp4')

# **Co-occurrence of emotions at speaker turns**

In [ ]:
# --- Config ---
features_path = '/content/drive/MyDrive/PBD/Project/Project_Data/Features'
visual_files = [f for f in os.listdir(features_path) if f.endswith('_visual.pkl')]
chunk_size = 100  # Number of frames per simulated speaker chunk

# --- Storage ---
speaker_emotion_counts_all = {}  # video_name: [Counter_speaker_0, Counter_speaker_1]

# --- Process each video ---
for file in visual_files:
    video_name = file.replace('_visual.pkl', '')
    visual_path = os.path.join(features_path, file)

    try:
        data_visual = pd.read_pickle(visual_path)
        emotion_counter_by_speaker = [Counter(), Counter()]

        for i, row in data_visual.iterrows():
            fer_data = row.get('fer', [])
            speaker_id = (i // chunk_size) % 2  # alternate speakers

            if isinstance(fer_data, list):
                for face in fer_data:
                    emotion = face.get('emotion')
                    if isinstance(emotion, str):
                        emotion_counter_by_speaker[speaker_id][emotion] += 1

        speaker_emotion_counts_all[video_name] = emotion_counter_by_speaker
        print(f"✅ Processed {video_name}")

    except Exception as e:
        print(f"⚠️ Error processing {video_name}: {e}")


In [ ]:
# --- Combine speaker emotion counters into total counts per video ---
video_emotion_summary = {}

for video_name, (speaker0, speaker1) in speaker_emotion_counts_all.items():
    combined = speaker0 + speaker1  # Merge both speaker counters
    video_emotion_summary[video_name] = dict(combined)

# --- Convert to DataFrame ---
emotion_df = pd.DataFrame.from_dict(video_emotion_summary, orient='index').fillna(0).astype(int)

# --- Sort by video name (optional) ---
emotion_df = emotion_df.sort_index()

# --- Plot heatmap ---
plt.figure(figsize=(12, len(emotion_df) * 0.4 + 2))  # Dynamic height
sns.heatmap(emotion_df, annot=True, fmt='d', cmap='YlOrRd')
plt.title("Dominant Emotion Frequency per Video (Simulated Speakers Combined)")
plt.xlabel("Emotion")
plt.ylabel("Video")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation between columns (emotions)
emotion_correlation = emotion_df.corr(method='pearson')  # or 'spearman'

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.heatmap(emotion_correlation, annot=True, cmap='coolwarm')
plt.title("Correlation Between Emotions (Across Videos)")
plt.show()


In [ ]:
# Compute correlation between videos based on emotion profiles
video_correlation = emotion_df.T.corr(method='pearson')

# Optional: reorder using clustering
clustermap = sns.clustermap(video_correlation, col_cluster=False, row_cluster=True, cmap='coolwarm', figsize=(0.1, 0.1))

# Convert row indices to label names for proper reordering
reordered_labels = video_correlation.index[clustermap.dendrogram_row.reordered_ind]
video_correlation = video_correlation.loc[reordered_labels, reordered_labels]

# Now plot the heatmap with clean layout
plt.figure(figsize=(14, 10))
sns.heatmap(
    video_correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    linecolor='gray',
    square=True,
    cbar_kws={"shrink": 0.8},
    annot_kws={"size": 8}
)

plt.title("Correlation Between Videos Based on Emotion Profiles", fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
reduced = pca.fit_transform(emotion_df)

plt.figure(figsize=(10, 6))
plt.scatter(reduced[:, 0], reduced[:, 1])

for i, label in enumerate(emotion_df.index):
    plt.text(reduced[i, 0], reduced[i, 1], label)

plt.title("PCA Projection of Videos by Emotion Distribution")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()


In [ ]:
# Normalise each row so the sum of emotions for each party = 1 (proportions)
party_summary_norm = party_summary_df.div(party_summary_df.sum(axis=1), axis=0)

# Plot the normalised heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(party_summary_norm, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title("Normalised Emotion Proportions per Party (Facial Expression Based)")
plt.xlabel("Emotion")
plt.ylabel("Party")
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure party_summary_df exists (rows = parties, columns = emotions)
parties = party_summary_df.index.tolist()

# Initialise empty DataFrames
correlation_matrix = pd.DataFrame(index=parties, columns=parties, dtype=float)
pvalue_matrix = pd.DataFrame(index=parties, columns=parties, dtype=float)

# Compute pairwise correlations
for p1 in parties:
    for p2 in parties:
        corr, pval = pearsonr(party_summary_df.loc[p1], party_summary_df.loc[p2])
        correlation_matrix.loc[p1, p2] = corr
        pvalue_matrix.loc[p1, p2] = pval

# --- Plot the correlation heatmap ---
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix.astype(float), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation Between Parties Based on Emotion Profiles")
plt.xlabel("Party")
plt.ylabel("Party")
plt.tight_layout()
plt.show()

# --- Plot the p-value heatmap ---
plt.figure(figsize=(10, 6))
sns.heatmap(pvalue_matrix.astype(float), annot=True, cmap='crest', vmin=0, vmax=1)
plt.title("P-values for Party Emotion Profile Correlations")
plt.xlabel("Party")
plt.ylabel("Party")
plt.tight_layout()
plt.show()


# **Relations Between Data of different types**


In [ ]:
# @title Pose vs Emotion

# Assume 'all_videos_data' is already loaded with pose data.
# We will now explicitly create the dummy 'all_videos_emotion_embeddings'

print("\n--- Analyzing Pose vs Emotion by Person Across Videos ---")

# --- 1. Extract Pose Embeddings per Person per Frame ---

print("\n--- Extracting Pose Embeddings per Person per Frame ---")

# This will store pose embeddings organized by person ID and then by video/frame
person_pose_embeddings = {}

# We need a way to identify individuals across frames and videos.
# This is a crucial and potentially complex step depending on your data.
# If you have person tracking or IDs in your data, you would use them.
# If not, you might need a simple approach like assuming the first detected
# person in each frame is the same individual, which is highly unreliable
# across frames and videos.
# **For this example, we'll use a simplified approach:**
# Assuming the first detected pose in each frame belongs to the primary person of interest.
# This is a strong assumption and will only work if your videos consistently
# feature the same primary individual in the first pose slot.
# **You should replace this with a proper person tracking/identification mechanism if possible.**

# For now, let's create a 'pseudo' person ID per video, assuming
# the first person in each video is the one we are interested in.
# If you need to track the same person across different debates
# you would need a more sophisticated person re-identification method.


for video_name, video_data_df in all_videos_data.items():
    print(f"Processing pose data for {video_name} to extract person embeddings...")

    person_pseudo_id = f"{video_name}_person_1"
    person_pose_embeddings[person_pseudo_id] = []

    for frame_index in range(len(video_data_df)):
        poses = np.array(video_data_df.iloc[frame_index]['poses'])

        if len(poses) > 0:
            # Simplified assumption: take the first detected pose as the primary person
            pose = poses[0]

            # Ensure pose has enough keypoints and coordinate data (x, y)
            # Assuming keypoints have at least 2 coordinates (x, y)
            if pose.shape[1] >= 2:
                # Extract (x, y) coordinates for all keypoints in the pose
                # Flatten the coordinates into a 1D array to form the pose embedding
                pose_embedding = pose[:, :2].flatten()

                # Store the pose embedding along with the video and frame info
                person_pose_embeddings[person_pseudo_id].append({
                    'video_name': video_name,
                    'frame_index': frame_index,
                    'pose_embedding': pose_embedding
                })
            else:
                 print(f"Warning: Pose data for {video_name}, frame {frame_index} has unexpected shape for keypoint coordinates ({pose.shape}). Skipping.")
        # else:
            # print(f"No poses detected in {video_name}, frame {frame_index}. Skipping.")


print(f"Extracted pose embeddings for {len(person_pose_embeddings)} pseudo-persons.")

# --- 2. Align Pose Embeddings with Emotion Embeddings per Person ---

print("\n--- Aligning Pose Embeddings with Emotion Embeddings per Person ---")

# This will store aligned pose and emotion embeddings organized by person ID
aligned_person_data = {}

for person_pseudo_id, pose_data_list in person_pose_embeddings.items():
    print(f"Aligning data for {person_pseudo_id}...")

    person_video_name = person_pseudo_id.split('_person_1')[0] # Extract video name from pseudo ID

    if person_video_name in all_videos_emotion_embeddings:
        emotion_embeddings_df = all_videos_emotion_embeddings[person_video_name]
        person_aligned_frames = []

        for pose_info in pose_data_list:
            frame_index = pose_info['frame_index']

            # Ensure the emotion embeddings DataFrame has data for this frame index
            if frame_index < len(emotion_embeddings_df):
                # Get the emotion embedding for this frame
                emotion_embedding = emotion_embeddings_df.iloc[frame_index].values # Get the numpy array of embedding values

                # Store the aligned pose and emotion embeddings
                person_aligned_frames.append({
                    'pose_embedding': pose_info['pose_embedding'],
                    'emotion_embedding': emotion_embedding
                })
            # else:
                # print(f"Warning: No emotion embedding found for {person_pseudo_id} in {person_video_name} at frame {frame_index}. Skipping frame.")

        if person_aligned_frames:
            aligned_person_data[person_pseudo_id] = person_aligned_frames
            print(f"Aligned {len(person_aligned_frames)} frames for {person_pseudo_id}.")
        else:
             print(f"No frames could be aligned for {person_pseudo_id}.")

    else:
        print(f"Emotion embeddings not found for video '{person_video_name}' associated with {person_pseudo_id}. Skipping.")


print(f"Aligned data for {len(aligned_person_data)} pseudo-persons.")


# --- 3. Analyze Relationship per Person ---

print("\n--- Analyzing Relationship per Person ---")

if not aligned_person_data:
    print("No aligned data available for analysis.")
else:
    for person_pseudo_id, aligned_frames in aligned_person_data.items():
        print(f"\nAnalyzing data for {person_pseudo_id}...")

        # Convert list of dictionaries to DataFrame for easier analysis
        # Each row will be a frame, with columns for pose embedding dimensions
        # and emotion embedding dimensions.
        pose_embedding_dim = len(aligned_frames[0]['pose_embedding']) if aligned_frames else 0
        emotion_embedding_dim = len(aligned_frames[0]['emotion_embedding']) if aligned_frames else 0

        if pose_embedding_dim == 0 or emotion_embedding_dim == 0:
            print(f"Skipping {person_pseudo_id}: Pose or emotion embedding dimensions are zero.")
            continue

        # Create DataFrame with columns for pose and emotion dimensions
        person_df = pd.DataFrame({
            f'pose_{i}': [frame['pose_embedding'][i] for frame in aligned_frames] for i in range(pose_embedding_dim)
        })
        for i in range(emotion_embedding_dim):
            person_df[f'emotion_{i}'] = [frame['emotion_embedding'][i] for frame in aligned_frames]

        print(f"DataFrame shape for {person_pseudo_id}: {person_df.shape}")


        # --- Example Analysis per Person: Correlation ---

        print(f"\nCorrelation Matrix for {person_pseudo_id}:")
        numerical_cols = person_df.select_dtypes(include=np.number).columns
        correlation_matrix = person_df[numerical_cols].corr()
        # print(correlation_matrix)

        # Visualize the correlation matrix
        plt.figure(figsize=(14, 12)) # Adjust figure size based on number of dimensions
        sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt=".2f")
        plt.title(f"Correlation Matrix: Pose Embeddings vs. Emotion Embeddings ({person_pseudo_id})")
        plt.tight_layout()
        plt.show()


        # --- Example Analysis per Person: Dimensionality Reduction and Visualization (PCA) ---
        print(f"\nPerforming PCA for {person_pseudo_id}...")

        pose_embedding_cols = [f'pose_{i}' for i in range(pose_embedding_dim)]
        emotion_embedding_cols = [f'emotion_{i}' for i in range(emotion_embedding_dim)]

        if not pose_embedding_cols or not emotion_embedding_cols:
             print(f"Skipping PCA for {person_pseudo_id}: Could not identify pose or emotion embedding columns.")
             continue

        pose_embeddings_data = person_df[pose_embedding_cols].values
        emotion_embeddings_data = person_df[emotion_embedding_cols].values

        # Scale the data before PCA
        scaler_pose = StandardScaler()
        scaler_emotion = StandardScaler()
        scaled_pose_embeddings = scaler_pose.fit_transform(pose_embeddings_data)
        scaled_emotion_embeddings = scaler_emotion.fit_transform(emotion_embeddings_data)

        # Apply PCA to retain at least 90% variance
        print("Applying PCA to retain at least 90% variance...")

        # PCA for Pose Embeddings
        pca_pose = PCA(n_components=0.9) # Retain 90% variance
        pca_pose_result = pca_pose.fit_transform(scaled_pose_embeddings)
        print(f"PCA for Pose Embeddings ({person_pseudo_id}): Retained {pca_pose.n_components_} components to explain >90% variance.")
        print(f"Explained variance ratio: {np.sum(pca_pose.explained_variance_ratio_):.4f}")


        # PCA for Emotion Embeddings
        pca_emotion = PCA(n_components=0.9) # Retain 90% variance
        pca_emotion_result = pca_emotion.fit_transform(scaled_emotion_embeddings)
        print(f"PCA for Emotion Embeddings ({person_pseudo_id}): Retained {pca_emotion.n_components_} components to explain >90% variance.")
        print(f"Explained variance ratio: {np.sum(pca_emotion.explained_variance_ratio_):.4f}")


        # Visualize the PCA results (if enough components for 2D visualization)
        if pca_pose_result.shape[1] >= 2 or pca_emotion_result.shape[1] >= 2:
            plt.figure(figsize=(14, 6))

            if pca_pose_result.shape[1] >= 2:
                plt.subplot(1, 2, 1)
                plt.scatter(pca_pose_result[:, 0], pca_pose_result[:, 1], alpha=0.5, s=10)
                plt.title(f"PCA (>{np.sum(pca_pose.explained_variance_ratio_)*100:.0f}% Var) of Pose Embeddings ({person_pseudo_id})")
                plt.xlabel("Principal Component 1")
                plt.ylabel("Principal Component 2")
                plt.grid(True)
            else:
                plt.subplot(1, 2, 1)
                plt.text(0.5, 0.5, 'Not enough components (>2) for 2D PCA plot', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
                plt.title(f"PCA (>{np.sum(pca_pose.explained_variance_ratio_)*100:.0f}% Var) of Pose Embeddings ({person_pseudo_id})")


            if pca_emotion_result.shape[1] >= 2:
                plt.subplot(1, 2, 2)
                plt.scatter(pca_emotion_result[:, 0], pca_emotion_result[:, 1], alpha=0.5, s=10)
                plt.title(f"PCA (>{np.sum(pca_emotion.explained_variance_ratio_)*100:.0f}% Var) of Emotion Embeddings ({person_pseudo_id})")
                plt.xlabel("Principal Component 1")
                plt.ylabel("Principal Component 2")
                plt.grid(True)
            else:
                 plt.subplot(1, 2, 2)
                 plt.text(0.5, 0.5, 'Not enough components (>2) for 2D PCA plot', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
                 plt.title(f"PCA (>{np.sum(pca_emotion.explained_variance_ratio_)*100:.0f}% Var) of Emotion Embeddings ({person_pseudo_id})")


            plt.tight_layout()
            plt.show()
        else:
             print(f"Not enough components retained by PCA for 2D visualization for {person_pseudo_id}.")

        # You could also try t-SNE for visualization, but it's more computationally
        # expensive and sensitive to hyperparameters.

        # --- Further Analysis Options per Person ---
        # - Cluster the combined data for this person and analyze the clusters
        # - Train a model to predict emotion embedding dimensions from pose embedding dimensions for this person
        # - Analyze time-series relationships between pose and emotion for this person

    print("\nAnalysis per person completed.")
    print("Consider implementing more sophisticated person tracking for more robust cross-video analysis.")
    print("Explore clustering and predictive modeling techniques for deeper insights.")